# Historical research notebook — PUBLIC REDACTED CODE LOG
**Archive only; not runnable from start to finish.** All original outputs and cells embedding Hebrew transcription data were removed. It contains installation, data manipulation and potentially destructive operations. Inspect every cell before use. The complete original remains private pending data-rights review.


In [ ]:
import requests
from PIL import Image
from io import BytesIO
from pathlib import Path
import hashlib
import csv
import time

# =========================
# CONFIGURAÇÃO
# =========================

ZOOM = 7
X_MIN = 0
X_MAX = 63
Y_MIN = 0
Y_MAX = 8

BASE_URL = "http://tiles.imj.org.il/tiles/community/{}_{}_{}.jpg"

BASE_DIR = Path("Qumran_Analytics/1QS")
TILES_DIR = BASE_DIR / "raw_tiles"
OUT_DIR = BASE_DIR / "reconstructed"
META_DIR = BASE_DIR / "metadata"

TILES_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

metadata = []
tiles = {}

session = requests.Session()

# =========================
# DOWNLOAD DOS TILES
# =========================

for y in range(Y_MIN, Y_MAX + 1):
    for x in range(X_MIN, X_MAX + 1):

        filename = f"{ZOOM}_{x}_{y}.jpg"
        filepath = TILES_DIR / filename
        url = BASE_URL.format(ZOOM, x, y)

        try:
            response = session.get(url, timeout=20)

            if response.status_code != 200:
                print(f"Ausente: {filename}")
                continue

            data = response.content

            # Confirma que recebemos imagem
            if "image" not in response.headers.get("Content-Type", ""):
                continue

            filepath.write_bytes(data)

            sha256 = hashlib.sha256(data).hexdigest()

            img = Image.open(BytesIO(data)).convert("RGB")

            tiles[(x, y)] = img

            metadata.append({
                "zoom": ZOOM,
                "x": x,
                "y": y,
                "filename": filename,
                "url": url,
                "width": img.width,
                "height": img.height,
                "bytes": len(data),
                "sha256": sha256
            })

            print(f"OK: {filename}")

            # pequena pausa por respeito ao servidor
            time.sleep(0.15)

        except Exception as e:
            print(f"Erro em {filename}: {e}")

# =========================
# METADADOS
# =========================

metadata_file = META_DIR / "tiles_metadata.csv"

with open(metadata_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "zoom", "x", "y", "filename",
            "url", "width", "height",
            "bytes", "sha256"
        ]
    )

    writer.writeheader()
    writer.writerows(metadata)

# =========================
# RECONSTRUÇÃO
# =========================

if not tiles:
    raise RuntimeError("Nenhum tile válido foi encontrado.")

tile_w = max(img.width for img in tiles.values())
tile_h = max(img.height for img in tiles.values())

canvas_width = (X_MAX - X_MIN + 1) * tile_w
canvas_height = (Y_MAX - Y_MIN + 1) * tile_h

canvas = Image.new(
    "RGB",
    (canvas_width, canvas_height),
    "white"
)

for (x, y), img in tiles.items():

    px = (x - X_MIN) * tile_w
    py = (y - Y_MIN) * tile_h

    canvas.paste(img, (px, py))

output_file = OUT_DIR / "1QS_zoom7_full.png"
canvas.save(output_file, lossless=True)

print("\n=========================")
print("PROCESSO CONCLUÍDO")
print("=========================")
print(f"Tiles encontrados: {len(tiles)}")
print(f"Imagem: {output_file.resolve()}")
print(f"Metadados: {metadata_file.resolve()}")
print(f"Dimensões finais: {canvas.size}")

In [ ]:
from google.colab import files

files.download(
    "/content/Qumran_Analytics/1QS/reconstructed/1QS_zoom7_full.png"
)

In [ ]:
from PIL import Image
from google.colab import files

img = Image.open(
    "/content/Qumran_Analytics/1QS/reconstructed/1QS_zoom7_full.png"
)

# cria apenas uma prévia para análise visual
target_width = 4000
target_height = round(img.height * target_width / img.width)

preview = img.resize(
    (target_width, target_height),
    Image.Resampling.LANCZOS
)

preview_path = "/content/1QS_preview.jpg"

preview.save(
    preview_path,
    "JPEG",
    quality=90,
    optimize=True
)

print("Original:", img.size)
print("Preview:", preview.size)

files.download(preview_path)

In [ ]:
from PIL import Image
from pathlib import Path

# Arquivo reconstruído em alta resolução
source = Path(
    "/content/Qumran_Analytics/1QS/reconstructed/1QS_zoom7_full.png"
)

img = Image.open(source)

print("Imagem original:", img.size)

# Pasta de saída
out_dir = Path(
    "/content/Qumran_Analytics/1QS/columns"
)
out_dir.mkdir(parents=True, exist_ok=True)

# Limites encontrados na imagem original de 16000 px
boundaries = [
    0,
    1516,
    3141,
    4555,
    5930,
    7430,
    9039,
    10711,
    12438,
    13695,
    14797,
    16000
]

# A imagem está organizada XI -> I da esquerda para a direita
column_names = [
    "XI", "X", "IX", "VIII", "VII",
    "VI", "V", "IV", "III", "II", "I"
]

# Pequena margem de segurança
OVERLAP = 100

for i, name in enumerate(column_names):

    left = boundaries[i]
    right = boundaries[i + 1]

    # adiciona sobreposição sem ultrapassar imagem
    crop_left = max(0, left - OVERLAP)
    crop_right = min(img.width, right + OVERLAP)

    col = img.crop(
        (crop_left, 0, crop_right, img.height)
    )

    filename = out_dir / f"1QS_col_{name}.png"
    col.save(filename)

    print(
        f"Coluna {name}: "
        f"x={crop_left}:{crop_right} "
        f"-> {filename}"
    )

print("\nConcluído!")

In [ ]:
from google.colab import files

files.download(
    "/content/Qumran_Analytics/1QS/columns/1QS_col_III.png"
)

In [ ]:
from PIL import Image
from pathlib import Path
import csv

source = Path(
    "/content/Qumran_Analytics/1QS/columns/1QS_col_III.png"
)

img = Image.open(source)

out_dir = Path(
    "/content/Qumran_Analytics/1QS/lines/III"
)
out_dir.mkdir(parents=True, exist_ok=True)

metadata_dir = Path(
    "/content/Qumran_Analytics/1QS/metadata"
)
metadata_dir.mkdir(parents=True, exist_ok=True)

# Área útil da coluna III
X_LEFT = 90
X_RIGHT = 1190

# Centros das 26 linhas detectadas na imagem
line_centers = [
    138, 198, 256, 320, 381, 440, 506,
    562, 622, 679, 756, 811, 926, 977,
    1047, 1103, 1174, 1232, 1293, 1354,
    1412, 1485, 1547, 1618, 1676, 1743
]

# Calcula limites entre as linhas
bounds = []

first_gap = line_centers[1] - line_centers[0]
bounds.append(
    max(0, int(line_centers[0] - first_gap / 2))
)

for a, b in zip(line_centers[:-1], line_centers[1:]):
    bounds.append(round((a + b) / 2))

last_gap = line_centers[-1] - line_centers[-2]
bounds.append(
    min(img.height, int(line_centers[-1] + last_gap / 2))
)

records = []

for i in range(26):

    y1 = bounds[i]
    y2 = bounds[i + 1]

    crop = img.crop(
        (X_LEFT, y1, X_RIGHT, y2)
    )

    line_number = i + 1
    filename = f"1QS_III_L{line_number:02d}.png"
    path = out_dir / filename

    crop.save(path)

    records.append({
        "manuscript": "1QS",
        "column": "III",
        "line": line_number,
        "filename": filename,
        "x1": X_LEFT,
        "y1": y1,
        "x2": X_RIGHT,
        "y2": y2,
        "diplomatic_text": "",
        "human_verified": False,
        "notes": ""
    })

csv_path = metadata_dir / "1QS_III_lines.csv"

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=records[0].keys()
    )
    writer.writeheader()
    writer.writerows(records)

print("Concluído.")
print("Linhas:", out_dir)
print("Manifesto:", csv_path)

In [ ]:
from PIL import Image, ImageOps, ImageDraw, ImageFont
from pathlib import Path
from google.colab import files
import math

lines_dir = Path("/content/Qumran_Analytics/1QS/lines/III")
out_path = Path("/content/1QS_III_contact_sheet.jpg")

files_list = sorted(lines_dir.glob("1QS_III_L*.png"))

thumbs = []

for path in files_list:
    img = Image.open(path).convert("RGB")

    # Reduz largura apenas para a prancha
    target_w = 900
    target_h = round(img.height * target_w / img.width)

    img = img.resize(
        (target_w, target_h),
        Image.Resampling.LANCZOS
    )

    canvas = Image.new(
        "RGB",
        (target_w, target_h + 35),
        "white"
    )

    canvas.paste(img, (0, 35))

    draw = ImageDraw.Draw(canvas)
    draw.text(
        (10, 8),
        path.stem,
        fill="black"
    )

    thumbs.append(canvas)

# uma linha por faixa, para facilitar a leitura
sheet_w = max(x.width for x in thumbs)
sheet_h = sum(x.height for x in thumbs)

sheet = Image.new(
    "RGB",
    (sheet_w, sheet_h),
    "white"
)

y = 0
for img in thumbs:
    sheet.paste(img, (0, y))
    y += img.height

sheet.save(
    out_path,
    "JPEG",
    quality=90,
    optimize=True
)

print(out_path)
files.download(out_path)

In [ ]:
from google.colab import files

files.download(
    "/content/Qumran_Analytics/1QS/metadata/1QS_III_lines.csv"
)

In [ ]:
from PIL import Image, ImageOps, ImageDraw
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

source = Path(
    "/content/Qumran_Analytics/1QS/columns/1QS_col_III.png"
)

img = Image.open(source).convert("RGB")

# Recorte horizontal apenas para excluir as colunas vizinhas
X1 = 90
X2 = 1190
roi = img.crop((X1, 0, X2, img.height))

# Cinza
gray = ImageOps.grayscale(roi)
a = np.array(gray)

# Pixels escuros recebem maior peso
ink = 255 - a

# Soma da "tinta" por linha horizontal
profile = ink.sum(axis=1)

# Suavização
window = 15
smooth = np.convolve(
    profile,
    np.ones(window) / window,
    mode="same"
)

plt.figure(figsize=(16, 5))
plt.plot(smooth)
plt.gca().invert_xaxis()
plt.title("Projeção horizontal da tinta — 1QS III")
plt.xlabel("Coordenada Y")
plt.ylabel("Quantidade de tinta")
plt.grid()
plt.show()

In [ ]:
from PIL import Image
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d

# ==========================================
# 1. CARREGA A COLUNA III ORIGINAL
# ==========================================

source = Path(
    "/content/Qumran_Analytics/1QS/columns/1QS_col_III.png"
)

img = np.array(Image.open(source).convert("RGB"))

# Exclui visualmente partes das colunas vizinhas
X1 = 90
X2 = 1190

roi = img[:, X1:X2]

gray = cv2.cvtColor(roi, cv2.COLOR_RGB2GRAY)

# ==========================================
# 2. CORRIGE VARIAÇÃO DO PERGAMINHO
# ==========================================

# Estima o fundo lentamente variável
background = cv2.GaussianBlur(
    gray,
    (0, 0),
    sigmaX=25,
    sigmaY=25
)

# Normalização do fundo
normalized = cv2.divide(
    gray,
    background,
    scale=255
)

# ==========================================
# 3. SEPARA TINTA DO PERGAMINHO
# ==========================================

_, ink_mask = cv2.threshold(
    normalized,
    0,
    255,
    cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
)

# Pequena limpeza de ruído
kernel = np.ones((2, 2), np.uint8)

ink_mask = cv2.morphologyEx(
    ink_mask,
    cv2.MORPH_OPEN,
    kernel
)

# ==========================================
# 4. PROJEÇÃO HORIZONTAL
# ==========================================

profile = ink_mask.sum(axis=1) / 255

smooth = gaussian_filter1d(
    profile.astype(float),
    sigma=4
)

# ==========================================
# 5. DETECÇÃO AUTOMÁTICA DOS PICOS
# ==========================================

prominence = 0.08 * (
    smooth.max() - smooth.min()
)

peaks, properties = find_peaks(
    smooth,
    distance=45,
    prominence=prominence
)

print("Linhas candidatas encontradas:", len(peaks))
print("Coordenadas Y:")
print(peaks)

# ==========================================
# 6. GRÁFICO DE DIAGNÓSTICO
# ==========================================

plt.figure(figsize=(16, 6))

plt.plot(
    np.arange(len(smooth)),
    smooth,
    label="Projeção suavizada"
)

plt.scatter(
    peaks,
    smooth[peaks],
    marker="x",
    s=80,
    label="Linhas detectadas"
)

for p in peaks:
    plt.axvline(
        p,
        alpha=0.25
    )

plt.xlabel("Coordenada Y da imagem")
plt.ylabel("Quantidade de tinta")
plt.title(
    "Detecção automática das linhas — 1QS III"
)
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# Converte ROI para PIL para desenhar diagnóstico
diagnostic = Image.fromarray(roi.copy())

from PIL import ImageDraw

draw = ImageDraw.Draw(diagnostic)

for i, y in enumerate(peaks, start=1):

    draw.line(
        [(0, int(y)), (diagnostic.width, int(y))],
        fill=(255, 0, 0),
        width=2
    )

    draw.text(
        (10, max(0, int(y) - 18)),
        f"L{i:02d}",
        fill=(255, 0, 0)
    )

diagnostic_path = (
    "/content/1QS_III_lines_detected.jpg"
)

diagnostic.save(
    diagnostic_path,
    "JPEG",
    quality=92
)

print(diagnostic_path)

In [ ]:
from google.colab import files

files.download(
    "/content/1QS_III_lines_detected.jpg"
)

In [ ]:
from PIL import Image, ImageDraw
from pathlib import Path
import numpy as np
import cv2
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d

source = Path(
    "/content/Qumran_Analytics/1QS/columns/1QS_col_III.png"
)

img_pil = Image.open(source).convert("RGB")
img = np.array(img_pil)

# -------------------------------------
# 1. DETECÇÃO DAS LINHAS LONGAS
# -------------------------------------

X1 = 90
X2 = 1190

roi = img[:, X1:X2]

gray = cv2.cvtColor(roi, cv2.COLOR_RGB2GRAY)

background = cv2.GaussianBlur(
    gray, (0, 0), sigmaX=25, sigmaY=25
)

normalized = cv2.divide(
    gray, background, scale=255
)

_, mask = cv2.threshold(
    normalized,
    0,
    255,
    cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
)

kernel = np.ones((2, 2), np.uint8)

mask = cv2.morphologyEx(
    mask,
    cv2.MORPH_OPEN,
    kernel
)

profile = mask.sum(axis=1) / 255

smooth = gaussian_filter1d(
    profile.astype(float),
    sigma=4
)

prominence = 0.08 * (
    smooth.max() - smooth.min()
)

peaks, _ = find_peaks(
    smooth,
    distance=45,
    prominence=prominence
)

print("Linhas principais:", len(peaks))

# -------------------------------------
# 2. PROCURA GRANDES ESPAÇOS
# -------------------------------------

gaps = np.diff(peaks)
median_gap = np.median(gaps)

print("Espaçamento mediano:", median_gap)

candidate_extra = []

for i, gap in enumerate(gaps):

    # espaço grande demais para apenas uma linha
    if gap > 1.55 * median_gap:

        y_start = peaks[i] + 20
        y_end = peaks[i + 1] - 20

        # usa praticamente toda a largura da coluna
        gap_roi = img[y_start:y_end, 40:1280]

        gap_gray = cv2.cvtColor(
            gap_roi,
            cv2.COLOR_RGB2GRAY
        )

        bg = cv2.GaussianBlur(
            gap_gray,
            (0, 0),
            sigmaX=20,
            sigmaY=20
        )

        norm = cv2.divide(
            gap_gray,
            bg,
            scale=255
        )

        _, gap_mask = cv2.threshold(
            norm,
            0,
            255,
            cv2.THRESH_BINARY_INV
            + cv2.THRESH_OTSU
        )

        gap_profile = (
            gap_mask.sum(axis=1) / 255
        )

        gap_smooth = gaussian_filter1d(
            gap_profile.astype(float),
            sigma=3
        )

        if gap_smooth.max() > 0:

            local_y = int(
                np.argmax(gap_smooth)
            )

            global_y = (
                y_start + local_y
            )

            candidate_extra.append(
                global_y
            )

# -------------------------------------
# 3. JUNTA TODAS AS LINHAS
# -------------------------------------

all_lines = sorted(
    list(peaks)
    + candidate_extra
)

print(
    "Linhas após busca de linhas curtas:",
    len(all_lines)
)

print(all_lines)

# -------------------------------------
# 4. DESENHA DIAGNÓSTICO
# -------------------------------------

diagnostic = img_pil.copy()
draw = ImageDraw.Draw(diagnostic)

for i, y in enumerate(
    all_lines,
    start=1
):

    draw.line(
        [(0, int(y)),
         (diagnostic.width, int(y))],
        fill=(255, 0, 0),
        width=2
    )

    draw.text(
        (5, max(0, int(y) - 18)),
        f"L{i:02d}",
        fill=(255, 0, 0)
    )

output = (
    "/content/"
    "1QS_III_lines_detected_v2.jpg"
)

diagnostic.save(
    output,
    "JPEG",
    quality=94
)

print(output)

In [ ]:
from google.colab import files

files.download(
    "/content/"
    "1QS_III_lines_detected_v2.jpg"
)

In [ ]:
from PIL import Image, ImageDraw
from pathlib import Path
import numpy as np
import cv2
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d

# ==========================================
# IMAGEM ORIGINAL
# ==========================================

source = Path(
    "/content/Qumran_Analytics/1QS/columns/1QS_col_III.png"
)

img_pil = Image.open(source).convert("RGB")
img = np.array(img_pil)

# Área principal da coluna
X1 = 90
X2 = 1190

roi = img[:, X1:X2]

gray = cv2.cvtColor(
    roi,
    cv2.COLOR_RGB2GRAY
)

# ==========================================
# CORREÇÃO DO FUNDO
# ==========================================

background = cv2.GaussianBlur(
    gray,
    (0, 0),
    sigmaX=25,
    sigmaY=25
)

normalized = cv2.divide(
    gray,
    background,
    scale=255
)

_, mask = cv2.threshold(
    normalized,
    0,
    255,
    cv2.THRESH_BINARY_INV +
    cv2.THRESH_OTSU
)

mask = cv2.morphologyEx(
    mask,
    cv2.MORPH_OPEN,
    np.ones((2, 2), np.uint8)
)

# ==========================================
# PROJEÇÃO PRINCIPAL
# ==========================================

profile = mask.sum(axis=1) / 255

smooth = gaussian_filter1d(
    profile.astype(float),
    sigma=4
)

prominence = 0.08 * (
    smooth.max() - smooth.min()
)

peaks, _ = find_peaks(
    smooth,
    distance=45,
    prominence=prominence
)

print("Linhas principais:", len(peaks))

# ==========================================
# DETECÇÃO DE LINHAS CURTAS
# ==========================================

gaps = np.diff(peaks)
median_gap = np.median(gaps)

print("Espaçamento mediano:", median_gap)

extras = []

for i, gap in enumerate(gaps):

    if gap > 1.55 * median_gap:

        # procura APENAS na região central
        # do espaço entre as duas linhas
        y_start = int(
            peaks[i] + 0.25 * gap
        )

        y_end = int(
            peaks[i + 1] - 0.25 * gap
        )

        gap_roi = img[
            y_start:y_end,
            40:1280
        ]

        gap_gray = cv2.cvtColor(
            gap_roi,
            cv2.COLOR_RGB2GRAY
        )

        bg = cv2.GaussianBlur(
            gap_gray,
            (0, 0),
            sigmaX=20,
            sigmaY=20
        )

        norm = cv2.divide(
            gap_gray,
            bg,
            scale=255
        )

        _, gap_mask = cv2.threshold(
            norm,
            0,
            255,
            cv2.THRESH_BINARY_INV +
            cv2.THRESH_OTSU
        )

        gap_profile = (
            gap_mask.sum(axis=1) / 255
        )

        gap_smooth = gaussian_filter1d(
            gap_profile.astype(float),
            sigma=2
        )

        local_y = int(
            np.argmax(gap_smooth)
        )

        global_y = (
            y_start + local_y
        )

        extras.append(global_y)

# ==========================================
# TODAS AS LINHAS
# ==========================================

all_lines = sorted(
    list(peaks) + extras
)

print(
    "Linhas finais:",
    len(all_lines)
)

print(
    "Coordenadas Y:",
    all_lines
)

# ==========================================
# DIAGNÓSTICO
# ==========================================

diagnostic = img_pil.copy()
draw = ImageDraw.Draw(diagnostic)

for i, y in enumerate(
    all_lines,
    start=1
):

    draw.line(
        [
            (0, int(y)),
            (diagnostic.width, int(y))
        ],
        fill=(255, 0, 0),
        width=2
    )

    draw.text(
        (
            5,
            max(0, int(y) - 18)
        ),
        f"L{i:02d}",
        fill=(255, 0, 0)
    )

output = (
    "/content/"
    "1QS_III_lines_detected_v3.jpg"
)

diagnostic.save(
    output,
    "JPEG",
    quality=94
)

print(output)

In [ ]:
from google.colab import files

files.download(
    "/content/1QS_III_lines_detected_v3.jpg"
)

In [ ]:
from PIL import Image
from pathlib import Path
import pandas as pd
import hashlib

# ============================================================
# 1. CAMINHOS
# ============================================================

source = Path(
    "/content/Qumran_Analytics/1QS/columns/1QS_col_III.png"
)

metadata_dir = Path(
    "/content/Qumran_Analytics/1QS/metadata"
)

lines_dir = Path(
    "/content/Qumran_Analytics/1QS/lines/III_validated"
)

metadata_dir.mkdir(parents=True, exist_ok=True)
lines_dir.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. ABRE A COLUNA III
# ============================================================

img = Image.open(source).convert("RGB")

print("Imagem:", img.size)

# ============================================================
# 3. CENTROS DAS 26 LINHAS VALIDADAS
# ============================================================
# L12 foi corrigida manualmente após inspeção visual.
# Não estamos forçando artificialmente o total:
# 25 foram encontradas pela projeção principal e uma linha curta
# foi recuperada na etapa complementar.

centers = [
    168,
    228,
    282,
    349,
    404,
    467,
    536,
    606,
    659,
    719,
    785,
    864,   # L12 - linha curta corrigida
    897,
    960,
    1019,
    1084,
    1142,
    1220,
    1285,
    1351,
    1413,
    1470,
    1543,
    1604,
    1676,
    1742
]

assert len(centers) == 26

# ============================================================
# 4. CALCULA OS LIMITES ENTRE AS LINHAS
# ============================================================

bounds = []

first_gap = centers[1] - centers[0]

bounds.append(
    max(
        0,
        round(centers[0] - first_gap / 2)
    )
)

for a, b in zip(
    centers[:-1],
    centers[1:]
):
    bounds.append(
        round((a + b) / 2)
    )

last_gap = centers[-1] - centers[-2]

bounds.append(
    min(
        img.height,
        round(
            centers[-1] + last_gap / 2
        )
    )
)

# Região horizontal útil
X1 = 60
X2 = 1260

# pequena margem vertical para não cortar hastes
OVERLAP = 8

# ============================================================
# 5. GERA OS 26 RECORTES VALIDADOS
# ============================================================

records = []

for i in range(26):

    line_no = i + 1

    y1 = max(
        0,
        bounds[i] - OVERLAP
    )

    y2 = min(
        img.height,
        bounds[i + 1] + OVERLAP
    )

    crop = img.crop(
        (X1, y1, X2, y2)
    )

    filename = (
        f"1QS_III_L{line_no:02d}.png"
    )

    filepath = (
        lines_dir / filename
    )

    crop.save(filepath)

    sha256 = hashlib.sha256(
        filepath.read_bytes()
    ).hexdigest()

    note = ""

    if line_no == 12:
        note = (
            "Short right-aligned line; "
            "position corrected after visual QC."
        )

    records.append({
        "manuscript": "1QS",
        "column": "III",
        "line": line_no,
        "filename": filename,
        "center_y": centers[i],
        "x1": X1,
        "y1": y1,
        "x2": X2,
        "y2": y2,
        "segmentation_method":
            "automatic_projection_plus_human_QC",
        "segmentation_validated": True,
        "special_note": note,
        "diplomatic_text": pd.NA,
        "human_transcription_verified": False,
        "sha256": sha256
    })

# ============================================================
# 6. CRIA DATAFRAME COM TIPOS CORRETOS
# ============================================================

df = pd.DataFrame(records)

df["diplomatic_text"] = (
    df["diplomatic_text"]
    .astype("string")
)

df["human_transcription_verified"] = (
    df["human_transcription_verified"]
    .astype(bool)
)

# ============================================================
# 7. SALVA MANIFESTO VALIDADO
# ============================================================

validated_manifest = (
    metadata_dir /
    "1QS_III_lines_validated.csv"
)

df.to_csv(
    validated_manifest,
    index=False,
    encoding="utf-8-sig"
)

print("\nMANIFESTO VALIDADO CRIADO:")
print(validated_manifest)

print("\nTotal de linhas:")
print(len(df))

print("\nLinhas:")
print(
    df[
        [
            "line",
            "filename",
            "center_y",
            "segmentation_validated"
        ]
    ]
)

**[Public-release redaction]** Original research-log cell 17 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


**[Public-release redaction]** Original research-log cell 18 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


In [ ]:
from PIL import Image, ImageEnhance, ImageDraw
from pathlib import Path
from google.colab import files

lines_dir = Path(
    "/content/Qumran_Analytics/1QS/lines/III_validated"
)

targets = [
    ("III.2", lines_dir / "1QS_III_L02.png"),
    ("III.3", lines_dir / "1QS_III_L03.png")
]

panels = []

for label, path in targets:

    img = Image.open(path).convert("RGB")

    # Amplia sem alterar o arquivo original
    scale = 4

    enlarged = img.resize(
        (img.width * scale, img.height * scale),
        Image.Resampling.LANCZOS
    )

    # Versão levemente realçada apenas para inspeção
    enhanced = ImageEnhance.Contrast(
        enlarged
    ).enhance(1.35)

    enhanced = ImageEnhance.Sharpness(
        enhanced
    ).enhance(1.5)

    header = 70

    canvas = Image.new(
        "RGB",
        (enhanced.width, enhanced.height + header),
        "white"
    )

    canvas.paste(
        enhanced,
        (0, header)
    )

    draw = ImageDraw.Draw(canvas)

    draw.text(
        (20, 20),
        label,
        fill="black"
    )

    panels.append(canvas)

width = max(p.width for p in panels)
height = sum(p.height for p in panels)

sheet = Image.new(
    "RGB",
    (width, height),
    "white"
)

y = 0

for p in panels:
    sheet.paste(p, (0, y))
    y += p.height

output = "/content/1QS_III_differences_review.png"

sheet.save(output)

print(output)

files.download(output)

**[Public-release redaction]** Original research-log cell 20 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


**[Public-release redaction]** Original research-log cell 21 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


In [ ]:
from PIL import Image, ImageDraw
from pathlib import Path
import pandas as pd

lines_dir = Path(
    "/content/Qumran_Analytics/1QS/lines/III_validated"
)

out_dir = Path(
    "/content/Qumran_Analytics/1QS/blind_validation"
)
out_dir.mkdir(parents=True, exist_ok=True)

selected_lines = range(6, 11)

# -----------------------------
# CSV cego
# -----------------------------

records = []

for line_no in selected_lines:

    filename = f"1QS_III_L{line_no:02d}.png"

    records.append({
        "manuscript": "1QS",
        "column": "III",
        "line": line_no,
        "image_file": filename,
        "image_transcription_preliminary": "",
        "transcription_source": "image_only",
        "AI_assisted": True,
        "researcher_reviewed": False,
        "expert_verified": False,
        "transcription_frozen": False,
        "notes": ""
    })

blind_df = pd.DataFrame(records)

blind_csv = (
    out_dir /
    "1QS_III_lines_06_10_blind.csv"
)

blind_df.to_csv(
    blind_csv,
    index=False,
    encoding="utf-8-sig"
)

# -----------------------------
# Prancha das 5 linhas
# -----------------------------

panels = []

for line_no in selected_lines:

    path = (
        lines_dir /
        f"1QS_III_L{line_no:02d}.png"
    )

    img = Image.open(path).convert("RGB")

    scale = 3

    enlarged = img.resize(
        (
            img.width * scale,
            img.height * scale
        ),
        Image.Resampling.LANCZOS
    )

    header = 55

    panel = Image.new(
        "RGB",
        (
            enlarged.width,
            enlarged.height + header
        ),
        "white"
    )

    panel.paste(
        enlarged,
        (0, header)
    )

    draw = ImageDraw.Draw(panel)

    draw.text(
        (20, 15),
        f"1QS III.{line_no}",
        fill="black"
    )

    panels.append(panel)

width = max(
    panel.width
    for panel in panels
)

height = sum(
    panel.height
    for panel in panels
)

sheet = Image.new(
    "RGB",
    (width, height),
    "white"
)

y = 0

for panel in panels:

    sheet.paste(
        panel,
        (0, y)
    )

    y += panel.height

sheet_path = (
    out_dir /
    "1QS_III_06_10_blind_sheet.png"
)

sheet.save(sheet_path)

print("CSV:")
print(blind_csv)

print("\nImagem:")
print(sheet_path)

In [ ]:
from google.colab import files

files.download(
    "/content/Qumran_Analytics/1QS/"
    "blind_validation/"
    "1QS_III_06_10_blind_sheet.png"
)

**[Public-release redaction]** Original research-log cell 24 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


**[Public-release redaction]** Original research-log cell 25 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


In [ ]:
import pandas as pd
from pathlib import Path

frozen_path = Path(
    "/content/Qumran_Analytics/1QS/"
    "blind_validation/"
    "1QS_III_lines_06_10_FROZEN.csv"
)

etcbc_path = Path("/content/1QS_palavras.csv")

frozen = pd.read_csv(frozen_path)
etcbc = pd.read_csv(etcbc_path)

etcbc_subset = etcbc[
    (etcbc["fragmento_coluna"].astype(str) == "3")
    &
    (etcbc["linha"].between(6, 10))
].copy()

print("Diagnóstico por linha:\n")

for _, row in frozen.iterrows():
    line_no = int(row["line"])
    image_text = str(row["image_transcription_preliminary"])
    image_words = image_text.split()

    etcbc_line = etcbc_subset[
        etcbc_subset["linha"] == line_no
    ].copy()

    print(f"Linha III.{line_no}")
    print(f"  Palavras da imagem: {len(image_words)}")
    print(f"  Tokens ETCBC:      {len(etcbc_line)}")
    print(f"  Máx. coberto com group_size=4: {len(image_words) * 4}")
    print("-" * 50)

**[Public-release redaction]** Original research-log cell 27 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


In [ ]:
from google.colab import files

files.download(
    "/content/Qumran_Analytics/1QS/blind_validation/"
    "1QS_III_06_10_many_to_many_ETCBC.csv"
)

**[Public-release redaction]** Original research-log cell 29 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


In [ ]:
import pandas as pd
from pathlib import Path

alignment_path = Path(
    "/content/Qumran_Analytics/1QS/"
    "blind_validation/"
    "1QS_III_06_10_CONTINUOUS_ALIGNMENT.csv"
)

df = pd.read_csv(alignment_path)

# Mantém apenas divergências
diff = df[
    df["operation"] != "MATCH"
].copy()

# Numeração das divergências
diff = diff.reset_index(drop=False)
diff["review_id"] = [
    f"D{i:03d}"
    for i in range(1, len(diff) + 1)
]

# Campos para revisão posterior
diff["review_class"] = ""
diff["image_supports"] = ""
diff["review_confidence"] = ""
diff["review_note"] = ""
diff["researcher_reviewed"] = False

# Organiza colunas
cols = [
    "review_id",
    "operation",
    "image_line",
    "etcbc_line",
    "image_char",
    "etcbc_char",
    "word_node",
    "query_index",
    "reference_index",
    "review_class",
    "image_supports",
    "review_confidence",
    "review_note",
    "researcher_reviewed"
]

diff = diff[
    [c for c in cols if c in diff.columns]
]

output = Path(
    "/content/Qumran_Analytics/1QS/"
    "blind_validation/"
    "1QS_III_06_10_DIVERGENCES_FOR_REVIEW.csv"
)

diff.to_csv(
    output,
    index=False,
    encoding="utf-8-sig"
)

print("Total de divergências:", len(diff))

print("\nTipos:")
print(diff["operation"].value_counts())

print("\nArquivo criado:")
print(output)

display(diff)

In [ ]:
from PIL import Image, ImageEnhance, ImageDraw
from pathlib import Path
from google.colab import files

lines_dir = Path(
    "/content/Qumran_Analytics/1QS/lines/III_validated"
)

target_lines = [6, 8, 10]

panels = []

for line_no in target_lines:

    path = lines_dir / f"1QS_III_L{line_no:02d}.png"

    img = Image.open(path).convert("RGB")

    # ampliação para inspeção paleográfica
    scale = 5

    enlarged = img.resize(
        (img.width * scale, img.height * scale),
        Image.Resampling.LANCZOS
    )

    # realce APENAS para inspeção, não substitui o original
    enhanced = ImageEnhance.Contrast(
        enlarged
    ).enhance(1.30)

    enhanced = ImageEnhance.Sharpness(
        enhanced
    ).enhance(1.35)

    header = 80

    panel = Image.new(
        "RGB",
        (enhanced.width, enhanced.height + header),
        "white"
    )

    panel.paste(enhanced, (0, header))

    draw = ImageDraw.Draw(panel)

    draw.text(
        (20, 20),
        f"1QS III.{line_no}",
        fill="black"
    )

    panels.append(panel)

width = max(p.width for p in panels)
height = sum(p.height for p in panels)

sheet = Image.new(
    "RGB",
    (width, height),
    "white"
)

y = 0

for panel in panels:
    sheet.paste(panel, (0, y))
    y += panel.height

output = "/content/1QS_III_06_08_10_review.png"

sheet.save(output)

print(output)

files.download(output)

**[Public-release redaction]** Original research-log cell 32 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


In [ ]:
from PIL import Image, ImageDraw
from pathlib import Path
import pandas as pd

lines_dir = Path(
    "/content/Qumran_Analytics/1QS/lines/III_validated"
)

base_dir = Path(
    "/content/Qumran_Analytics/1QS/blind_validation"
)

base_dir.mkdir(
    parents=True,
    exist_ok=True
)

batches = {
    "11_15": range(11, 16),
    "16_20": range(16, 21),
    "21_26": range(21, 27)
}

for batch_name, selected_lines in batches.items():

    # ==========================================
    # CSV PARA TRANSCRIÇÃO
    # ==========================================

    records = []

    for line_no in selected_lines:

        records.append({
            "manuscript": "1QS",
            "column": "III",
            "line": line_no,

            "image_file":
                f"1QS_III_L{line_no:02d}.png",

            "image_transcription_preliminary": "",

            "transcription_source":
                "image_only_reference_hidden",

            "AI_assisted": True,

            "researcher_reviewed": False,

            "expert_verified": False,

            "transcription_frozen": False,

            "notes": ""
        })

    df = pd.DataFrame(records)

    csv_path = (
        base_dir /
        f"1QS_III_lines_{batch_name}_blind.csv"
    )

    df.to_csv(
        csv_path,
        index=False,
        encoding="utf-8-sig"
    )

    # ==========================================
    # PRANCHA DE IMAGENS
    # ==========================================

    panels = []

    for line_no in selected_lines:

        path = (
            lines_dir /
            f"1QS_III_L{line_no:02d}.png"
        )

        img = Image.open(
            path
        ).convert("RGB")

        # ampliação para leitura
        scale = 3

        enlarged = img.resize(
            (
                img.width * scale,
                img.height * scale
            ),
            Image.Resampling.LANCZOS
        )

        header = 55

        panel = Image.new(
            "RGB",
            (
                enlarged.width,
                enlarged.height + header
            ),
            "white"
        )

        panel.paste(
            enlarged,
            (0, header)
        )

        draw = ImageDraw.Draw(panel)

        draw.text(
            (20, 15),
            f"1QS III.{line_no}",
            fill="black"
        )

        panels.append(panel)

    width = max(
        p.width
        for p in panels
    )

    height = sum(
        p.height
        for p in panels
    )

    sheet = Image.new(
        "RGB",
        (width, height),
        "white"
    )

    y = 0

    for panel in panels:

        sheet.paste(
            panel,
            (0, y)
        )

        y += panel.height

    sheet_path = (
        base_dir /
        f"1QS_III_{batch_name}_blind_sheet.png"
    )

    sheet.save(sheet_path)

    print("\nLOTE:", batch_name)
    print("CSV:", csv_path)
    print("Imagem:", sheet_path)

In [ ]:
from google.colab import files

files.download(
    "/content/Qumran_Analytics/1QS/"
    "blind_validation/"
    "1QS_III_11_15_blind_sheet.png"
)

In [ ]:
from PIL import Image, ImageDraw, ImageEnhance
from pathlib import Path

source = Path(
    "/content/Qumran_Analytics/1QS/"
    "columns/1QS_col_III.png"
)

img = Image.open(source).convert("RGB")

# Janelas revisadas após inspeção visual
windows = {
    11: (60, 755, 1260, 822),

    # linha curta, alinhada à direita
    12: (800, 825, 1260, 878),

    13: (60, 880, 1260, 930),
    14: (60, 935, 1260, 990),
    15: (60, 995, 1260, 1055)
}

panels = []

for line_no, (x1, y1, x2, y2) in windows.items():

    crop = img.crop(
        (x1, y1, x2, y2)
    )

    crop = crop.resize(
        (
            crop.width * 2,
            crop.height * 2
        ),
        Image.Resampling.LANCZOS
    )

    # realce somente para inspeção
    crop = ImageEnhance.Contrast(
        crop
    ).enhance(1.15)

    canvas_width = 2400

    panel = Image.new(
        "RGB",
        (
            canvas_width,
            crop.height + 35
        ),
        "white"
    )

    # L12 é curta e está do lado direito
    if line_no == 12:
        xpos = canvas_width - crop.width
    else:
        xpos = 0

    panel.paste(
        crop,
        (xpos, 35)
    )

    draw = ImageDraw.Draw(panel)

    draw.text(
        (8, 8),
        f"1QS III.{line_no}",
        fill="black"
    )

    panels.append(panel)

sheet = Image.new(
    "RGB",
    (
        2400,
        sum(p.height for p in panels)
    ),
    "white"
)

y = 0

for panel in panels:

    sheet.paste(
        panel,
        (0, y)
    )

    y += panel.height

output = (
    "/content/Qumran_Analytics/1QS/"
    "blind_validation/"
    "1QS_III_11_15_blind_sheet_CORRECTED.png"
)

sheet.save(output)

print(output)

In [ ]:
from PIL import Image
from pathlib import Path

source = Path(
    "/content/Qumran_Analytics/1QS/columns/1QS_col_III.png"
)

img = Image.open(source).convert("RGB")

out_dir = Path(
    "/content/Qumran_Analytics/1QS/"
    "blind_validation/11_15_corrected"
)
out_dir.mkdir(parents=True, exist_ok=True)

windows = {
    11: (60, 755, 1260, 822),
    12: (800, 825, 1260, 878),
    13: (60, 880, 1260, 930),
    14: (60, 935, 1260, 990),
    15: (60, 995, 1260, 1055)
}

for line_no, box in windows.items():
    crop = img.crop(box)

    path = out_dir / f"1QS_III_L{line_no:02d}_corrected.png"
    crop.save(path)

    print(line_no, path)

print("\nRecortes corrigidos criados.")

**[Public-release redaction]** Original research-log cell 37 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


**[Public-release redaction]** Original research-log cell 38 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

log_path = Path(
    "/content/Qumran_Analytics/1QS/"
    "blind_validation/METHODOLOGICAL_LOG.csv"
)

entry = pd.DataFrame([{
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "1QS_III_06_10",
    "status": "INVALIDATED_FOR_METRIC_USE",
    "reason": (
        "Post-freeze comparison showed that several preliminary "
        "transcriptions extended beyond their visual line boundaries, "
        "causing duplicated text across batches. The image data are "
        "preserved, but character-level metrics from this batch must "
        "not be reported as validation results."
    ),
    "action": (
        "Re-transcribe III.6-10 later from individually quality-controlled "
        "line crops. Do not overwrite the original frozen files."
    )
}])

if log_path.exists():
    old = pd.read_csv(log_path)
    log = pd.concat([old, entry], ignore_index=True)
else:
    log = entry

log.to_csv(
    log_path,
    index=False,
    encoding="utf-8-sig"
)

print(log_path)
display(log.tail())

**[Public-release redaction]** Original research-log cell 40 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


In [ ]:
from google.colab import files

files.download(
    "/content/Qumran_Analytics/1QS/"
    "blind_validation/"
    "1QS_III_16_20_blind_sheet.png"
)

In [ ]:
windows = {
    16: (60, 1045, 1260, 1115),
    17: (60, 1110, 1260, 1180),
    18: (60, 1180, 1260, 1250),
    19: (60, 1245, 1260, 1320),
    20: (60, 1315, 1260, 1385)
}

In [ ]:
from PIL import Image
from pathlib import Path

source = Path(
    "/content/Qumran_Analytics/1QS/columns/1QS_col_III.png"
)

img = Image.open(source).convert("RGB")

out_dir = Path(
    "/content/Qumran_Analytics/1QS/"
    "blind_validation/16_20_corrected"
)

out_dir.mkdir(parents=True, exist_ok=True)

windows = {
    16: (60, 1045, 1260, 1115),
    17: (60, 1110, 1260, 1180),
    18: (60, 1180, 1260, 1250),
    19: (60, 1245, 1260, 1320),
    20: (60, 1315, 1260, 1385)
}

for line_no, box in windows.items():

    crop = img.crop(box)

    path = (
        out_dir /
        f"1QS_III_L{line_no:02d}_corrected.png"
    )

    crop.save(path)

    print(line_no, path)

print("Recortes III.16–20 corrigidos.")

**[Public-release redaction]** Original research-log cell 44 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


In [ ]:
!rm -rf /content/kraken_env
!pip -q install virtualenv

In [ ]:
!python -m virtualenv \
    --system-site-packages \
    /content/kraken_env

In [ ]:
!/content/kraken_env/bin/python -m pip --version

In [ ]:
!/content/kraken_env/bin/python -m pip install -U kraken

In [ ]:
!which kraken
!kraken --help | head -40

In [ ]:
from pathlib import Path

column_path = Path(
    "/content/Qumran_Analytics/1QS/columns/1QS_col_III.png"
)

segmentation_path = Path(
    "/content/Qumran_Analytics/1QS/"
    "automatic_segmentation/1QS_III_kraken.json"
)

segmentation_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

print("Entrada:", column_path)
print("Saída:", segmentation_path)
print("Imagem existe:", column_path.exists())

In [ ]:
!kraken \
    -i "$column_path" "$segmentation_path" \
    segment -bl -d horizontal-rl

In [ ]:
!pip install -q --upgrade --no-deps "safetensors==0.8.0"

In [ ]:
import safetensors
import transformers
import kraken

print("safetensors:", safetensors.__version__)
print("transformers:", transformers.__version__)
print("kraken:", kraken.__version__)

In [ ]:
from importlib.metadata import version

print("kraken:", version("kraken"))

In [ ]:
from pathlib import Path

column_path = Path(
    "/content/Qumran_Analytics/1QS/columns/1QS_col_III.png"
)

segmentation_path = Path(
    "/content/Qumran_Analytics/1QS/"
    "automatic_segmentation/1QS_III_kraken.json"
)

segmentation_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

print("Imagem existe:", column_path.exists())
print("Entrada:", column_path)
print("Saída:", segmentation_path)

In [ ]:
!kraken \
    -i "$column_path" "$segmentation_path" \
    segment -bl -d horizontal-rl

In [ ]:
print("JSON criado:", segmentation_path.exists())

if segmentation_path.exists():
    print(
        "Tamanho:",
        segmentation_path.stat().st_size,
        "bytes"
    )

In [ ]:
import json

with open(
    segmentation_path,
    "r",
    encoding="utf-8"
) as f:
    seg = json.load(f)

print(seg.keys())

lines = seg.get("lines", [])

print(
    "Linhas detectadas pelo Kraken:",
    len(lines)
)

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

segmentation_path = Path(
    "/content/Qumran_Analytics/1QS/"
    "automatic_segmentation/1QS_III_kraken.json"
)

with open(segmentation_path, "r", encoding="utf-8") as f:
    seg = json.load(f)

lines = seg["lines"]

print("Kraken:", len(lines), "linhas")

# Referência humana validada da coluna III
human_centers = np.array([
    168, 228, 282, 349, 404, 467, 536,
    606, 659, 719, 785, 864, 897,
    960, 1019, 1084, 1142, 1220,
    1285, 1351, 1413, 1470, 1543,
    1604, 1676, 1742
])

# ------------------------------------------------------------
# Extrai Y médio das baselines do Kraken
# ------------------------------------------------------------

kraken_rows = []

for i, line in enumerate(lines, start=1):

    baseline = line.get("baseline", [])

    if not baseline:
        continue

    ys = [p[1] for p in baseline]

    center_y = float(np.median(ys))

    xs = [p[0] for p in baseline]

    kraken_rows.append({
        "kraken_id": i,
        "center_y": center_y,
        "x_start": min(xs),
        "x_end": max(xs),
        "baseline_width": max(xs) - min(xs)
    })

kdf = pd.DataFrame(kraken_rows)

kdf = kdf.sort_values("center_y").reset_index(drop=True)

print("\nBaselines Kraken:")
display(kdf)

In [ ]:
# ------------------------------------------------------------
# Pareamento 1:1 Kraken x referência humana
# ------------------------------------------------------------

TOLERANCE = 35  # pixels

used_kraken = set()
matches = []

for human_no, hy in enumerate(human_centers, start=1):

    candidates = kdf[
        ~kdf["kraken_id"].isin(used_kraken)
    ].copy()

    candidates["distance"] = abs(
        candidates["center_y"] - hy
    )

    best = candidates.sort_values(
        "distance"
    ).iloc[0]

    if best["distance"] <= TOLERANCE:

        used_kraken.add(
            int(best["kraken_id"])
        )

        matches.append({
            "human_line": human_no,
            "human_y": hy,
            "kraken_id":
                int(best["kraken_id"]),
            "kraken_y":
                best["center_y"],
            "distance_px":
                best["distance"],
            "status": "MATCHED"
        })

    else:

        matches.append({
            "human_line": human_no,
            "human_y": hy,
            "kraken_id": None,
            "kraken_y": None,
            "distance_px": None,
            "status": "MISSED_BY_KRAKEN"
        })

match_df = pd.DataFrame(matches)

extras = kdf[
    ~kdf["kraken_id"].isin(
        used_kraken
    )
].copy()

print("================================")
print("COMPARAÇÃO DE SEGMENTAÇÃO")
print("================================")

print("Referência humana:", len(human_centers))
print("Kraken:", len(kdf))

print(
    "Linhas humanas encontradas:",
    (match_df["status"] == "MATCHED").sum()
)

print(
    "Linhas humanas perdidas:",
    (match_df["status"] == "MISSED_BY_KRAKEN").sum()
)

print(
    "Detecções Kraken extras:",
    len(extras)
)

print("\nEXTRAS:")
display(extras)

print("\nPAREAMENTO:")
display(match_df)

In [ ]:
from PIL import Image, ImageDraw

column_path = Path(
    "/content/Qumran_Analytics/1QS/"
    "columns/1QS_col_III.png"
)

img = Image.open(
    column_path
).convert("RGB")

overlay = img.copy()
draw = ImageDraw.Draw(overlay)

# Referência humana
for i, y in enumerate(
    human_centers,
    start=1
):
    draw.line(
        [(0, int(y)),
         (img.width, int(y))],
        fill=(0, 100, 255),
        width=2
    )

    draw.text(
        (5, int(y)-18),
        f"H{i:02d}",
        fill=(0, 100, 255)
    )

# Kraken
for _, row in kdf.iterrows():

    y = int(row["center_y"])

    draw.line(
        [
            (int(row["x_start"]), y),
            (int(row["x_end"]), y)
        ],
        fill=(255, 0, 0),
        width=3
    )

    draw.text(
        (
            int(row["x_start"]),
            y + 5
        ),
        f"K{int(row['kraken_id']):02d}",
        fill=(255, 0, 0)
    )

output = (
    "/content/"
    "1QS_III_HUMAN_vs_KRAKEN.jpg"
)

overlay.save(
    output,
    "JPEG",
    quality=94
)

print(output)

In [ ]:
from google.colab import files

files.download(
    "/content/1QS_III_HUMAN_vs_KRAKEN.jpg"
)

In [ ]:
from PIL import Image
from pathlib import Path

source = Path(
    "/content/Qumran_Analytics/1QS/"
    "columns/1QS_col_III.png"
)

img = Image.open(source).convert("RGB")

print("Dimensões originais:", img.size)

# ROI do corpo textual da coluna III
X1 = 70
X2 = 970
Y1 = 90
Y2 = 1790

roi = img.crop(
    (X1, Y1, X2, Y2)
)

roi_path = Path(
    "/content/Qumran_Analytics/1QS/"
    "automatic_segmentation/"
    "1QS_III_text_body.png"
)

roi_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

roi.save(roi_path)

print("ROI salva em:", roi_path)
print("Dimensões ROI:", roi.size)

In [ ]:
from pathlib import Path

roi_path = Path(
    "/content/Qumran_Analytics/1QS/"
    "automatic_segmentation/"
    "1QS_III_text_body.png"
)

seg_path = Path(
    "/content/Qumran_Analytics/1QS/"
    "automatic_segmentation/"
    "1QS_III_kraken_ROI.json"
)

!kraken \
    -i "$roi_path" "$seg_path" \
    segment -bl -d horizontal-rl

In [ ]:
import json

with open(
    seg_path,
    "r",
    encoding="utf-8"
) as f:
    seg_roi = json.load(f)

lines_roi = seg_roi.get("lines", [])

print(
    "Linhas detectadas dentro da ROI:",
    len(lines_roi)
)

In [ ]:
import json
import math
import statistics
import pandas as pd
from pathlib import Path

seg_path = Path(
    "/content/Qumran_Analytics/1QS/"
    "automatic_segmentation/1QS_III_kraken_ROI.json"
)

with open(seg_path, "r", encoding="utf-8") as f:
    seg = json.load(f)

lines = seg.get("lines", [])
print("Linhas brutas do Kraken:", len(lines))

rows = []

for i, line in enumerate(lines, start=1):
    baseline = line.get("baseline", [])
    boundary = line.get("boundary", [])

    if not baseline:
        continue

    xs = [pt[0] for pt in baseline]
    ys = [pt[1] for pt in baseline]

    x_min = min(xs)
    x_max = max(xs)
    y_mean = sum(ys) / len(ys)
    y_min = min(ys)
    y_max = max(ys)
    width = x_max - x_min

    rows.append({
        "kraken_id": i,
        "x_min": x_min,
        "x_max": x_max,
        "width": width,
        "y_mean": y_mean,
        "y_min": y_min,
        "y_max": y_max,
        "baseline": baseline,
        "boundary": boundary
    })

df = pd.DataFrame(rows).sort_values("y_mean").reset_index(drop=True)
df["y_gap_prev"] = df["y_mean"].diff()

display(df[["kraken_id", "x_min", "x_max", "width", "y_mean", "y_gap_prev"]])

valid_gaps = df["y_gap_prev"].dropna()
median_gap = valid_gaps.median()
print("Espaçamento mediano entre detecções:", median_gap)

In [ ]:
merge_tol = max(10, median_gap * 0.28)
print("Tolerância de agrupamento em Y:", merge_tol)

groups = []
current_group = [df.iloc[0].to_dict()]

for idx in range(1, len(df)):
    prev = df.iloc[idx - 1]
    curr = df.iloc[idx]

    if abs(curr["y_mean"] - prev["y_mean"]) <= merge_tol:
        current_group.append(curr.to_dict())
    else:
        groups.append(current_group)
        current_group = [curr.to_dict()]

groups.append(current_group)

print("Grupos encontrados:", len(groups))
print("Tamanhos dos grupos:", [len(g) for g in groups])

In [ ]:
candidate_pairs = []

for idx in range(1, len(df)):

    prev = df.iloc[idx - 1]
    curr = df.iloc[idx]

    gap = abs(
        curr["y_mean"]
        - prev["y_mean"]
    )

    if gap <= merge_tol:

        # interseção horizontal
        overlap = max(
            0,
            min(prev["x_max"], curr["x_max"])
            - max(prev["x_min"], curr["x_min"])
        )

        smallest_width = min(
            prev["width"],
            curr["width"]
        )

        if smallest_width > 0:
            overlap_ratio = (
                overlap / smallest_width
            )
        else:
            overlap_ratio = 0

        candidate_pairs.append({
            "kraken_A":
                int(prev["kraken_id"]),

            "kraken_B":
                int(curr["kraken_id"]),

            "y_A":
                round(prev["y_mean"], 2),

            "y_B":
                round(curr["y_mean"], 2),

            "y_gap":
                round(gap, 2),

            "x_A":
                f"{int(prev['x_min'])}-{int(prev['x_max'])}",

            "x_B":
                f"{int(curr['x_min'])}-{int(curr['x_max'])}",

            "width_A":
                int(prev["width"]),

            "width_B":
                int(curr["width"]),

            "horizontal_overlap_ratio":
                round(overlap_ratio, 3)
        })

pairs_df = pd.DataFrame(candidate_pairs)

display(pairs_df)

In [ ]:
import numpy as np
import pandas as pd

# Centros das 26 linhas validadas na imagem ORIGINAL
human_centers_original = np.array([
    168, 228, 282, 349, 404, 467, 536,
    606, 659, 719, 785, 864, 897,
    960, 1019, 1084, 1142, 1220,
    1285, 1351, 1413, 1470, 1543,
    1604, 1676, 1742
])

# Nossa ROI começou em Y = 90
ROI_Y1 = 90

human_centers_roi = (
    human_centers_original - ROI_Y1
)

assignments = []

for _, row in df.iterrows():

    distances = np.abs(
        human_centers_roi - row["y_mean"]
    )

    idx = int(np.argmin(distances))

    assignments.append({
        "kraken_id": int(row["kraken_id"]),
        "kraken_y": round(row["y_mean"], 2),
        "x_min": int(row["x_min"]),
        "x_max": int(row["x_max"]),
        "width": int(row["width"]),

        "nearest_human_line": idx + 1,
        "human_y": int(human_centers_roi[idx]),
        "distance_px": round(float(distances[idx]), 2)
    })

assignment_df = pd.DataFrame(assignments)

display(
    assignment_df.sort_values(
        ["nearest_human_line", "kraken_y"]
    )
)

In [ ]:
summary = (
    assignment_df
    .groupby("nearest_human_line")
    .size()
    .reindex(
        range(1, 27),
        fill_value=0
    )
    .reset_index(name="kraken_detections")
)

print("Mapeamento Kraken → 26 linhas humanas:")
display(summary)

print("\nLINHAS NÃO DETECTADAS:")
display(
    summary[
        summary["kraken_detections"] == 0
    ]
)

print("\nLINHAS COM MAIS DE UMA DETECÇÃO:")
display(
    summary[
        summary["kraken_detections"] > 1
    ]
)

In [ ]:
import numpy as np

# Centros originais, mas com III.11–20 atualizadas
human_centers_original = np.array([
    168, 228, 282, 349, 404,
    467, 536, 606, 659, 719,

    # III.11–15 revisadas
    789,   # 11
    852,   # 12
    905,   # 13
    963,   # 14
    1025,  # 15

    # III.16–20 revisadas
    1080,  # 16
    1145,  # 17
    1215,  # 18
    1283,  # 19
    1350,  # 20

    1413, 1470, 1543,
    1604, 1676, 1742
])

ROI_Y1 = 90
human_centers_roi = human_centers_original - ROI_Y1

In [ ]:
from scipy.optimize import linear_sum_assignment
import numpy as np
import pandas as pd

kraken_y = df["y_mean"].to_numpy()

# custo = distância vertical absoluta
cost = np.abs(
    human_centers_roi[:, None]
    - kraken_y[None, :]
)

human_idx, kraken_idx = linear_sum_assignment(cost)

global_matches = []

for h, k in zip(human_idx, kraken_idx):

    row = df.iloc[k]

    global_matches.append({
        "human_line": h + 1,
        "human_y": human_centers_roi[h],

        "kraken_id":
            int(row["kraken_id"]),

        "kraken_y":
            round(row["y_mean"], 2),

        "x_min":
            int(row["x_min"]),

        "x_max":
            int(row["x_max"]),

        "width":
            int(row["width"]),

        "distance_px":
            round(cost[h, k], 2)
    })

global_df = pd.DataFrame(global_matches)

used = set(kraken_idx)

unmatched_kraken = (
    df.iloc[
        [
            i for i in range(len(df))
            if i not in used
        ]
    ]
    .copy()
)

print("Linhas humanas pareadas:", len(global_df))
print("Detecções Kraken sem par:", len(unmatched_kraken))

print("\nMAIORES DISTÂNCIAS:")
display(
    global_df
    .sort_values(
        "distance_px",
        ascending=False
    )
    .head(10)
)

print("\nKRAKEN SEM PAR:")
display(
    unmatched_kraken[
        [
            "kraken_id",
            "y_mean",
            "x_min",
            "x_max",
            "width"
        ]
    ]
)

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# 1. FUNDE FRAGMENTOS DA MESMA LINHA
# ============================================================

Y_TOL = 20
MAX_HORIZONTAL_OVERLAP = 0.15

groups = []
current_group = [df.iloc[0].to_dict()]

for idx in range(1, len(df)):

    prev = df.iloc[idx - 1]
    curr = df.iloc[idx]

    y_gap = abs(
        curr["y_mean"] - prev["y_mean"]
    )

    overlap = max(
        0,
        min(prev["x_max"], curr["x_max"])
        - max(prev["x_min"], curr["x_min"])
    )

    min_width = min(
        prev["width"],
        curr["width"]
    )

    overlap_ratio = (
        overlap / min_width
        if min_width > 0
        else 0
    )

    should_merge = (
        y_gap <= Y_TOL
        and
        overlap_ratio <= MAX_HORIZONTAL_OVERLAP
    )

    if should_merge:

        current_group.append(
            curr.to_dict()
        )

    else:

        groups.append(
            current_group
        )

        current_group = [
            curr.to_dict()
        ]

groups.append(current_group)

print(
    "Linhas físicas detectadas:",
    len(groups)
)

print(
    "Segmentos por linha:",
    [len(g) for g in groups]
)

In [ ]:
# ============================================================
# AGRUPAMENTO CALIBRADO NA COLUNA III
# ============================================================

Y_TOL = 16
MAX_HORIZONTAL_OVERLAP = 0.15

groups = []
current_group = [df.iloc[0].to_dict()]

merged_pairs = []

for idx in range(1, len(df)):

    prev = df.iloc[idx - 1]
    curr = df.iloc[idx]

    y_gap = abs(
        curr["y_mean"] - prev["y_mean"]
    )

    overlap = max(
        0,
        min(prev["x_max"], curr["x_max"])
        - max(prev["x_min"], curr["x_min"])
    )

    min_width = min(
        prev["width"],
        curr["width"]
    )

    overlap_ratio = (
        overlap / min_width
        if min_width > 0
        else 0
    )

    should_merge = (
        y_gap <= Y_TOL
        and overlap_ratio <= MAX_HORIZONTAL_OVERLAP
    )

    if should_merge:

        merged_pairs.append({
            "kraken_A": int(prev["kraken_id"]),
            "kraken_B": int(curr["kraken_id"]),
            "y_gap": round(y_gap, 2),
            "overlap_ratio": round(overlap_ratio, 3)
        })

        current_group.append(
            curr.to_dict()
        )

    else:

        groups.append(current_group)
        current_group = [curr.to_dict()]

groups.append(current_group)

print("Baselines Kraken originais:", len(df))
print("Fusões realizadas:", len(merged_pairs))
print("Linhas físicas após fusão:", len(groups))

print("\nPares fundidos:")
display(pd.DataFrame(merged_pairs))

In [ ]:
physical_lines = []

for i, group in enumerate(groups, start=1):

    ys = [item["y_mean"] for item in group]
    xmins = [item["x_min"] for item in group]
    xmaxs = [item["x_max"] for item in group]

    physical_lines.append({
        "candidate_id": i,
        "y_center": float(np.median(ys)),
        "x_min": min(xmins),
        "x_max": max(xmaxs),
        "width": max(xmaxs) - min(xmins),
        "n_fragments": len(group),
        "kraken_ids": [
            int(item["kraken_id"])
            for item in group
        ]
    })

physical_df = (
    pd.DataFrame(physical_lines)
    .sort_values("y_center")
    .reset_index(drop=True)
)

physical_df["gap_to_next"] = (
    physical_df["y_center"].shift(-1)
    - physical_df["y_center"]
)

median_spacing = (
    physical_df["gap_to_next"]
    .dropna()
    .median()
)

print("Linhas físicas:", len(physical_df))
print("Espaçamento mediano:", median_spacing)

display(
    physical_df[
        [
            "candidate_id",
            "y_center",
            "gap_to_next",
            "width",
            "n_fragments",
            "kraken_ids"
        ]
    ]
)

In [ ]:
display(
    physical_df[
        [
            "candidate_id",
            "y_center",
            "gap_to_next"
        ]
    ]
    .sort_values(
        "gap_to_next",
        ascending=False
    )
    .head(6)
)

In [ ]:
from PIL import Image
from pathlib import Path

source = Path(
    "/content/Qumran_Analytics/1QS/"
    "columns/1QS_col_III.png"
)

img = Image.open(source).convert("RGB")

# Mantemos o corpo vertical,
# mas recuperamos a região direita.
X1 = 70
X2 = 1260

Y1 = 90
Y2 = 1790

roi_wide = img.crop(
    (X1, Y1, X2, Y2)
)

roi_wide_path = Path(
    "/content/Qumran_Analytics/1QS/"
    "automatic_segmentation/"
    "1QS_III_text_body_WIDE.png"
)

roi_wide.save(roi_wide_path)

print("ROI ampla:", roi_wide.size)
print(roi_wide_path)

In [ ]:
wide_seg_path = Path(
    "/content/Qumran_Analytics/1QS/"
    "automatic_segmentation/"
    "1QS_III_kraken_ROI_WIDE.json"
)

!kraken \
    -i "$roi_wide_path" "$wide_seg_path" \
    segment -bl -d horizontal-rl

In [ ]:
wide_rows = []

for i, line in enumerate(
    wide_lines,
    start=1
):

    baseline = line.get(
        "baseline",
        []
    )

    if not baseline:
        continue

    xs = [
        p[0]
        for p in baseline
    ]

    ys = [
        p[1]
        for p in baseline
    ]

    wide_rows.append({
        "kraken_id": i,
        "y_mean": sum(ys) / len(ys),
        "x_min": min(xs),
        "x_max": max(xs),
        "width": max(xs) - min(xs)
    })

wide_df = (
    pd.DataFrame(wide_rows)
    .sort_values("y_mean")
    .reset_index(drop=True)
)

# Região do grande buraco observado
display(
    wide_df[
        wide_df["y_mean"].between(
            790,
            900
        )
    ]
)

In [ ]:
import json
import pandas as pd
from pathlib import Path

wide_seg_path = Path(
    "/content/Qumran_Analytics/1QS/"
    "automatic_segmentation/"
    "1QS_III_kraken_ROI_WIDE.json"
)

print("JSON existe:", wide_seg_path.exists())

if not wide_seg_path.exists():
    raise FileNotFoundError(
        f"Não encontrei: {wide_seg_path}"
    )

with open(
    wide_seg_path,
    "r",
    encoding="utf-8"
) as f:
    wide_seg = json.load(f)

wide_lines = wide_seg.get("lines", [])

print(
    "Baselines Kraken na ROI ampla:",
    len(wide_lines)
)

wide_rows = []

for i, line in enumerate(
    wide_lines,
    start=1
):

    baseline = line.get(
        "baseline",
        []
    )

    if not baseline:
        continue

    xs = [p[0] for p in baseline]
    ys = [p[1] for p in baseline]

    wide_rows.append({
        "kraken_id": i,
        "y_mean": sum(ys) / len(ys),
        "x_min": min(xs),
        "x_max": max(xs),
        "width": max(xs) - min(xs)
    })

wide_df = (
    pd.DataFrame(wide_rows)
    .sort_values("y_mean")
    .reset_index(drop=True)
)

print("\nDetecções entre y=790 e y=900:")

display(
    wide_df[
        wide_df["y_mean"].between(
            790,
            900
        )
    ]
)

In [ ]:
from pathlib import Path

columns_dir = Path(
    "/content/Qumran_Analytics/1QS/columns"
)

files = sorted(columns_dir.glob("*"))

for f in files:
    print(f.name)

In [ ]:
from pathlib import Path
from PIL import Image
import subprocess
import json
import pandas as pd
import shutil

# ============================================================
# CONFIGURAÇÃO
# ============================================================

base = Path("/content/Qumran_Analytics/1QS")

columns_dir = base / "columns"
roi_dir = base / "automatic_segmentation" / "roi"
json_dir = base / "automatic_segmentation" / "json"

roi_dir.mkdir(parents=True, exist_ok=True)
json_dir.mkdir(parents=True, exist_ok=True)

kraken_exe = shutil.which("kraken")

if kraken_exe is None:
    raise RuntimeError("Kraken não encontrado no ambiente.")

print("Kraken:", kraken_exe)

columns = [
    "I", "II", "III", "IV", "V", "VI",
    "VII", "VIII", "IX", "X", "XI"
]

# ROI proporcional baseada no piloto da coluna III
LEFT_RATIO   = 0.055
RIGHT_RATIO  = 0.77
TOP_RATIO    = 0.05
BOTTOM_RATIO = 0.995

summary = []

# ============================================================
# PROCESSAMENTO EM LOTE
# ============================================================

for col in columns:

    image_path = columns_dir / f"1QS_col_{col}.png"

    if not image_path.exists():
        print(f"[{col}] arquivo não encontrado.")
        continue

    img = Image.open(image_path).convert("RGB")

    w, h = img.size

    x1 = int(w * LEFT_RATIO)
    x2 = int(w * RIGHT_RATIO)
    y1 = int(h * TOP_RATIO)
    y2 = int(h * BOTTOM_RATIO)

    roi = img.crop((x1, y1, x2, y2))

    roi_path = (
        roi_dir /
        f"1QS_col_{col}_ROI.png"
    )

    roi.save(roi_path)

    json_path = (
        json_dir /
        f"1QS_col_{col}_kraken.json"
    )

    cmd = [
        kraken_exe,
        "-i",
        str(roi_path),
        str(json_path),
        "segment",
        "-bl",
        "-d",
        "horizontal-rl"
    ]

    print(f"\n[{col}] segmentando...")

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:

        print(f"[{col}] ERRO:")
        print(result.stderr[-1500:])

        summary.append({
            "column": col,
            "image_width": w,
            "image_height": h,
            "roi_width": roi.width,
            "roi_height": roi.height,
            "raw_baselines": None,
            "status": "ERROR"
        })

        continue

    # --------------------------------------------------------
    # LÊ RESULTADO
    # --------------------------------------------------------

    with open(
        json_path,
        "r",
        encoding="utf-8"
    ) as f:
        seg = json.load(f)

    lines = seg.get("lines", [])

    count = len(lines)

    print(
        f"[{col}] baselines detectadas: {count}"
    )

    summary.append({
        "column": col,
        "image_width": w,
        "image_height": h,
        "roi_width": roi.width,
        "roi_height": roi.height,
        "raw_baselines": count,
        "status": "OK"
    })

# ============================================================
# RESUMO
# ============================================================

summary_df = pd.DataFrame(summary)

summary_path = (
    base /
    "automatic_segmentation/"
    "1QS_KRAKEN_ALL_COLUMNS_SUMMARY.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n================================")
print("SEGMENTAÇÃO DO 1QS CONCLUÍDA")
print("================================")

display(summary_df)

print("\nResumo salvo em:")
print(summary_path)

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

base = Path("/content/Qumran_Analytics/1QS")

json_dir = base / "automatic_segmentation" / "json"
roi_dir  = base / "automatic_segmentation" / "roi"

columns = [
    "I", "II", "III", "IV", "V", "VI",
    "VII", "VIII", "IX", "X", "XI"
]

records = []

for col in columns:

    json_path = json_dir / f"1QS_col_{col}_kraken.json"
    roi_path  = roi_dir  / f"1QS_col_{col}_ROI.png"

    if not json_path.exists():
        continue

    from PIL import Image

    roi_img = Image.open(roi_path)
    roi_w, roi_h = roi_img.size

    with open(json_path, "r", encoding="utf-8") as f:
        seg = json.load(f)

    lines = seg.get("lines", [])

    temp = []

    for i, line in enumerate(lines, start=1):

        baseline = line.get("baseline", [])

        if not baseline:
            continue

        xs = np.array([p[0] for p in baseline])
        ys = np.array([p[1] for p in baseline])

        temp.append({
            "column": col,
            "kraken_id": i,
            "y_mean": float(np.mean(ys)),
            "x_min": int(xs.min()),
            "x_max": int(xs.max()),
            "width": int(xs.max() - xs.min()),
            "roi_width": roi_w,
            "roi_height": roi_h
        })

    cdf = (
        pd.DataFrame(temp)
        .sort_values("y_mean")
        .reset_index(drop=True)
    )

    if len(cdf) == 0:
        continue

    cdf["gap_prev"] = cdf["y_mean"].diff()
    cdf["gap_next"] = cdf["y_mean"].shift(-1) - cdf["y_mean"]

    median_width = cdf["width"].median()
    median_gap = cdf["gap_next"].dropna().median()

    cdf["width_ratio"] = (
        cdf["width"] / median_width
    )

    cdf["center_x"] = (
        (cdf["x_min"] + cdf["x_max"]) / 2
    )

    # --------------------------------------------------------
    # FLAGS — NÃO EXCLUEM NADA
    # --------------------------------------------------------

    cdf["flag_short"] = (
        cdf["width_ratio"] < 0.30
    )

    cdf["flag_edge_right"] = (
        cdf["x_min"] > 0.78 * roi_w
    )

    cdf["flag_edge_left"] = (
        cdf["x_max"] < 0.22 * roi_w
    )

    cdf["flag_close_y"] = (
        cdf["gap_prev"] < 0.30 * median_gap
    ) | (
        cdf["gap_next"] < 0.30 * median_gap
    )

    cdf["flag_large_gap_after"] = (
        cdf["gap_next"] > 1.60 * median_gap
    )

    cdf["median_width_column"] = median_width
    cdf["median_gap_column"] = median_gap

    flags = []

    for _, row in cdf.iterrows():

        f = []

        if row["flag_short"]:
            f.append("SHORT")

        if row["flag_edge_right"]:
            f.append("RIGHT_EDGE")

        if row["flag_edge_left"]:
            f.append("LEFT_EDGE")

        if row["flag_close_y"]:
            f.append("CLOSE_Y")

        if row["flag_large_gap_after"]:
            f.append("LARGE_GAP_AFTER")

        flags.append(
            "|".join(f) if f else "ACCEPT_CANDIDATE"
        )

    cdf["review_flag"] = flags

    records.append(cdf)

all_lines = pd.concat(
    records,
    ignore_index=True
)

output = (
    base /
    "automatic_segmentation/"
    "1QS_ALL_BASELINES_DIAGNOSTIC.csv"
)

all_lines.to_csv(
    output,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo salvo:")
print(output)

print("\nResumo por coluna:")

summary = (
    all_lines
    .groupby("column")
    .agg(
        raw_baselines=("kraken_id", "count"),
        median_width=("width", "median"),
        median_gap=("median_gap_column", "first"),
        short=("flag_short", "sum"),
        edge_right=("flag_edge_right", "sum"),
        edge_left=("flag_edge_left", "sum"),
        close_y=("flag_close_y", "sum"),
        large_gaps=("flag_large_gap_after", "sum")
    )
)

display(summary)

In [ ]:
print("CASOS QUE PRECISAM DE REVISÃO:")

review = all_lines[
    all_lines["review_flag"]
    != "ACCEPT_CANDIDATE"
].copy()

display(
    review[
        [
            "column",
            "kraken_id",
            "y_mean",
            "x_min",
            "x_max",
            "width",
            "gap_prev",
            "gap_next",
            "review_flag"
        ]
    ]
)

print("\nTotal de baselines:", len(all_lines))
print("Aceitas sem alerta:",
      (all_lines["review_flag"] == "ACCEPT_CANDIDATE").sum())
print("Casos sinalizados:", len(review))

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

base = Path("/content/Qumran_Analytics/1QS")

input_path = (
    base /
    "automatic_segmentation/"
    "1QS_ALL_BASELINES_DIAGNOSTIC.csv"
)

df = pd.read_csv(input_path)

# ============================================================
# CLASSIFICAÇÃO AUTOMÁTICA DOS ALERTAS
# ============================================================

def classify(row):

    flags = str(row["review_flag"]).split("|")

    # --------------------------------------------------------
    # 1. RUÍDO MUITO PROVÁVEL
    # pequeno + margem extrema
    # --------------------------------------------------------

    if (
        row["flag_short"]
        and
        (
            row["flag_edge_right"]
            or row["flag_edge_left"]
        )
        and
        row["width_ratio"] < 0.20
    ):
        return "LIKELY_NOISE"

    # --------------------------------------------------------
    # 2. POSSÍVEL FRAGMENTO DA MESMA LINHA
    # baseline muito próxima verticalmente
    # --------------------------------------------------------

    if row["flag_close_y"]:
        return "MERGE_REVIEW"

    # --------------------------------------------------------
    # 3. POSSÍVEL LINHA AUSENTE
    # grande espaço depois
    # --------------------------------------------------------

    if row["flag_large_gap_after"]:
        return "GAP_REVIEW"

    # --------------------------------------------------------
    # 4. Linha curta legítima possível
    # --------------------------------------------------------

    if row["flag_short"]:
        return "SHORT_LINE_REVIEW"

    # --------------------------------------------------------
    # 5. Marginalia possível
    # --------------------------------------------------------

    if (
        row["flag_edge_right"]
        or row["flag_edge_left"]
    ):
        return "MARGIN_REVIEW"

    return "ACCEPT"


df["review_class"] = df.apply(
    classify,
    axis=1
)

print("Classificação geral:")
print(
    df["review_class"]
    .value_counts()
)

print("\nPor coluna:")
display(
    pd.crosstab(
        df["column"],
        df["review_class"]
    )
)

In [ ]:
# ============================================================
# PRIORIZAÇÃO
# ============================================================

priority_map = {
    "GAP_REVIEW": 1,
    "MERGE_REVIEW": 2,
    "SHORT_LINE_REVIEW": 3,
    "MARGIN_REVIEW": 4,
    "LIKELY_NOISE": 5,
    "ACCEPT": 99
}

df["priority"] = (
    df["review_class"]
    .map(priority_map)
)

review_queue = (
    df[
        df["review_class"]
        != "ACCEPT"
    ]
    .sort_values(
        ["priority", "column", "y_mean"]
    )
)

print(
    "Casos totais para revisão:",
    len(review_queue)
)

print("\nFila prioritária:")
display(
    review_queue[
        [
            "column",
            "kraken_id",
            "y_mean",
            "width",
            "width_ratio",
            "gap_prev",
            "gap_next",
            "review_flag",
            "review_class"
        ]
    ]
)

output = (
    base /
    "automatic_segmentation/"
    "1QS_REVIEW_QUEUE.csv"
)

review_queue.to_csv(
    output,
    index=False,
    encoding="utf-8-sig"
)

print("\nSalvo em:")
print(output)

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import pandas as pd
import json
import math

base = Path("/content/Qumran_Analytics/1QS")

queue_path = (
    base /
    "automatic_segmentation/"
    "1QS_REVIEW_QUEUE.csv"
)

roi_dir = (
    base /
    "automatic_segmentation/"
    "roi"
)

json_dir = (
    base /
    "automatic_segmentation/"
    "json"
)

out_dir = (
    base /
    "automatic_segmentation/"
    "review_sheets"
)

out_dir.mkdir(
    parents=True,
    exist_ok=True
)

queue = pd.read_csv(queue_path)

print("Distribuição das classes:")
print(queue["review_class"].value_counts())

# ============================================================
# PRIMEIRO: APENAS CASOS DE PRIORIDADE ALTA
# ============================================================

priority_classes = [
    "GAP_REVIEW",
    "MERGE_REVIEW"
]

priority = queue[
    queue["review_class"].isin(priority_classes)
].copy()

print("\nCasos prioritários:", len(priority))

# ============================================================
# GERA PRANCHA POR COLUNA
# ============================================================

for col in priority["column"].unique():

    col_df = (
        priority[
            priority["column"] == col
        ]
        .sort_values("y_mean")
    )

    roi_path = (
        roi_dir /
        f"1QS_col_{col}_ROI.png"
    )

    json_path = (
        json_dir /
        f"1QS_col_{col}_kraken.json"
    )

    if not roi_path.exists():
        continue

    img = Image.open(
        roi_path
    ).convert("RGB")

    with open(
        json_path,
        "r",
        encoding="utf-8"
    ) as f:
        seg = json.load(f)

    kraken_lines = seg.get("lines", [])

    panels = []

    for _, row in col_df.iterrows():

        kid = int(row["kraken_id"])

        # contexto vertical
        y = int(row["y_mean"])

        top = max(
            0,
            y - 90
        )

        bottom = min(
            img.height,
            y + 90
        )

        crop = img.crop(
            (
                0,
                top,
                img.width,
                bottom
            )
        )

        # amplia para inspeção
        scale = 2

        crop = crop.resize(
            (
                crop.width * scale,
                crop.height * scale
            ),
            Image.Resampling.LANCZOS
        )

        header = 70

        panel = Image.new(
            "RGB",
            (
                crop.width,
                crop.height + header
            ),
            "white"
        )

        panel.paste(
            crop,
            (0, header)
        )

        draw = ImageDraw.Draw(panel)

        label = (
            f"Col. {col} | K{kid:02d} | "
            f"{row['review_class']} | "
            f"y={row['y_mean']:.1f} | "
            f"gap_prev={row['gap_prev']:.1f} | "
            f"gap_next={row['gap_next']:.1f}"
        )

        draw.text(
            (15, 15),
            label,
            fill="black"
        )

        # marca centro da baseline no recorte
        relative_y = (
            (y - top) * scale
            + header
        )

        draw.line(
            [
                (0, relative_y),
                (panel.width, relative_y)
            ],
            fill="red",
            width=3
        )

        panels.append(panel)

    if not panels:
        continue

    width = max(
        p.width
        for p in panels
    )

    height = sum(
        p.height
        for p in panels
    )

    sheet = Image.new(
        "RGB",
        (width, height),
        "white"
    )

    y_offset = 0

    for panel in panels:

        sheet.paste(
            panel,
            (0, y_offset)
        )

        y_offset += panel.height

    output = (
        out_dir /
        f"1QS_col_{col}_PRIORITY_REVIEW.png"
    )

    sheet.save(output)

    print(
        f"Coluna {col}: "
        f"{len(panels)} casos -> {output}"
    )

print("\nPranchas prioritárias concluídas.")

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw
import pandas as pd

base = Path("/content/Qumran_Analytics/1QS")

queue_path = (
    base /
    "automatic_segmentation/"
    "1QS_REVIEW_QUEUE.csv"
)

roi_dir = (
    base /
    "automatic_segmentation/"
    "roi"
)

out_dir = (
    base /
    "automatic_segmentation/"
    "review_sheets"
)

out_dir.mkdir(
    parents=True,
    exist_ok=True
)

queue = pd.read_csv(queue_path)

gaps = (
    queue[
        queue["review_class"]
        == "GAP_REVIEW"
    ]
    .sort_values(
        ["column", "y_mean"]
    )
)

print("GAP_REVIEW encontrados:", len(gaps))

panels = []

for _, row in gaps.iterrows():

    col = str(row["column"])
    kid = int(row["kraken_id"])

    roi_path = (
        roi_dir /
        f"1QS_col_{col}_ROI.png"
    )

    img = Image.open(
        roi_path
    ).convert("RGB")

    y = int(row["y_mean"])

    # contexto maior porque queremos enxergar
    # se existe uma linha não detectada depois
    top = max(
        0,
        y - 100
    )

    bottom = min(
        img.height,
        y + int(row["gap_next"]) + 100
    )

    crop = img.crop(
        (
            0,
            top,
            img.width,
            bottom
        )
    )

    scale = 2

    crop = crop.resize(
        (
            crop.width * scale,
            crop.height * scale
        ),
        Image.Resampling.LANCZOS
    )

    header = 90

    panel = Image.new(
        "RGB",
        (
            crop.width,
            crop.height + header
        ),
        "white"
    )

    panel.paste(
        crop,
        (0, header)
    )

    draw = ImageDraw.Draw(panel)

    label = (
        f"Coluna {col} | K{kid:02d} | GAP_REVIEW | "
        f"y={row['y_mean']:.1f} | "
        f"gap_next={row['gap_next']:.1f}"
    )

    draw.text(
        (15, 15),
        label,
        fill="black"
    )

    baseline_y = (
        (y - top) * scale
        + header
    )

    draw.line(
        [
            (0, baseline_y),
            (panel.width, baseline_y)
        ],
        fill="red",
        width=3
    )

    # marca aproximadamente onde o gap termina
    next_y = (
        (y + row["gap_next"] - top)
        * scale
        + header
    )

    draw.line(
        [
            (0, int(next_y)),
            (panel.width, int(next_y))
        ],
        fill="blue",
        width=3
    )

    panels.append(panel)

width = max(
    p.width
    for p in panels
)

height = sum(
    p.height
    for p in panels
)

sheet = Image.new(
    "RGB",
    (width, height),
    "white"
)

offset = 0

for panel in panels:

    sheet.paste(
        panel,
        (0, offset)
    )

    offset += panel.height

output = (
    out_dir /
    "1QS_ALL_GAP_REVIEW.png"
)

sheet.save(output)

print("\nPrancha criada:")
print(output)

In [ ]:
from google.colab import files

files.download(
    "/content/Qumran_Analytics/1QS/"
    "automatic_segmentation/"
    "review_sheets/"
    "1QS_ALL_GAP_REVIEW.png"
)

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

base = Path("/content/Qumran_Analytics/1QS")

queue_path = (
    base /
    "automatic_segmentation/"
    "1QS_REVIEW_QUEUE.csv"
)

queue = pd.read_csv(queue_path)

gap_cases = (
    queue[
        queue["review_class"] == "GAP_REVIEW"
    ]
    .sort_values(
        ["column", "y_mean"]
    )
    .reset_index(drop=True)
)

print("GAP_REVIEW encontrados:", len(gap_cases))

# Decisões visuais do lote revisado
decisions = [
    "NORMAL_LARGE_GAP",
    "DAMAGED_REGION",
    "NORMAL_LARGE_GAP",
    "NORMAL_LARGE_GAP",
    "DAMAGED_REGION"
]

if len(gap_cases) != len(decisions):
    raise ValueError(
        f"Esperávamos {len(decisions)} GAP_REVIEW, "
        f"mas foram encontrados {len(gap_cases)}."
    )

gap_cases["adjudication"] = decisions
gap_cases["adjudication_source"] = (
    "AI_assisted_visual_review"
)
gap_cases["researcher_confirmed"] = False
gap_cases["expert_verified"] = False
gap_cases["resolved"] = True
gap_cases["review_timestamp_utc"] = (
    datetime.now(timezone.utc).isoformat()
)

gap_output = (
    base /
    "automatic_segmentation/"
    "1QS_GAP_ADJUDICATION.csv"
)

gap_cases.to_csv(
    gap_output,
    index=False,
    encoding="utf-8-sig"
)

print("\nResultado:")
display(
    gap_cases[
        [
            "column",
            "kraken_id",
            "y_mean",
            "gap_next",
            "adjudication",
            "resolved"
        ]
    ]
)

print("\nSalvo em:")
print(gap_output)

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw
import pandas as pd
import json

base = Path("/content/Qumran_Analytics/1QS")

queue_path = (
    base /
    "automatic_segmentation/"
    "1QS_REVIEW_QUEUE.csv"
)

roi_dir = (
    base /
    "automatic_segmentation/"
    "roi"
)

json_dir = (
    base /
    "automatic_segmentation/"
    "json"
)

out_dir = (
    base /
    "automatic_segmentation/"
    "review_sheets"
)

out_dir.mkdir(
    parents=True,
    exist_ok=True
)

queue = pd.read_csv(queue_path)

merges = (
    queue[
        queue["review_class"] == "MERGE_REVIEW"
    ]
    .sort_values(
        ["column", "y_mean"]
    )
)

print("MERGE_REVIEW encontrados:", len(merges))

# ============================================================
# UMA PRANCHA POR COLUNA
# ============================================================

for col in merges["column"].unique():

    col_cases = (
        merges[
            merges["column"] == col
        ]
        .sort_values("y_mean")
    )

    roi_path = (
        roi_dir /
        f"1QS_col_{col}_ROI.png"
    )

    json_path = (
        json_dir /
        f"1QS_col_{col}_kraken.json"
    )

    img = Image.open(
        roi_path
    ).convert("RGB")

    with open(
        json_path,
        "r",
        encoding="utf-8"
    ) as f:
        seg = json.load(f)

    all_kraken = seg.get("lines", [])

    panels = []

    for _, row in col_cases.iterrows():

        target_id = int(row["kraken_id"])
        center_y = int(row["y_mean"])

        # bastante contexto acima e abaixo
        top = max(
            0,
            center_y - 100
        )

        bottom = min(
            img.height,
            center_y + 100
        )

        crop = img.crop(
            (
                0,
                top,
                img.width,
                bottom
            )
        )

        scale = 2

        crop = crop.resize(
            (
                crop.width * scale,
                crop.height * scale
            ),
            Image.Resampling.LANCZOS
        )

        header = 85

        panel = Image.new(
            "RGB",
            (
                crop.width,
                crop.height + header
            ),
            "white"
        )

        panel.paste(
            crop,
            (0, header)
        )

        draw = ImageDraw.Draw(panel)

        label = (
            f"Col. {col} | alvo K{target_id:02d} | "
            f"MERGE_REVIEW | "
            f"gap_prev={row['gap_prev']:.1f} | "
            f"gap_next={row['gap_next']:.1f}"
        )

        draw.text(
            (15, 15),
            label,
            fill="black"
        )

        # Desenha todas as baselines dentro do contexto
        for kid, line in enumerate(
            all_kraken,
            start=1
        ):

            baseline = line.get(
                "baseline",
                []
            )

            if not baseline:
                continue

            ys = [
                p[1]
                for p in baseline
            ]

            line_y = sum(ys) / len(ys)

            if not (
                top <= line_y <= bottom
            ):
                continue

            pts = []

            for x, y in baseline:

                pts.append(
                    (
                        int(x * scale),
                        int(
                            (y - top)
                            * scale
                            + header
                        )
                    )
                )

            if len(pts) >= 2:

                # alvo = mais espesso
                width = (
                    5
                    if kid == target_id
                    else 2
                )

                draw.line(
                    pts,
                    fill="red",
                    width=width
                )

                draw.text(
                    (
                        pts[0][0],
                        pts[0][1] - 20
                    ),
                    f"K{kid:02d}",
                    fill="blue"
                )

        panels.append(panel)

    if not panels:
        continue

    width = max(
        p.width
        for p in panels
    )

    height = sum(
        p.height
        for p in panels
    )

    sheet = Image.new(
        "RGB",
        (width, height),
        "white"
    )

    y_offset = 0

    for panel in panels:

        sheet.paste(
            panel,
            (0, y_offset)
        )

        y_offset += panel.height

    output = (
        out_dir /
        f"1QS_col_{col}_MERGE_REVIEW.png"
    )

    sheet.save(output)

    print(
        f"Coluna {col}: "
        f"{len(panels)} casos -> "
        f"{output}"
    )

print("\nPranchas MERGE_REVIEW concluídas.")

In [ ]:
merge_sheets = sorted(
    out_dir.glob(
        "*_MERGE_REVIEW.png"
    )
)

print(
    "Pranchas criadas:",
    len(merge_sheets)
)

for f in merge_sheets:
    print(f.name)

In [ ]:
from IPython.display import display
from PIL import Image

img = Image.open("/content/Qumran_Analytics/1QS/automatic_segmentation/review_sheets/1QS_col_VII_MERGE_REVIEW.png")
display(img)

In [ ]:
from IPython.display import display
from PIL import Image

img = Image.open("/content/Qumran_Analytics/1QS/automatic_segmentation/review_sheets/1QS_col_VII_MERGE_REVIEW.png")
img.thumbnail((1600, 1600))
display(img)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

base = Path("/content/Qumran_Analytics/1QS")

diag_path = (
    base /
    "automatic_segmentation/"
    "1QS_ALL_BASELINES_DIAGNOSTIC.csv"
)

df = pd.read_csv(diag_path)

events = []

for col, cdf in df.groupby("column"):

    cdf = (
        cdf
        .sort_values("y_mean")
        .reset_index(drop=True)
    )

    median_gap = cdf["median_gap_column"].iloc[0]

    for i in range(len(cdf) - 1):

        a = cdf.iloc[i]
        b = cdf.iloc[i + 1]

        y_gap = abs(
            b["y_mean"] - a["y_mean"]
        )

        # Só considera pares realmente próximos
        if y_gap >= 0.30 * median_gap:
            continue

        overlap = max(
            0,
            min(a["x_max"], b["x_max"])
            -
            max(a["x_min"], b["x_min"])
        )

        min_width = min(
            a["width"],
            b["width"]
        )

        overlap_ratio = (
            overlap / min_width
            if min_width > 0
            else 0
        )

        horizontal_gap = max(
            0,
            max(a["x_min"], b["x_min"])
            -
            min(a["x_max"], b["x_max"])
        )

        events.append({
            "column": col,

            "kraken_A":
                int(a["kraken_id"]),

            "kraken_B":
                int(b["kraken_id"]),

            "y_A":
                a["y_mean"],

            "y_B":
                b["y_mean"],

            "y_gap":
                y_gap,

            "xA_min":
                int(a["x_min"]),

            "xA_max":
                int(a["x_max"]),

            "xB_min":
                int(b["x_min"]),

            "xB_max":
                int(b["x_max"]),

            "width_A":
                int(a["width"]),

            "width_B":
                int(b["width"]),

            "overlap_ratio":
                overlap_ratio,

            "horizontal_gap":
                horizontal_gap
        })

events_df = pd.DataFrame(events)

events_df["event_id"] = [
    f"M{i:03d}"
    for i in range(1, len(events_df) + 1)
]

print(
    "Baselines anteriormente marcadas MERGE_REVIEW:",
    (df["flag_close_y"] == True).sum()
)

print(
    "Eventos de pares reais para revisar:",
    len(events_df)
)

print("\nColuna VII:")
display(
    events_df[
        events_df["column"].astype(str) == "VII"
    ][
        [
            "event_id",
            "kraken_A",
            "kraken_B",
            "y_gap",
            "width_A",
            "width_B",
            "overlap_ratio",
            "horizontal_gap"
        ]
    ]
)

output = (
    base /
    "automatic_segmentation/"
    "1QS_MERGE_EVENTS.csv"
)

events_df.to_csv(
    output,
    index=False,
    encoding="utf-8-sig"
)

print("\nSalvo em:")
print(output)

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

base = Path("/content/Qumran_Analytics/1QS")

events_path = (
    base /
    "automatic_segmentation/"
    "1QS_MERGE_EVENTS.csv"
)

events = pd.read_csv(events_path)

decisions = {
    "M017": ("TRUE_MERGE", "HIGH"),
    "M018": ("KEEP_SEPARATE_SUPRALINEAR", "HIGH"),
    "M019": ("NOISE_FRAGMENT", "HIGH"),
    "M020": ("TRUE_MERGE", "HIGH"),
    "M021": ("TRUE_MERGE", "HIGH"),
    "M022": ("TRUE_MERGE", "HIGH"),
    "M023": ("TRUE_MERGE", "MEDIUM")
}

review = events[
    events["event_id"].isin(decisions)
].copy()

review["adjudication"] = review["event_id"].map(
    lambda x: decisions[x][0]
)

review["confidence"] = review["event_id"].map(
    lambda x: decisions[x][1]
)

review["adjudication_source"] = (
    "AI_assisted_visual_and_geometric_review"
)

review["researcher_confirmed"] = False
review["expert_verified"] = False

review["timestamp_utc"] = (
    datetime.now(timezone.utc).isoformat()
)

output = (
    base /
    "automatic_segmentation/"
    "1QS_COL_VII_MERGE_ADJUDICATION.csv"
)

review.to_csv(
    output,
    index=False,
    encoding="utf-8-sig"
)

display(
    review[
        [
            "event_id",
            "kraken_A",
            "kraken_B",
            "y_gap",
            "horizontal_gap",
            "adjudication",
            "confidence"
        ]
    ]
)

print(output)

In [ ]:
import pandas as pd
from pathlib import Path

base = Path("/content/Qumran_Analytics/1QS")

events = pd.read_csv(
    base /
    "automatic_segmentation/"
    "1QS_MERGE_EVENTS.csv"
)

diag = pd.read_csv(
    base /
    "automatic_segmentation/"
    "1QS_ALL_BASELINES_DIAGNOSTIC.csv"
)

# Medianas por coluna
stats = (
    diag
    .groupby("column")
    .agg(
        median_width=("width", "median"),
        median_gap=("median_gap_column", "first")
    )
    .reset_index()
)

events = events.merge(
    stats,
    on="column",
    how="left"
)

# Razões normalizadas
events["min_width_ratio"] = (
    events[["width_A", "width_B"]].min(axis=1)
    / events["median_width"]
)

events["max_width_ratio"] = (
    events[["width_A", "width_B"]].max(axis=1)
    / events["median_width"]
)

events["horizontal_gap_ratio"] = (
    events["horizontal_gap"]
    / events["median_width"]
)

events["y_gap_ratio"] = (
    events["y_gap"]
    / events["median_gap"]
)

def classify_merge_event(row):

    # Fragmento minúsculo: provável falso positivo
    if row["min_width_ratio"] < 0.03:
        return "AUTO_NOISE_FRAGMENT"

    # Dois fragmentos curtos muito separados:
    # padrão observado no material supralinear da VII
    if (
        row["max_width_ratio"] < 0.30
        and row["horizontal_gap_ratio"] > 0.40
    ):
        return "AUTO_KEEP_SEPARATE"

    # Fragmentos próximos verticalmente e
    # horizontalmente complementares
    if (
        row["y_gap_ratio"] <= 0.25
        and row["overlap_ratio"] <= 0.10
        and row["horizontal_gap_ratio"] <= 0.25
    ):
        return "AUTO_TRUE_MERGE"

    return "MANUAL_REVIEW"

events["automatic_decision"] = events.apply(
    classify_merge_event,
    axis=1
)

print("CLASSIFICAÇÃO AUTOMÁTICA:")
print(
    events["automatic_decision"]
    .value_counts()
)

print("\nCASOS QUE AINDA EXIGEM INSPEÇÃO:")
manual = events[
    events["automatic_decision"]
    == "MANUAL_REVIEW"
]

display(
    manual[
        [
            "event_id",
            "column",
            "kraken_A",
            "kraken_B",
            "y_gap",
            "horizontal_gap",
            "min_width_ratio",
            "max_width_ratio",
            "automatic_decision"
        ]
    ]
)

output = (
    base /
    "automatic_segmentation/"
    "1QS_MERGE_EVENTS_CLASSIFIED.csv"
)

events.to_csv(
    output,
    index=False,
    encoding="utf-8-sig"
)

print("\nSalvo em:")
print(output)

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw
import pandas as pd
import json

base = Path("/content/Qumran_Analytics/1QS")

events_path = (
    base /
    "automatic_segmentation/"
    "1QS_MERGE_EVENTS_CLASSIFIED.csv"
)

roi_dir = (
    base /
    "automatic_segmentation/"
    "roi"
)

json_dir = (
    base /
    "automatic_segmentation/"
    "json"
)

out_dir = (
    base /
    "automatic_segmentation/"
    "review_sheets"
)

out_dir.mkdir(
    parents=True,
    exist_ok=True
)

events = pd.read_csv(events_path)

manual = (
    events[
        events["automatic_decision"]
        == "MANUAL_REVIEW"
    ]
    .sort_values(
        ["column", "y_A"]
    )
    .reset_index(drop=True)
)

print("Casos manuais:", len(manual))

panels = []

for _, row in manual.iterrows():

    col = str(row["column"])

    kid_a = int(row["kraken_A"])
    kid_b = int(row["kraken_B"])

    roi_path = (
        roi_dir /
        f"1QS_col_{col}_ROI.png"
    )

    json_path = (
        json_dir /
        f"1QS_col_{col}_kraken.json"
    )

    img = Image.open(
        roi_path
    ).convert("RGB")

    with open(
        json_path,
        "r",
        encoding="utf-8"
    ) as f:
        seg = json.load(f)

    lines = seg.get("lines", [])

    line_a = lines[kid_a - 1]
    line_b = lines[kid_b - 1]

    baseline_a = line_a.get(
        "baseline", []
    )

    baseline_b = line_b.get(
        "baseline", []
    )

    ys_a = [
        p[1]
        for p in baseline_a
    ]

    ys_b = [
        p[1]
        for p in baseline_b
    ]

    center_a = (
        sum(ys_a) / len(ys_a)
    )

    center_b = (
        sum(ys_b) / len(ys_b)
    )

    center = int(
        (center_a + center_b) / 2
    )

    # contexto vertical generoso
    top = max(
        0,
        center - 100
    )

    bottom = min(
        img.height,
        center + 100
    )

    crop = img.crop(
        (
            0,
            top,
            img.width,
            bottom
        )
    )

    scale = 2

    crop = crop.resize(
        (
            crop.width * scale,
            crop.height * scale
        ),
        Image.Resampling.LANCZOS
    )

    header = 110

    panel = Image.new(
        "RGB",
        (
            crop.width,
            crop.height + header
        ),
        "white"
    )

    panel.paste(
        crop,
        (0, header)
    )

    draw = ImageDraw.Draw(panel)

    label1 = (
        f"{row['event_id']} | Coluna {col} | "
        f"K{kid_a} + K{kid_b}"
    )

    label2 = (
        f"y_gap={row['y_gap']:.1f} | "
        f"horizontal_gap={row['horizontal_gap']:.0f} | "
        f"widths={row['width_A']:.0f}/{row['width_B']:.0f}"
    )

    draw.text(
        (15, 15),
        label1,
        fill="black"
    )

    draw.text(
        (15, 50),
        label2,
        fill="black"
    )

    # baseline A
    pts_a = [
        (
            int(x * scale),
            int(
                (y - top)
                * scale
                + header
            )
        )
        for x, y in baseline_a
    ]

    if len(pts_a) >= 2:

        draw.line(
            pts_a,
            fill="red",
            width=5
        )

        draw.text(
            (
                pts_a[0][0],
                pts_a[0][1] - 25
            ),
            f"K{kid_a}",
            fill="red"
        )

    # baseline B
    pts_b = [
        (
            int(x * scale),
            int(
                (y - top)
                * scale
                + header
            )
        )
        for x, y in baseline_b
    ]

    if len(pts_b) >= 2:

        draw.line(
            pts_b,
            fill="blue",
            width=5
        )

        draw.text(
            (
                pts_b[0][0],
                pts_b[0][1] - 25
            ),
            f"K{kid_b}",
            fill="blue"
        )

    panels.append(panel)

# monta uma única prancha
width = max(
    p.width
    for p in panels
)

height = sum(
    p.height
    for p in panels
)

sheet = Image.new(
    "RGB",
    (width, height),
    "white"
)

offset = 0

for panel in panels:

    sheet.paste(
        panel,
        (0, offset)
    )

    offset += panel.height

output = (
    out_dir /
    "1QS_MANUAL_MERGE_REVIEW.png"
)

sheet.save(output)

print("\nPrancha criada:")
print(output)

In [ ]:
from google.colab import files

files.download(
    "/content/Qumran_Analytics/1QS/"
    "automatic_segmentation/"
    "review_sheets/"
    "1QS_MANUAL_MERGE_REVIEW.png"
)

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

base = Path("/content/Qumran_Analytics/1QS")

events_path = (
    base /
    "automatic_segmentation/"
    "1QS_MERGE_EVENTS_CLASSIFIED.csv"
)

events = pd.read_csv(events_path)

decisions = {
    "M001": {
        "adjudication": "TRUE_MERGE",
        "confidence": "HIGH",
        "action": "MERGE_K1_K4",
        "note": "Same physical line separated by manuscript damage."
    },

    "M003": {
        "adjudication": "TRUE_MERGE",
        "confidence": "HIGH",
        "action": "MERGE_K30_K31",
        "note": "Same physical line continuing across a material lacuna."
    },

    "M004": {
        "adjudication": "DUPLICATE_BASELINE",
        "confidence": "HIGH",
        "action": "KEEP_K15_DROP_K14",
        "note": (
            "K14 is a partial redundant detection of the physical "
            "line already represented more completely by K15."
        )
    },

    "M025": {
        "adjudication": "TRUE_MERGE",
        "confidence": "MEDIUM_HIGH",
        "action": "MERGE_K9_K8",
        "note": (
            "Complementary segments at essentially the same physical "
            "line level."
        )
    },

    "M026": {
        "adjudication": "KEEP_SEPARATE",
        "confidence": "MEDIUM",
        "action": "KEEP_K10_K11_SEPARATE",
        "note": (
            "Segments are vertically close but separated by a large "
            "horizontal blank without clear material loss; conservative "
            "decision is to preserve them separately."
        )
    }
}

manual_ids = list(decisions.keys())

review = events[
    events["event_id"].isin(manual_ids)
].copy()

review["adjudication"] = review["event_id"].map(
    lambda x: decisions[x]["adjudication"]
)

review["confidence"] = review["event_id"].map(
    lambda x: decisions[x]["confidence"]
)

review["action"] = review["event_id"].map(
    lambda x: decisions[x]["action"]
)

review["review_note"] = review["event_id"].map(
    lambda x: decisions[x]["note"]
)

review["review_source"] = "AI_assisted_visual_adjudication"
review["researcher_confirmed"] = False
review["expert_verified"] = False
review["timestamp_utc"] = datetime.now(timezone.utc).isoformat()

output = (
    base /
    "automatic_segmentation/"
    "1QS_MANUAL_MERGE_ADJUDICATION.csv"
)

review.to_csv(
    output,
    index=False,
    encoding="utf-8-sig"
)

display(
    review[
        [
            "event_id",
            "column",
            "kraken_A",
            "kraken_B",
            "adjudication",
            "confidence",
            "action"
        ]
    ]
)

print(output)

In [ ]:
import pandas as pd
from pathlib import Path

base = Path("/content/Qumran_Analytics/1QS")

events_path = (
    base /
    "automatic_segmentation/"
    "1QS_MERGE_EVENTS_CLASSIFIED.csv"
)

events = pd.read_csv(events_path)

samples = []

sampling_plan = {
    "AUTO_TRUE_MERGE": 4,
    "AUTO_NOISE_FRAGMENT": 2,
    "AUTO_KEEP_SEPARATE": 1
}

for cls, n in sampling_plan.items():

    subset = events[
        events["automatic_decision"] == cls
    ].copy()

    n_take = min(n, len(subset))

    sample = subset.sample(
        n=n_take,
        random_state=42
    )

    samples.append(sample)

audit = (
    pd.concat(samples)
    .sort_values(
        ["automatic_decision", "column", "y_A"]
    )
    .reset_index(drop=True)
)

print("Amostra de auditoria:")
display(
    audit[
        [
            "event_id",
            "column",
            "kraken_A",
            "kraken_B",
            "y_gap",
            "horizontal_gap",
            "automatic_decision"
        ]
    ]
)

audit_path = (
    base /
    "automatic_segmentation/"
    "1QS_AUTOMATIC_CLASSIFICATION_AUDIT_SAMPLE.csv"
)

audit.to_csv(
    audit_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nSalvo em:")
print(audit_path)

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw
import json

roi_dir = (
    base /
    "automatic_segmentation/"
    "roi"
)

json_dir = (
    base /
    "automatic_segmentation/"
    "json"
)

review_dir = (
    base /
    "automatic_segmentation/"
    "review_sheets"
)

review_dir.mkdir(
    parents=True,
    exist_ok=True
)

panels = []

for _, row in audit.iterrows():

    col = str(row["column"])

    ka = int(row["kraken_A"])
    kb = int(row["kraken_B"])

    roi_path = (
        roi_dir /
        f"1QS_col_{col}_ROI.png"
    )

    json_path = (
        json_dir /
        f"1QS_col_{col}_kraken.json"
    )

    img = Image.open(
        roi_path
    ).convert("RGB")

    with open(
        json_path,
        "r",
        encoding="utf-8"
    ) as f:
        seg = json.load(f)

    lines = seg.get("lines", [])

    baseline_a = lines[ka - 1]["baseline"]
    baseline_b = lines[kb - 1]["baseline"]

    ya = sum(p[1] for p in baseline_a) / len(baseline_a)
    yb = sum(p[1] for p in baseline_b) / len(baseline_b)

    center = int((ya + yb) / 2)

    top = max(
        0,
        center - 100
    )

    bottom = min(
        img.height,
        center + 100
    )

    crop = img.crop(
        (
            0,
            top,
            img.width,
            bottom
        )
    )

    scale = 2

    crop = crop.resize(
        (
            crop.width * scale,
            crop.height * scale
        ),
        Image.Resampling.LANCZOS
    )

    header = 110

    panel = Image.new(
        "RGB",
        (
            crop.width,
            crop.height + header
        ),
        "white"
    )

    panel.paste(
        crop,
        (0, header)
    )

    draw = ImageDraw.Draw(panel)

    draw.text(
        (15, 15),
        (
            f"{row['event_id']} | Col. {col} | "
            f"{row['automatic_decision']}"
        ),
        fill="black"
    )

    draw.text(
        (15, 50),
        (
            f"K{ka} + K{kb} | "
            f"y_gap={row['y_gap']:.1f} | "
            f"h_gap={row['horizontal_gap']:.0f}"
        ),
        fill="black"
    )

    pts_a = [
        (
            int(x * scale),
            int((y - top) * scale + header)
        )
        for x, y in baseline_a
    ]

    pts_b = [
        (
            int(x * scale),
            int((y - top) * scale + header)
        )
        for x, y in baseline_b
    ]

    if len(pts_a) >= 2:
        draw.line(
            pts_a,
            fill="red",
            width=5
        )

    if len(pts_b) >= 2:
        draw.line(
            pts_b,
            fill="blue",
            width=5
        )

    panels.append(panel)

width = max(p.width for p in panels)
height = sum(p.height for p in panels)

sheet = Image.new(
    "RGB",
    (width, height),
    "white"
)

offset = 0

for panel in panels:
    sheet.paste(panel, (0, offset))
    offset += panel.height

output = (
    review_dir /
    "1QS_AUTOMATIC_CLASSIFICATION_AUDIT.png"
)

sheet.save(output)

print(output)

In [ ]:
from google.colab import files

files.download(
    "/content/Qumran_Analytics/1QS/"
    "automatic_segmentation/"
    "review_sheets/"
    "1QS_AUTOMATIC_CLASSIFICATION_AUDIT.png"
)

In [ ]:
# ============================================================
# CLASSIFICADOR V2 — mais conservador
# ============================================================

def classify_merge_event_v2(row):

    # 1. Fragmento minúsculo: ruído praticamente certo
    if row["min_width_ratio"] < 0.03:
        return "AUTO_NOISE_FRAGMENT"

    # 2. Fragmento muito pequeno:
    # não permitir merge automático
    if row["min_width_ratio"] < 0.10:
        return "SMALL_FRAGMENT_REVIEW"

    # 3. Dois elementos pequenos muito distantes
    if (
        row["max_width_ratio"] < 0.30
        and
        row["horizontal_gap_ratio"] > 0.40
    ):
        return "AUTO_KEEP_SEPARATE"

    # 4. Fragmentos complementares da mesma linha
    if (
        row["y_gap_ratio"] <= 0.25
        and
        row["overlap_ratio"] <= 0.10
        and
        row["horizontal_gap_ratio"] <= 0.25
    ):
        return "AUTO_TRUE_MERGE"

    return "MANUAL_REVIEW"


events["automatic_decision_v2"] = events.apply(
    classify_merge_event_v2,
    axis=1
)

print("CLASSIFICAÇÃO V2:")
print(
    events["automatic_decision_v2"]
    .value_counts()
)

print("\nCASOS QUE NÃO SERÃO DECIDIDOS AUTOMATICAMENTE:")

review_v2 = events[
    events["automatic_decision_v2"].isin(
        [
            "MANUAL_REVIEW",
            "SMALL_FRAGMENT_REVIEW"
        ]
    )
]

display(
    review_v2[
        [
            "event_id",
            "column",
            "kraken_A",
            "kraken_B",
            "min_width_ratio",
            "y_gap",
            "horizontal_gap",
            "automatic_decision_v2"
        ]
    ]
)

output = (
    base /
    "automatic_segmentation/"
    "1QS_MERGE_EVENTS_CLASSIFIED_V2.csv"
)

events.to_csv(
    output,
    index=False,
    encoding="utf-8-sig"
)

print("\nSalvo em:")
print(output)

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw
import pandas as pd
import json

base = Path("/content/Qumran_Analytics/1QS")

events = pd.read_csv(
    base /
    "automatic_segmentation/"
    "1QS_MERGE_EVENTS_CLASSIFIED_V2.csv"
)

pending = (
    events[
        events["automatic_decision_v2"]
        == "SMALL_FRAGMENT_REVIEW"
    ]
    .sort_values(["column", "y_A"])
    .reset_index(drop=True)
)

print("Casos pendentes:", len(pending))

roi_dir = (
    base /
    "automatic_segmentation/"
    "roi"
)

json_dir = (
    base /
    "automatic_segmentation/"
    "json"
)

out_dir = (
    base /
    "automatic_segmentation/"
    "review_sheets"
)

out_dir.mkdir(
    parents=True,
    exist_ok=True
)

panels = []

for _, row in pending.iterrows():

    col = str(row["column"])
    ka = int(row["kraken_A"])
    kb = int(row["kraken_B"])

    img = Image.open(
        roi_dir /
        f"1QS_col_{col}_ROI.png"
    ).convert("RGB")

    with open(
        json_dir /
        f"1QS_col_{col}_kraken.json",
        "r",
        encoding="utf-8"
    ) as f:
        seg = json.load(f)

    lines = seg["lines"]

    ba = lines[ka - 1]["baseline"]
    bb = lines[kb - 1]["baseline"]

    ya = sum(p[1] for p in ba) / len(ba)
    yb = sum(p[1] for p in bb) / len(bb)

    center = int((ya + yb) / 2)

    top = max(0, center - 110)
    bottom = min(img.height, center + 110)

    crop = img.crop(
        (0, top, img.width, bottom)
    )

    scale = 2

    crop = crop.resize(
        (
            crop.width * scale,
            crop.height * scale
        ),
        Image.Resampling.LANCZOS
    )

    header = 115

    panel = Image.new(
        "RGB",
        (
            crop.width,
            crop.height + header
        ),
        "white"
    )

    panel.paste(crop, (0, header))

    draw = ImageDraw.Draw(panel)

    draw.text(
        (15, 15),
        (
            f"{row['event_id']} | Col. {col} | "
            f"K{ka} + K{kb}"
        ),
        fill="black"
    )

    draw.text(
        (15, 50),
        (
            f"SMALL_FRAGMENT_REVIEW | "
            f"min_ratio={row['min_width_ratio']:.3f} | "
            f"y_gap={row['y_gap']:.1f} | "
            f"h_gap={row['horizontal_gap']:.0f}"
        ),
        fill="black"
    )

    pts_a = [
        (
            int(x * scale),
            int((y - top) * scale + header)
        )
        for x, y in ba
    ]

    pts_b = [
        (
            int(x * scale),
            int((y - top) * scale + header)
        )
        for x, y in bb
    ]

    if len(pts_a) >= 2:
        draw.line(
            pts_a,
            fill="red",
            width=5
        )
        draw.text(
            (pts_a[0][0], pts_a[0][1] - 22),
            f"K{ka}",
            fill="red"
        )

    if len(pts_b) >= 2:
        draw.line(
            pts_b,
            fill="blue",
            width=5
        )
        draw.text(
            (pts_b[0][0], pts_b[0][1] - 22),
            f"K{kb}",
            fill="blue"
        )

    panels.append(panel)

width = max(p.width for p in panels)
height = sum(p.height for p in panels)

sheet = Image.new(
    "RGB",
    (width, height),
    "white"
)

offset = 0

for panel in panels:
    sheet.paste(panel, (0, offset))
    offset += panel.height

output = (
    out_dir /
    "1QS_SMALL_FRAGMENT_FINAL_REVIEW.png"
)

sheet.save(output)

print("\nPrancha criada:")
print(output)

In [ ]:
from google.colab import files

files.download(
    "/content/Qumran_Analytics/1QS/"
    "automatic_segmentation/"
    "review_sheets/"
    "1QS_SMALL_FRAGMENT_FINAL_REVIEW.png"
)

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

base = Path("/content/Qumran_Analytics/1QS")

events = pd.read_csv(
    base /
    "automatic_segmentation/"
    "1QS_MERGE_EVENTS_CLASSIFIED_V2.csv"
)

# ============================================================
# 1. ADJUDICAÇÃO DOS 3 ÚLTIMOS CASOS
# ============================================================

small_decisions = {
    "M002": {
        "adjudication": "NOISE_FRAGMENT",
        "confidence": "MEDIUM_HIGH",
        "note": (
            "Very small isolated detection without convincing "
            "graphic continuity with the main baseline."
        )
    },

    "M006": {
        "adjudication": "TRUE_MERGE",
        "confidence": "HIGH",
        "note": (
            "Small fragment lies on the continuation of the main "
            "physical line with minimal horizontal separation."
        )
    },

    "M008": {
        "adjudication": "TRUE_MERGE",
        "confidence": "HIGH",
        "note": (
            "Short segment is a clear continuation of the long "
            "baseline at essentially the same vertical level."
        )
    }
}

small_review = events[
    events["event_id"].isin(
        small_decisions.keys()
    )
].copy()

small_review["adjudication"] = (
    small_review["event_id"]
    .map(
        lambda x:
        small_decisions[x]["adjudication"]
    )
)

small_review["confidence"] = (
    small_review["event_id"]
    .map(
        lambda x:
        small_decisions[x]["confidence"]
    )
)

small_review["review_note"] = (
    small_review["event_id"]
    .map(
        lambda x:
        small_decisions[x]["note"]
    )
)

small_review["review_source"] = (
    "AI_assisted_visual_adjudication"
)

small_review["researcher_confirmed"] = False
small_review["expert_verified"] = False

small_review["timestamp_utc"] = (
    datetime.now(timezone.utc).isoformat()
)

small_path = (
    base /
    "automatic_segmentation/"
    "1QS_SMALL_FRAGMENT_ADJUDICATION.csv"
)

small_review.to_csv(
    small_path,
    index=False,
    encoding="utf-8-sig"
)

print("Três casos finais:")
display(
    small_review[
        [
            "event_id",
            "column",
            "kraken_A",
            "kraken_B",
            "adjudication",
            "confidence"
        ]
    ]
)

In [ ]:
# ============================================================
# 2. TABELA FINAL DOS EVENTOS
# ============================================================

final = events.copy()

final["final_decision"] = pd.NA
final["final_confidence"] = pd.NA
final["decision_source"] = pd.NA
final["final_action"] = pd.NA

# ------------------------------------------------------------
# Decisões automáticas V2
# ------------------------------------------------------------

auto_map = {
    "AUTO_TRUE_MERGE":
        "TRUE_MERGE",

    "AUTO_NOISE_FRAGMENT":
        "NOISE_FRAGMENT",

    "AUTO_KEEP_SEPARATE":
        "KEEP_SEPARATE"
}

for auto_class, decision in auto_map.items():

    mask = (
        final["automatic_decision_v2"]
        == auto_class
    )

    final.loc[
        mask,
        "final_decision"
    ] = decision

    final.loc[
        mask,
        "final_confidence"
    ] = "AUTO_HIGH"

    final.loc[
        mask,
        "decision_source"
    ] = "automatic_rule_v2"


# ------------------------------------------------------------
# Revisões manuais anteriores
# ------------------------------------------------------------

manual_path = (
    base /
    "automatic_segmentation/"
    "1QS_MANUAL_MERGE_ADJUDICATION.csv"
)

manual = pd.read_csv(manual_path)

for _, r in manual.iterrows():

    mask = (
        final["event_id"]
        == r["event_id"]
    )

    final.loc[
        mask,
        "final_decision"
    ] = r["adjudication"]

    final.loc[
        mask,
        "final_confidence"
    ] = r["confidence"]

    final.loc[
        mask,
        "decision_source"
    ] = "visual_adjudication"

    final.loc[
        mask,
        "final_action"
    ] = r["action"]


# ------------------------------------------------------------
# Três últimos casos
# ------------------------------------------------------------

for _, r in small_review.iterrows():

    mask = (
        final["event_id"]
        == r["event_id"]
    )

    final.loc[
        mask,
        "final_decision"
    ] = r["adjudication"]

    final.loc[
        mask,
        "final_confidence"
    ] = r["confidence"]

    final.loc[
        mask,
        "decision_source"
    ] = "visual_adjudication"

In [ ]:
# ============================================================
# 3. DEFINE AÇÃO ESTRUTURAL
# ============================================================

def derive_action(row):

    # preserva ações manuais explícitas
    if pd.notna(row["final_action"]):
        return row["final_action"]

    a = int(row["kraken_A"])
    b = int(row["kraken_B"])

    decision = row["final_decision"]

    if decision == "TRUE_MERGE":
        return f"MERGE_K{a}_K{b}"

    if decision == "KEEP_SEPARATE":
        return f"KEEP_K{a}_K{b}_SEPARATE"

    if decision == "NOISE_FRAGMENT":

        # elimina o menor dos dois segmentos
        if row["width_A"] <= row["width_B"]:
            return f"DROP_K{a}_KEEP_K{b}"
        else:
            return f"DROP_K{b}_KEEP_K{a}"

    return "REVIEW_REQUIRED"


final["final_action"] = final.apply(
    derive_action,
    axis=1
)

# ============================================================
# 4. CHECAGEM FINAL
# ============================================================

unresolved = final[
    final["final_decision"].isna()
]

print("==============================")
print("EVENTOS ESTRUTURAIS FINAIS")
print("==============================")

print("Total de eventos:", len(final))
print("Não resolvidos:", len(unresolved))

print("\nDecisões finais:")
print(
    final["final_decision"]
    .value_counts(dropna=False)
)

final_path = (
    base /
    "automatic_segmentation/"
    "1QS_MERGE_EVENTS_FINAL.csv"
)

final.to_csv(
    final_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nArquivo final:")
print(final_path)

In [ ]:
import json
import re
import shutil
import pandas as pd
import numpy as np

from pathlib import Path
from PIL import Image, ImageDraw

# ============================================================
# CONFIGURAÇÃO
# ============================================================

base = Path("/content/Qumran_Analytics/1QS")

roi_dir = (
    base /
    "automatic_segmentation" /
    "roi"
)

json_dir = (
    base /
    "automatic_segmentation" /
    "json"
)

events_path = (
    base /
    "automatic_segmentation" /
    "1QS_MERGE_EVENTS_FINAL.csv"
)

final_root = (
    base /
    "final_segmentation"
)

crops_root = (
    final_root /
    "line_crops"
)

overlay_root = (
    final_root /
    "overlays"
)

final_root.mkdir(
    parents=True,
    exist_ok=True
)

crops_root.mkdir(
    parents=True,
    exist_ok=True
)

overlay_root.mkdir(
    parents=True,
    exist_ok=True
)

# limpa apenas resultados finais antigos
for p in crops_root.iterdir():
    if p.is_dir():
        shutil.rmtree(p)

for p in overlay_root.glob("*.png"):
    p.unlink()


columns = [
    "I", "II", "III", "IV", "V", "VI",
    "VII", "VIII", "IX", "X", "XI"
]

events = pd.read_csv(events_path)

# ============================================================
# UNION-FIND
# ============================================================

class UnionFind:

    def __init__(self, items):
        self.parent = {
            x: x
            for x in items
        }

    def find(self, x):

        if self.parent[x] != x:
            self.parent[x] = self.find(
                self.parent[x]
            )

        return self.parent[x]

    def union(self, a, b):

        ra = self.find(a)
        rb = self.find(b)

        if ra != rb:
            self.parent[rb] = ra


# ============================================================
# AUXILIARES
# ============================================================

def get_points(line):

    boundary = line.get(
        "boundary",
        []
    )

    baseline = line.get(
        "baseline",
        []
    )

    if boundary:
        return boundary

    return baseline


def get_baseline_y(line):

    baseline = line.get(
        "baseline",
        []
    )

    if not baseline:
        return np.nan

    return float(
        np.median(
            [p[1] for p in baseline]
        )
    )


def extract_drop_id(action):

    if pd.isna(action):
        return None

    m = re.search(
        r"DROP_K(\d+)",
        str(action)
    )

    if m:
        return int(m.group(1))

    return None


# ============================================================
# PROCESSAMENTO FINAL
# ============================================================

manifest_rows = []
summary_rows = []

for col in columns:

    print("\n================================")
    print("COLUNA", col)
    print("================================")

    roi_path = (
        roi_dir /
        f"1QS_col_{col}_ROI.png"
    )

    json_path = (
        json_dir /
        f"1QS_col_{col}_kraken.json"
    )

    if not roi_path.exists():
        raise FileNotFoundError(
            roi_path
        )

    if not json_path.exists():
        raise FileNotFoundError(
            json_path
        )

    img = Image.open(
        roi_path
    ).convert("RGB")

    with open(
        json_path,
        "r",
        encoding="utf-8"
    ) as f:

        seg = json.load(f)

    kraken_lines = seg.get(
        "lines",
        []
    )

    ids = list(
        range(
            1,
            len(kraken_lines) + 1
        )
    )

    uf = UnionFind(ids)

    dropped = set()

    col_events = events[
        events["column"].astype(str)
        == col
    ].copy()

    # ========================================================
    # 1. IDENTIFICA DROPS
    # ========================================================

    for _, row in col_events.iterrows():

        decision = str(
            row["final_decision"]
        )

        action = row.get(
            "final_action",
            pd.NA
        )

        if decision in {
            "NOISE_FRAGMENT",
            "DUPLICATE_BASELINE"
        }:

            drop_id = extract_drop_id(
                action
            )

            if drop_id is None:

                # fallback para NOISE_FRAGMENT:
                # elimina o menor segmento
                a = int(row["kraken_A"])
                b = int(row["kraken_B"])

                wa = float(row["width_A"])
                wb = float(row["width_B"])

                drop_id = (
                    a if wa <= wb else b
                )

            dropped.add(
                drop_id
            )

    # ========================================================
    # 2. APLICA TRUE_MERGE
    # ========================================================

    merge_edges = []

    for _, row in col_events.iterrows():

        if (
            str(row["final_decision"])
            != "TRUE_MERGE"
        ):
            continue

        a = int(row["kraken_A"])
        b = int(row["kraken_B"])

        # proteção metodológica:
        # não fundir baseline já declarada ruído
        if (
            a in dropped
            or b in dropped
        ):

            raise RuntimeError(
                f"Conflito na coluna {col}: "
                f"TRUE_MERGE envolvendo baseline "
                f"descartada K{a}/K{b}"
            )

        uf.union(
            a,
            b
        )

        merge_edges.append(
            (a, b)
        )

    # ========================================================
    # 3. CRIA GRUPOS CONSOLIDADOS
    # ========================================================

    groups = {}

    for kid in ids:

        if kid in dropped:
            continue

        root = uf.find(kid)

        groups.setdefault(
            root,
            []
        ).append(kid)

    consolidated = []

    for source_ids in groups.values():

        ys = []

        all_points = []

        for kid in source_ids:

            line = kraken_lines[
                kid - 1
            ]

            y = get_baseline_y(
                line
            )

            if not np.isnan(y):
                ys.append(y)

            pts = get_points(
                line
            )

            all_points.extend(
                pts
            )

        if not all_points:
            continue

        xs = [
            p[0]
            for p in all_points
        ]

        pys = [
            p[1]
            for p in all_points
        ]

        y_center = float(
            np.median(ys)
        )

        consolidated.append({
            "source_ids":
                sorted(source_ids),

            "y_center":
                y_center,

            "x_min":
                int(min(xs)),

            "x_max":
                int(max(xs)),

            "y_min":
                int(min(pys)),

            "y_max":
                int(max(pys))
        })

    # ordem física: topo -> base
    consolidated = sorted(
        consolidated,
        key=lambda x: x["y_center"]
    )

    # ========================================================
    # 4. DIRETÓRIO DA COLUNA
    # ========================================================

    col_crop_dir = (
        crops_root /
        f"col_{col}"
    )

    col_crop_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # ========================================================
    # 5. GERA CROPS
    # ========================================================

    PAD_X = 20
    PAD_Y = 18

    overlay = img.copy()

    draw = ImageDraw.Draw(
        overlay
    )

    for line_no, group in enumerate(
        consolidated,
        start=1
    ):

        x1 = max(
            0,
            group["x_min"] - PAD_X
        )

        x2 = min(
            img.width,
            group["x_max"] + PAD_X
        )

        y1 = max(
            0,
            group["y_min"] - PAD_Y
        )

        y2 = min(
            img.height,
            group["y_max"] + PAD_Y
        )

        # garante altura mínima
        MIN_HEIGHT = 55

        if (
            y2 - y1
            < MIN_HEIGHT
        ):

            center = (
                y1 + y2
            ) // 2

            y1 = max(
                0,
                center - MIN_HEIGHT // 2
            )

            y2 = min(
                img.height,
                y1 + MIN_HEIGHT
            )

        line_id = (
            f"1QS_{col}_L"
            f"{line_no:03d}"
        )

        crop_path = (
            col_crop_dir /
            f"{line_id}.png"
        )

        crop = img.crop(
            (
                x1,
                y1,
                x2,
                y2
            )
        )

        crop.save(
            crop_path
        )

        source_ids = (
            group["source_ids"]
        )

        merged = (
            len(source_ids) > 1
        )

        # ----------------------------------------------------
        # procura eventos relacionados
        # ----------------------------------------------------

        related_events = []

        for _, ev in col_events.iterrows():

            a = int(
                ev["kraken_A"]
            )

            b = int(
                ev["kraken_B"]
            )

            if (
                a in source_ids
                or
                b in source_ids
            ):

                related_events.append(
                    str(ev["event_id"])
                )

        manifest_rows.append({
            "line_id":
                line_id,

            "column":
                col,

            "line_number":
                line_no,

            "source_kraken_ids":
                ",".join(
                    map(
                        str,
                        source_ids
                    )
                ),

            "merged":
                merged,

            "n_source_baselines":
                len(source_ids),

            "related_events":
                ",".join(
                    sorted(
                        set(
                            related_events
                        )
                    )
                ),

            "x1":
                x1,

            "y1":
                y1,

            "x2":
                x2,

            "y2":
                y2,

            "y_center":
                group["y_center"],

            "crop_width":
                crop.width,

            "crop_height":
                crop.height,

            "crop_path":
                str(crop_path),

            "segmentation_stage":
                "FINAL_CONSOLIDATED",

            "researcher_reviewed":
                False,

            "expert_verified":
                False
        })

        # ----------------------------------------------------
        # overlay visual
        # ----------------------------------------------------

        draw.rectangle(
            (
                x1,
                y1,
                x2,
                y2
            ),
            outline="red",
            width=2
        )

        draw.text(
            (
                x1 + 3,
                y1 + 3
            ),
            f"L{line_no:03d}",
            fill="blue"
        )

    overlay_path = (
        overlay_root /
        f"1QS_col_{col}_FINAL_OVERLAY.png"
    )

    overlay.save(
        overlay_path
    )

    summary_rows.append({
        "column":
            col,

        "raw_kraken_baselines":
            len(ids),

        "dropped_baselines":
            len(dropped),

        "true_merge_events":
            len(merge_edges),

        "final_physical_lines":
            len(consolidated),

        "overlay_path":
            str(overlay_path)
    })

    print(
        "Kraken bruto:",
        len(ids)
    )

    print(
        "Descartadas:",
        len(dropped)
    )

    print(
        "Merges aplicados:",
        len(merge_edges)
    )

    print(
        "Linhas físicas finais:",
        len(consolidated)
    )


# ============================================================
# 6. MANIFESTO FINAL
# ============================================================

manifest = pd.DataFrame(
    manifest_rows
)

manifest_path = (
    final_root /
    "1QS_FINAL_LINE_MANIFEST.csv"
)

manifest.to_csv(
    manifest_path,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 7. RESUMO FINAL
# ============================================================

summary = pd.DataFrame(
    summary_rows
)

summary_path = (
    final_root /
    "1QS_FINAL_SEGMENTATION_SUMMARY.csv"
)

summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n\n======================================")
print("SEGMENTAÇÃO CONSOLIDADA CONCLUÍDA")
print("======================================")

display(summary)

print(
    "\nTOTAL DE LINHAS FÍSICAS:",
    len(manifest)
)

print("\nManifesto:")
print(manifest_path)

print("\nResumo:")
print(summary_path)

In [ ]:
# ============================================================
# SANITY CHECK DA SEGMENTAÇÃO FINAL
# ============================================================

problems = []

for col in columns:

    col_manifest = manifest[
        manifest["column"].astype(str)
        == col
    ]

    seen = []

    for ids_string in col_manifest[
        "source_kraken_ids"
    ]:

        ids_local = [
            int(x)
            for x in str(ids_string).split(",")
            if str(x).strip()
        ]

        seen.extend(
            ids_local
        )

    duplicates = pd.Series(
        seen
    ).value_counts()

    duplicated_ids = (
        duplicates[
            duplicates > 1
        ]
        .index
        .tolist()
    )

    if duplicated_ids:

        problems.append({
            "column": col,
            "problem":
                "DUPLICATED_SOURCE_ID",
            "ids":
                duplicated_ids
        })

print("==============================")
print("SANITY CHECK")
print("==============================")

if not problems:
    print("Nenhuma inconsistência estrutural encontrada.")
else:
    display(
        pd.DataFrame(
            problems
        )
    )

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw

base = Path("/content/Qumran_Analytics/1QS")

overlay_dir = (
    base /
    "final_segmentation" /
    "overlays"
)

columns = [
    "I", "II", "III", "IV", "V", "VI",
    "VII", "VIII", "IX", "X", "XI"
]

panels = []

for col in columns:

    path = (
        overlay_dir /
        f"1QS_col_{col}_FINAL_OVERLAY.png"
    )

    img = Image.open(path).convert("RGB")

    # reduz só para inspeção
    target_width = 700

    target_height = round(
        img.height *
        target_width /
        img.width
    )

    img = img.resize(
        (target_width, target_height),
        Image.Resampling.LANCZOS
    )

    header = 45

    panel = Image.new(
        "RGB",
        (
            target_width,
            target_height + header
        ),
        "white"
    )

    panel.paste(
        img,
        (0, header)
    )

    draw = ImageDraw.Draw(panel)

    draw.text(
        (15, 12),
        f"1QS — Coluna {col}",
        fill="black"
    )

    panels.append(panel)

# 2 colunas de painéis
ncols = 2
nrows = (len(panels) + 1) // 2

cell_w = max(p.width for p in panels)
cell_h = max(p.height for p in panels)

sheet = Image.new(
    "RGB",
    (
        cell_w * ncols,
        cell_h * nrows
    ),
    "white"
)

for i, panel in enumerate(panels):

    x = (i % ncols) * cell_w
    y = (i // ncols) * cell_h

    sheet.paste(
        panel,
        (x, y)
    )

output = (
    base /
    "final_segmentation" /
    "1QS_ALL_FINAL_OVERLAYS_CONTACT_SHEET.jpg"
)

sheet.save(
    output,
    "JPEG",
    quality=90,
    optimize=True
)

print("Prancha criada:")
print(output)

In [ ]:
from google.colab import files

files.download(
    "/content/Qumran_Analytics/1QS/"
    "final_segmentation/"
    "1QS_ALL_FINAL_OVERLAYS_CONTACT_SHEET.jpg"
)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

base = Path("/content/Qumran_Analytics/1QS")

manifest_path = (
    base /
    "final_segmentation" /
    "1QS_FINAL_LINE_MANIFEST.csv"
)

manifest = pd.read_csv(manifest_path)

# estatísticas por coluna
stats = (
    manifest
    .groupby("column")
    .agg(
        median_width=("crop_width", "median"),
        median_height=("crop_height", "median")
    )
    .reset_index()
)

m = manifest.merge(
    stats,
    on="column",
    how="left"
)

m["width_ratio"] = (
    m["crop_width"] /
    m["median_width"]
)

m["height_ratio"] = (
    m["crop_height"] /
    m["median_height"]
)

# candidatos muito pequenos ou geometricamente estranhos
m["flag_tiny_width"] = (
    m["width_ratio"] < 0.18
)

m["flag_narrow"] = (
    m["width_ratio"] < 0.30
)

m["flag_vertical_shape"] = (
    m["crop_height"] >
    m["crop_width"] * 0.70
)

m["final_visual_review"] = (
    m["flag_tiny_width"]
    | m["flag_vertical_shape"]
)

suspects = (
    m[
        m["final_visual_review"]
    ]
    .sort_values(
        ["column", "y_center"]
    )
)

print(
    "Linhas finais:",
    len(m)
)

print(
    "Casos suspeitos finais:",
    len(suspects)
)

display(
    suspects[
        [
            "line_id",
            "column",
            "line_number",
            "source_kraken_ids",
            "crop_width",
            "crop_height",
            "width_ratio",
            "y_center",
            "related_events"
        ]
    ]
)

In [ ]:
import pandas as pd
from pathlib import Path
from PIL import Image, ImageDraw

base = Path("/content/Qumran_Analytics/1QS")

manifest = pd.read_csv(
    base /
    "final_segmentation" /
    "1QS_FINAL_LINE_MANIFEST.csv"
)

# ============================================================
# RECRIA A DETECÇÃO DOS SUSPEITOS
# ============================================================

stats = (
    manifest
    .groupby("column")
    .agg(
        median_width=("crop_width", "median"),
        median_height=("crop_height", "median")
    )
    .reset_index()
)

m = manifest.merge(
    stats,
    on="column",
    how="left"
)

m["width_ratio"] = (
    m["crop_width"] /
    m["median_width"]
)

m["flag_tiny_width"] = (
    m["width_ratio"] < 0.18
)

m["flag_vertical_shape"] = (
    m["crop_height"] >
    m["crop_width"] * 0.70
)

m["final_visual_review"] = (
    m["flag_tiny_width"]
    |
    m["flag_vertical_shape"]
)

suspects = (
    m[
        m["final_visual_review"]
    ]
    .sort_values(
        ["column", "y_center"]
    )
    .reset_index(drop=True)
)

print("Casos para revisão:", len(suspects))

# ============================================================
# DIRETÓRIOS
# ============================================================

roi_dir = (
    base /
    "automatic_segmentation" /
    "roi"
)

out_dir = (
    base /
    "final_segmentation" /
    "review"
)

out_dir.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# CRIA PAINÉIS
# ============================================================

panels = []

CONTEXT_Y = 130
SCALE = 2
HEADER = 100

for _, row in suspects.iterrows():

    col = str(row["column"])
    line_id = str(row["line_id"])

    roi_path = (
        roi_dir /
        f"1QS_col_{col}_ROI.png"
    )

    img = Image.open(
        roi_path
    ).convert("RGB")

    y_center = int(
        row["y_center"]
    )

    top = max(
        0,
        y_center - CONTEXT_Y
    )

    bottom = min(
        img.height,
        y_center + CONTEXT_Y
    )

    # contexto em toda a largura da coluna
    crop = img.crop(
        (
            0,
            top,
            img.width,
            bottom
        )
    )

    crop = crop.resize(
        (
            crop.width * SCALE,
            crop.height * SCALE
        ),
        Image.Resampling.LANCZOS
    )

    panel = Image.new(
        "RGB",
        (
            crop.width,
            crop.height + HEADER
        ),
        "white"
    )

    panel.paste(
        crop,
        (0, HEADER)
    )

    draw = ImageDraw.Draw(
        panel
    )

    draw.text(
        (15, 12),
        (
            f"{line_id} | Coluna {col} | "
            f"Kraken {row['source_kraken_ids']}"
        ),
        fill="black"
    )

    draw.text(
        (15, 48),
        (
            f"crop={int(row['crop_width'])}x"
            f"{int(row['crop_height'])} | "
            f"width_ratio={row['width_ratio']:.3f}"
        ),
        fill="black"
    )

    # ========================================================
    # LINHAS FINAIS VIZINHAS EM AZUL
    # ========================================================

    neighbors = manifest[
        (
            manifest["column"].astype(str)
            == col
        )
        &
        (
            manifest["y_center"]
            >= top
        )
        &
        (
            manifest["y_center"]
            <= bottom
        )
    ]

    for _, n in neighbors.iterrows():

        nx1 = int(n["x1"] * SCALE)
        nx2 = int(n["x2"] * SCALE)

        ny1 = int(
            (n["y1"] - top) *
            SCALE +
            HEADER
        )

        ny2 = int(
            (n["y2"] - top) *
            SCALE +
            HEADER
        )

        # suspeito = vermelho
        # vizinhos = azul
        if (
            str(n["line_id"])
            == line_id
        ):

            color = "red"
            width = 5

        else:

            color = "blue"
            width = 2

        draw.rectangle(
            (
                nx1,
                ny1,
                nx2,
                ny2
            ),
            outline=color,
            width=width
        )

        draw.text(
            (
                nx1 + 4,
                ny1 + 3
            ),
            str(n["line_id"]),
            fill=color
        )

    panels.append(
        panel
    )

# ============================================================
# MONTA PRANCHA ÚNICA
# ============================================================

max_width = max(
    p.width
    for p in panels
)

total_height = sum(
    p.height
    for p in panels
)

sheet = Image.new(
    "RGB",
    (
        max_width,
        total_height
    ),
    "white"
)

offset = 0

for panel in panels:

    sheet.paste(
        panel,
        (0, offset)
    )

    offset += panel.height

output = (
    out_dir /
    "1QS_FINAL_8_SUSPECTS_REVIEW.jpg"
)

sheet.save(
    output,
    "JPEG",
    quality=92
)

print("\nPrancha criada:")
print(output)

In [ ]:
from google.colab import files

files.download(
    "/content/Qumran_Analytics/1QS/"
    "final_segmentation/review/"
    "1QS_FINAL_8_SUSPECTS_REVIEW.jpg"
)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

base = Path("/content/Qumran_Analytics/1QS")

manifest = pd.read_csv(
    base /
    "final_segmentation" /
    "1QS_FINAL_LINE_MANIFEST.csv"
)

targets = [
    "1QS_I_L014",
    "1QS_I_L026",
    "1QS_VIII_L015",
    "1QS_X_L019"
]

for target in targets:

    target_rows = manifest[
        manifest["line_id"] == target
    ]

    if len(target_rows) == 0:
        print(f"\nATENÇÃO: {target} não encontrado.")
        continue

    t = target_rows.iloc[0]

    col = str(t["column"])

    candidates = manifest[
        (
            manifest["column"].astype(str) == col
        )
        &
        (
            manifest["line_id"] != target
        )
    ].copy()

    # ========================================================
    # DISTÂNCIA VERTICAL ENTRE CENTROS
    # ========================================================

    candidates["dy"] = (
        candidates["y_center"] -
        float(t["y_center"])
    ).abs()

    # ========================================================
    # INTERSEÇÃO VERTICAL ENTRE AS CAIXAS
    # ========================================================

    candidates["vertical_overlap"] = candidates.apply(
        lambda r: max(
            0.0,
            min(
                float(t["y2"]),
                float(r["y2"])
            )
            -
            max(
                float(t["y1"]),
                float(r["y1"])
            )
        ),
        axis=1
    )

    # ========================================================
    # DENOMINADOR CORRETO:
    # mínimo entre altura do target e altura de CADA candidato
    # ========================================================

    denominator = np.minimum(
        float(t["crop_height"]),
        candidates["crop_height"].astype(float)
    )

    # proteção contra divisão por zero
    denominator = denominator.replace(
        0,
        np.nan
    )

    candidates["vertical_overlap_ratio"] = (
        candidates["vertical_overlap"] /
        denominator
    ).fillna(0)

    # ========================================================
    # DISTÂNCIA HORIZONTAL
    # 0 = caixas se sobrepõem horizontalmente
    # ========================================================

    def horizontal_gap(r):

        tx1 = float(t["x1"])
        tx2 = float(t["x2"])

        rx1 = float(r["x1"])
        rx2 = float(r["x2"])

        if rx2 < tx1:
            return tx1 - rx2

        if tx2 < rx1:
            return rx1 - tx2

        return 0.0

    candidates["horizontal_gap"] = candidates.apply(
        horizontal_gap,
        axis=1
    )

    # ========================================================
    # ORDENA OS MELHORES VIZINHOS
    # ========================================================

    candidates = candidates.sort_values(
        [
            "dy",
            "vertical_overlap_ratio",
            "horizontal_gap"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )

    print("\n" + "=" * 70)
    print(target)
    print(
        f"Coluna {col} | "
        f"y_center={float(t['y_center']):.1f} | "
        f"box=({int(t['x1'])},{int(t['y1'])})-"
        f"({int(t['x2'])},{int(t['y2'])})"
    )
    print("=" * 70)

    display(
        candidates[
            [
                "line_id",
                "source_kraken_ids",
                "y_center",
                "dy",
                "vertical_overlap_ratio",
                "horizontal_gap",
                "x1",
                "x2",
                "y1",
                "y2"
            ]
        ].head(5)
    )

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import shutil

# ============================================================
# CONFIGURAÇÃO
# ============================================================

base = Path("/content/Qumran_Analytics/1QS")

manifest_path = (
    base /
    "final_segmentation" /
    "1QS_FINAL_LINE_MANIFEST.csv"
)

roi_dir = (
    base /
    "automatic_segmentation" /
    "roi"
)

frozen_root = (
    base /
    "frozen_segmentation"
)

main_dir = frozen_root / "main_text"
secondary_dir = frozen_root / "secondary_text"

main_dir.mkdir(parents=True, exist_ok=True)
secondary_dir.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(manifest_path)

columns_order = [
    "I", "II", "III", "IV", "V", "VI",
    "VII", "VIII", "IX", "X", "XI"
]

# ============================================================
# 1. PRESERVA O MANIFESTO ORIGINAL COMO AUDITORIA
# ============================================================

audit = manifest.copy()

audit["final_role"] = "MAIN_TEXT"
audit["final_confidence"] = "UNREVIEWED_DEFAULT"
audit["final_review_note"] = ""
audit["merged_into"] = pd.NA

# ============================================================
# 2. ADJUDICAÇÕES FINAIS
# ============================================================

decisions = {

    "1QS_I_L014": (
        "SECONDARY_TEXT",
        "MEDIUM_HIGH",
        "Probable interlinear/supralinear textual element."
    ),

    "1QS_I_L026": (
        "MERGED_COMPONENT",
        "HIGH",
        "Fragment belongs to same physical line as 1QS_I_L027."
    ),

    "1QS_III_L027": (
        "NOISE",
        "HIGH",
        "Non-textual manuscript/support feature."
    ),

    "1QS_IV_L008": (
        "MAIN_TEXT",
        "HIGH",
        "Legitimate short main-text line."
    ),

    "1QS_V_L027": (
        "SECONDARY_TEXT",
        "MEDIUM",
        "Possible intentional secondary mark/text; preserved outside main HTR."
    ),

    "1QS_VII_L008": (
        "SECONDARY_TEXT",
        "HIGH",
        "Clear supralinear/interlinear textual element."
    ),

    "1QS_VIII_L015": (
        "SECONDARY_TEXT",
        "MEDIUM_HIGH",
        "Interlinear/supralinear element between main lines."
    ),

    "1QS_X_L019": (
        "SECONDARY_TEXT",
        "MEDIUM_HIGH",
        "Interlinear textual element distinct from main writing line."
    )
}

for line_id, (role, conf, note) in decisions.items():

    mask = audit["line_id"] == line_id

    audit.loc[mask, "final_role"] = role
    audit.loc[mask, "final_confidence"] = conf
    audit.loc[mask, "final_review_note"] = note

audit.loc[
    audit["line_id"] == "1QS_I_L026",
    "merged_into"
] = "1QS_I_L027"

# ============================================================
# 3. CONSTRÓI MANIFESTO ATIVO
# ============================================================

active = audit[
    ~audit["final_role"].isin(
        ["NOISE", "MERGED_COMPONENT"]
    )
].copy()

# ============================================================
# 4. APLICA MERGE I_L026 + I_L027
# ============================================================

component = audit[
    audit["line_id"] == "1QS_I_L026"
].iloc[0]

canonical_idx = active.index[
    active["line_id"] == "1QS_I_L027"
][0]

canonical = active.loc[canonical_idx].copy()

# união das caixas
new_x1 = int(min(component["x1"], canonical["x1"]))
new_y1 = int(min(component["y1"], canonical["y1"]))
new_x2 = int(max(component["x2"], canonical["x2"]))
new_y2 = int(max(component["y2"], canonical["y2"]))

active.loc[canonical_idx, "x1"] = new_x1
active.loc[canonical_idx, "y1"] = new_y1
active.loc[canonical_idx, "x2"] = new_x2
active.loc[canonical_idx, "y2"] = new_y2

active.loc[canonical_idx, "crop_width"] = new_x2 - new_x1
active.loc[canonical_idx, "crop_height"] = new_y2 - new_y1

# centro vertical consolidado
active.loc[canonical_idx, "y_center"] = np.mean(
    [
        float(component["y_center"]),
        float(canonical["y_center"])
    ]
)

# une IDs Kraken
def parse_ids(value):
    return [
        int(x)
        for x in str(value).split(",")
        if x.strip()
    ]

merged_source_ids = sorted(
    set(
        parse_ids(component["source_kraken_ids"]) +
        parse_ids(canonical["source_kraken_ids"])
    )
)

active.loc[
    canonical_idx,
    "source_kraken_ids"
] = ",".join(map(str, merged_source_ids))

active.loc[
    canonical_idx,
    "n_source_baselines"
] = len(merged_source_ids)

active.loc[
    canonical_idx,
    "merged"
] = True

active.loc[
    canonical_idx,
    "final_review_note"
] = (
    "Final physical line created by merging "
    "1QS_I_L026 with 1QS_I_L027."
)

active.loc[
    canonical_idx,
    "final_confidence"
] = "HIGH"

# ============================================================
# 5. ORDENA FISICAMENTE
# ============================================================

col_rank = {
    col: i
    for i, col in enumerate(columns_order)
}

active["column_rank"] = (
    active["column"]
    .astype(str)
    .map(col_rank)
)

active = (
    active
    .sort_values(
        ["column_rank", "y_center"]
    )
    .reset_index(drop=True)
)

# ============================================================
# 6. CRIA IDs DEFINITIVOS DO HTR
# ============================================================

active["htr_line_id"] = pd.NA
active["secondary_id"] = pd.NA

for col in columns_order:

    # MAIN TEXT
    mask_main = (
        (active["column"].astype(str) == col)
        &
        (active["final_role"] == "MAIN_TEXT")
    )

    main_indices = active[
        mask_main
    ].sort_values("y_center").index

    for n, idx in enumerate(main_indices, start=1):

        active.loc[
            idx,
            "htr_line_id"
        ] = f"1QS_{col}_M{n:03d}"

    # SECONDARY TEXT
    mask_secondary = (
        (active["column"].astype(str) == col)
        &
        (active["final_role"] == "SECONDARY_TEXT")
    )

    sec_indices = active[
        mask_secondary
    ].sort_values("y_center").index

    for n, idx in enumerate(sec_indices, start=1):

        active.loc[
            idx,
            "secondary_id"
        ] = f"1QS_{col}_S{n:03d}"

# ============================================================
# 7. RECRIA CROPS FINAIS DIRETAMENTE DOS ROIs
# ============================================================

for _, row in active.iterrows():

    col = str(row["column"])

    roi_path = (
        roi_dir /
        f"1QS_col_{col}_ROI.png"
    )

    img = Image.open(
        roi_path
    ).convert("RGB")

    x1 = int(row["x1"])
    y1 = int(row["y1"])
    x2 = int(row["x2"])
    y2 = int(row["y2"])

    crop = img.crop(
        (x1, y1, x2, y2)
    )

    if row["final_role"] == "MAIN_TEXT":

        filename = (
            str(row["htr_line_id"])
            + ".png"
        )

        output = (
            main_dir /
            filename
        )

    else:

        filename = (
            str(row["secondary_id"])
            + ".png"
        )

        output = (
            secondary_dir /
            filename
        )

    crop.save(output)

    active.loc[
        active["line_id"] == row["line_id"],
        "frozen_crop_path"
    ] = str(output)

# ============================================================
# 8. SALVA MANIFESTOS
# ============================================================

audit_path = (
    frozen_root /
    "1QS_SEGMENTATION_AUDIT_TRAIL.csv"
)

audit.to_csv(
    audit_path,
    index=False,
    encoding="utf-8-sig"
)

active_path = (
    frozen_root /
    "1QS_FROZEN_SEGMENTATION_MANIFEST.csv"
)

active.to_csv(
    active_path,
    index=False,
    encoding="utf-8-sig"
)

htr = active[
    active["final_role"] == "MAIN_TEXT"
].copy()

htr_path = (
    frozen_root /
    "1QS_FINAL_HTR_MANIFEST.csv"
)

htr.to_csv(
    htr_path,
    index=False,
    encoding="utf-8-sig"
)

secondary = active[
    active["final_role"] == "SECONDARY_TEXT"
].copy()

secondary_path = (
    frozen_root /
    "1QS_SECONDARY_TEXT_MANIFEST.csv"
)

secondary.to_csv(
    secondary_path,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 9. RESUMO
# ============================================================

print("=" * 65)
print("1QS — SEGMENTAÇÃO CONGELADA")
print("=" * 65)

print("\nObjetos originais consolidados:", len(audit))

print(
    "Linhas MAIN_TEXT:",
    len(htr)
)

print(
    "Objetos SECONDARY_TEXT:",
    len(secondary)
)

print(
    "NOISE:",
    (audit["final_role"] == "NOISE").sum()
)

print(
    "MERGED_COMPONENT:",
    (audit["final_role"] == "MERGED_COMPONENT").sum()
)

print(
    "\nObjetos ativos após consolidação:",
    len(active)
)

print("\nArquivos:")
print(active_path)
print(htr_path)
print(secondary_path)
print(audit_path)

In [ ]:
# ============================================================
# FINAL FREEZE CHECK
# ============================================================

print("MAIN_TEXT IDs únicos:",
      htr["htr_line_id"].is_unique)

print(
    "MAIN_TEXT sem ID:",
    htr["htr_line_id"].isna().sum()
)

print(
    "SECONDARY_TEXT sem ID:",
    secondary["secondary_id"].isna().sum()
)

# verifica IDs Kraken usados mais de uma vez no corpus principal

all_source_ids = []

for value in htr["source_kraken_ids"]:

    all_source_ids.extend(
        [
            int(x)
            for x in str(value).split(",")
            if x.strip()
        ]
    )

duplicates = (
    pd.Series(all_source_ids)
    .value_counts()
)

duplicates = duplicates[
    duplicates > 1
]

print(
    "IDs Kraken duplicados no HTR:",
    len(duplicates)
)

print("\nDistribuição por coluna:")

display(
    htr.groupby("column")
       .size()
       .reindex(columns_order)
       .rename("main_text_lines")
       .reset_index()
)

In [ ]:
# ============================================================
# SANITY CHECK CORRETO:
# Kraken ID é único DENTRO DE CADA COLUNA
# ============================================================

source_records = []

for _, row in htr.iterrows():

    col = str(row["column"])

    ids = [
        int(x)
        for x in str(row["source_kraken_ids"]).split(",")
        if x.strip()
    ]

    for kid in ids:

        source_records.append({
            "column": col,
            "kraken_id": kid,
            "htr_line_id": row["htr_line_id"]
        })

sources = pd.DataFrame(source_records)

# procura o MESMO Kraken ID sendo usado
# em mais de uma linha HTR DA MESMA COLUNA
duplicate_mask = sources.duplicated(
    subset=["column", "kraken_id"],
    keep=False
)

real_duplicates = sources[
    duplicate_mask
].sort_values(
    ["column", "kraken_id"]
)

print("=" * 60)
print("SANITY CHECK CORRIGIDO")
print("=" * 60)

print(
    "MAIN_TEXT IDs únicos:",
    htr["htr_line_id"].is_unique
)

print(
    "MAIN_TEXT sem ID:",
    htr["htr_line_id"].isna().sum()
)

print(
    "SECONDARY_TEXT sem ID:",
    secondary["secondary_id"].isna().sum()
)

print(
    "Duplicações reais Kraken "
    "(mesma coluna + mesmo K):",
    len(real_duplicates)
)

if len(real_duplicates) > 0:
    display(real_duplicates)
else:
    print(
        "\n✓ Nenhuma baseline Kraken foi reutilizada "
        "em duas linhas HTR da mesma coluna."
    )

print("\nTOTAL MAIN_TEXT:", len(htr))

print("\nDistribuição por coluna:")

display(
    htr.groupby("column")
       .size()
       .reindex(columns_order)
       .rename("main_text_lines")
       .reset_index()
)

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from datetime import datetime, timezone

base = Path("/content/Qumran_Analytics/1QS")

frozen = base / "frozen_segmentation"

files_to_hash = [
    frozen / "1QS_FROZEN_SEGMENTATION_MANIFEST.csv",
    frozen / "1QS_FINAL_HTR_MANIFEST.csv",
    frozen / "1QS_SECONDARY_TEXT_MANIFEST.csv",
    frozen / "1QS_SEGMENTATION_AUDIT_TRAIL.csv",
]

records = []

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()

for path in files_to_hash:

    records.append({
        "file": path.name,
        "sha256": sha256_file(path)
    })

# também registra todos os 284 crops MAIN_TEXT
main_dir = frozen / "main_text"

for path in sorted(main_dir.glob("*.png")):

    records.append({
        "file": f"main_text/{path.name}",
        "sha256": sha256_file(path)
    })

hashes = pd.DataFrame(records)

hash_path = frozen / "1QS_FREEZE_SHA256.csv"

hashes.to_csv(
    hash_path,
    index=False,
    encoding="utf-8-sig"
)

metadata = {
    "manuscript": "1QS",
    "segmentation_status": "FROZEN",
    "main_text_lines": 284,
    "secondary_text_objects": 5,
    "noise_objects": 1,
    "merged_components": 1,
    "freeze_timestamp_utc":
        datetime.now(timezone.utc).isoformat(),
    "note":
        "Frozen segmentation dataset prior to HTR training."
}

metadata_path = (
    frozen /
    "1QS_FREEZE_METADATA.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2
    )

print("SEGMENTAÇÃO 1QS LACRADA")
print("Hashes:", hash_path)
print("Metadata:", metadata_path)
print("Arquivos hashados:", len(hashes))

In [ ]:
import pandas as pd
import numpy as np
import shutil

from pathlib import Path

base = Path("/content/Qumran_Analytics/1QS")

frozen = (
    base /
    "frozen_segmentation"
)

htr_manifest = pd.read_csv(
    frozen /
    "1QS_FINAL_HTR_MANIFEST.csv"
)

gt_root = (
    base /
    "htr_ground_truth"
)

images_dir = (
    gt_root /
    "images"
)

images_dir.mkdir(
    parents=True,
    exist_ok=True
)

columns = [
    "I", "II", "III", "IV", "V", "VI",
    "VII", "VIII", "IX", "X", "XI"
]

SEED = 20261012

selection_rows = []

# ============================================================
# 7 LINHAS POR COLUNA
# 5 TRAIN + 1 VALIDATION + 1 TEST
# ============================================================

for col_index, col in enumerate(columns):

    subset = (
        htr_manifest[
            htr_manifest["column"].astype(str)
            == col
        ]
        .copy()
        .sort_values("y_center")
    )

    if len(subset) < 7:
        raise ValueError(
            f"Coluna {col} tem menos de 7 linhas."
        )

    # sorteio reproduzível
    selected = subset.sample(
        n=7,
        random_state=SEED + col_index
    ).copy()

    # embaralhamento independente para definir splits
    selected = selected.sample(
        frac=1,
        random_state=SEED + 100 + col_index
    ).reset_index(drop=True)

    selected["split"] = (
        ["TRAIN"] * 5
        + ["VALIDATION"]
        + ["TEST"]
    )

    selection_rows.append(selected)

selection = pd.concat(
    selection_rows,
    ignore_index=True
)

# ============================================================
# COPIA IMAGENS + CRIA .gt.txt VAZIO
# ============================================================

for _, row in selection.iterrows():

    line_id = str(
        row["htr_line_id"]
    )

    src = Path(
        row["frozen_crop_path"]
    )

    dst = (
        images_dir /
        f"{line_id}.png"
    )

    shutil.copy2(
        src,
        dst
    )

    gt_txt = (
        images_dir /
        f"{line_id}.gt.txt"
    )

    # placeholder vazio
    if not gt_txt.exists():
        gt_txt.write_text(
            "",
            encoding="utf-8"
        )

# ============================================================
# METADADOS DA SELEÇÃO
# ============================================================

selection["transcription_status"] = "PENDING"

selection["reference_visible_during_transcription"] = False

selection["researcher_confirmed"] = False

selection["expert_verified"] = False

selection_path = (
    gt_root /
    "1QS_GT_SELECTION.csv"
)

selection.to_csv(
    selection_path,
    index=False,
    encoding="utf-8-sig"
)

print("=================================")
print("GROUND TRUTH PILOT")
print("=================================")

print("Total selecionado:", len(selection))

print("\nSplits:")
print(
    selection["split"]
    .value_counts()
)

print("\nPor coluna:")
display(
    selection.groupby(
        ["column", "split"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(columns)
)

print("\nArquivo:")
print(selection_path)

In [ ]:
import pandas as pd
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import shutil
import zipfile

base = Path("/content/Qumran_Analytics/1QS")

gt_root = base / "htr_ground_truth"
images_dir = gt_root / "images"
sheets_dir = gt_root / "review_sheets"

sheets_dir.mkdir(
    parents=True,
    exist_ok=True
)

selection = pd.read_csv(
    gt_root /
    "1QS_GT_SELECTION.csv"
)

columns = [
    "I", "II", "III", "IV", "V", "VI",
    "VII", "VIII", "IX", "X", "XI"
]

# ============================================================
# 1. ATRIBUI BATCH
#
# 1–5 = TRAIN
# 6   = VALIDATION
# 7   = TEST
# ============================================================

selection["batch"] = pd.NA

for col in columns:

    # TRAIN: ordenamos por posição física só para
    # termos ordem reproduzível nas 5 pranchas
    train_idx = (
        selection[
            (selection["column"].astype(str) == col)
            &
            (selection["split"] == "TRAIN")
        ]
        .sort_values("y_center")
        .index
        .tolist()
    )

    if len(train_idx) != 5:
        raise ValueError(
            f"Coluna {col}: esperado 5 TRAIN, "
            f"encontrado {len(train_idx)}"
        )

    for batch_num, idx in enumerate(
        train_idx,
        start=1
    ):
        selection.loc[idx, "batch"] = batch_num

    # VALIDATION = batch 6
    val_idx = selection[
        (selection["column"].astype(str) == col)
        &
        (selection["split"] == "VALIDATION")
    ].index.tolist()

    if len(val_idx) != 1:
        raise ValueError(
            f"Coluna {col}: VALIDATION != 1"
        )

    selection.loc[
        val_idx[0],
        "batch"
    ] = 6

    # TEST = batch 7
    test_idx = selection[
        (selection["column"].astype(str) == col)
        &
        (selection["split"] == "TEST")
    ].index.tolist()

    if len(test_idx) != 1:
        raise ValueError(
            f"Coluna {col}: TEST != 1"
        )

    selection.loc[
        test_idx[0],
        "batch"
    ] = 7


selection["batch"] = (
    selection["batch"]
    .astype(int)
)

# salva associação definitiva
selection.to_csv(
    gt_root /
    "1QS_GT_SELECTION_BATCHED.csv",
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 2. FUNÇÃO PARA GERAR PAINEL DE UMA LINHA
# ============================================================

TARGET_WIDTH = 1500
HEADER_HEIGHT = 80
MARGIN = 20

def make_panel(row):

    line_id = str(
        row["htr_line_id"]
    )

    path = (
        images_dir /
        f"{line_id}.png"
    )

    img = Image.open(
        path
    ).convert("RGB")

    # aumenta ou diminui para largura padronizada
    scale = (
        TARGET_WIDTH /
        img.width
    )

    new_h = max(
        80,
        round(
            img.height * scale
        )
    )

    resized = img.resize(
        (
            TARGET_WIDTH,
            new_h
        ),
        Image.Resampling.LANCZOS
    )

    panel = Image.new(
        "RGB",
        (
            TARGET_WIDTH + 2 * MARGIN,
            HEADER_HEIGHT +
            new_h +
            2 * MARGIN
        ),
        "white"
    )

    panel.paste(
        resized,
        (
            MARGIN,
            HEADER_HEIGHT + MARGIN
        )
    )

    draw = ImageDraw.Draw(
        panel
    )

    header = (
        f"{line_id}   |   "
        f"Coluna {row['column']}   |   "
        f"{row['split']}"
    )

    draw.text(
        (
            MARGIN,
            20
        ),
        header,
        fill="black"
    )

    return panel


# ============================================================
# 3. GERA AS 7 PRANCHAS
# ============================================================

created = []

for batch in range(1, 8):

    subset = (
        selection[
            selection["batch"] == batch
        ]
        .copy()
    )

    # mantém ordem I -> XI
    subset["col_rank"] = (
        subset["column"]
        .astype(str)
        .map(
            {
                c: i
                for i, c in enumerate(columns)
            }
        )
    )

    subset = subset.sort_values(
        "col_rank"
    )

    if len(subset) != 11:
        raise ValueError(
            f"Batch {batch}: esperado 11 linhas, "
            f"encontrado {len(subset)}"
        )

    panels = [
        make_panel(row)
        for _, row in subset.iterrows()
    ]

    width = max(
        p.width
        for p in panels
    )

    total_height = sum(
        p.height
        for p in panels
    )

    sheet = Image.new(
        "RGB",
        (
            width,
            total_height
        ),
        "white"
    )

    y = 0

    for panel in panels:

        sheet.paste(
            panel,
            (0, y)
        )

        y += panel.height

    split_name = (
        "TRAIN"
        if batch <= 5
        else
        "VALIDATION"
        if batch == 6
        else
        "TEST"
    )

    output = (
        sheets_dir /
        f"1QS_GT_BATCH_{batch:02d}_{split_name}.jpg"
    )

    sheet.save(
        output,
        "JPEG",
        quality=95,
        subsampling=0
    )

    created.append(
        output
    )

    print(
        f"Batch {batch}: "
        f"{split_name} — {output}"
    )


# ============================================================
# 4. CRIA ZIP COM AS 7 PRANCHAS
# ============================================================

zip_path = (
    gt_root /
    "1QS_GT_REVIEW_SHEETS.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for path in created:
        z.write(
            path,
            arcname=path.name
        )

print("\n====================================")
print("7 PRANCHAS CRIADAS")
print("====================================")

print("ZIP:")
print(zip_path)

In [ ]:
from google.colab import files

files.download(
    "/content/Qumran_Analytics/1QS/"
    "htr_ground_truth/"
    "1QS_GT_REVIEW_SHEETS.zip"
)

In [ ]:
import json
import pandas as pd
from pathlib import Path
from PIL import Image, ImageDraw, ImageFilter

# ============================================================
# CONFIGURAÇÃO
# ============================================================

base = Path("/content/Qumran_Analytics/1QS")

htr_manifest_path = (
    base /
    "frozen_segmentation" /
    "1QS_FINAL_HTR_MANIFEST.csv"
)

roi_dir = (
    base /
    "automatic_segmentation" /
    "roi"
)

json_dir = (
    base /
    "automatic_segmentation" /
    "json"
)

htr_v2_root = (
    base /
    "htr_input_v2"
)

crops_dir = (
    htr_v2_root /
    "masked_line_crops"
)

crops_dir.mkdir(
    parents=True,
    exist_ok=True
)

htr = pd.read_csv(
    htr_manifest_path
)

# ============================================================
# FUNÇÕES
# ============================================================

def parse_ids(value):

    return [
        int(x)
        for x in str(value).split(",")
        if str(x).strip()
    ]


def clean_polygon(points):

    clean = []

    for p in points:

        if (
            isinstance(p, (list, tuple))
            and
            len(p) >= 2
        ):

            clean.append(
                (
                    int(round(float(p[0]))),
                    int(round(float(p[1])))
                )
            )

    return clean


# cache para não abrir cada coluna centenas de vezes
cache = {}

records = []

fallback_total = 0

# ============================================================
# PROCESSA AS 284 LINHAS
# ============================================================

for _, row in htr.iterrows():

    line_id = str(
        row["htr_line_id"]
    )

    col = str(
        row["column"]
    )

    # --------------------------------------------------------
    # carrega imagem + JSON da coluna
    # --------------------------------------------------------

    if col not in cache:

        roi_path = (
            roi_dir /
            f"1QS_col_{col}_ROI.png"
        )

        json_path = (
            json_dir /
            f"1QS_col_{col}_kraken.json"
        )

        img = Image.open(
            roi_path
        ).convert("RGB")

        with open(
            json_path,
            "r",
            encoding="utf-8"
        ) as f:

            seg = json.load(f)

        cache[col] = (
            img,
            seg
        )

    img, seg = cache[col]

    lines = seg["lines"]

    source_ids = parse_ids(
        row["source_kraken_ids"]
    )

    # --------------------------------------------------------
    # máscara vazia no tamanho da coluna
    # --------------------------------------------------------

    mask = Image.new(
        "L",
        img.size,
        0
    )

    draw = ImageDraw.Draw(
        mask
    )

    n_boundaries = 0
    n_fallbacks = 0

    # --------------------------------------------------------
    # une todos os polígonos que formam esta linha física
    # --------------------------------------------------------

    for kid in source_ids:

        kline = lines[
            kid - 1
        ]

        boundary = clean_polygon(
            kline.get(
                "boundary",
                []
            )
        )

        if len(boundary) >= 3:

            draw.polygon(
                boundary,
                fill=255
            )

            n_boundaries += 1

        else:

            # -----------------------------------------------
            # fallback somente se Kraken não tiver boundary
            # -----------------------------------------------

            baseline = clean_polygon(
                kline.get(
                    "baseline",
                    []
                )
            )

            if len(baseline) >= 2:

                xs = [
                    p[0]
                    for p in baseline
                ]

                ys = [
                    p[1]
                    for p in baseline
                ]

                PAD_X = 10
                PAD_Y = 35

                x1 = max(
                    0,
                    min(xs) - PAD_X
                )

                x2 = min(
                    img.width,
                    max(xs) + PAD_X
                )

                y1 = max(
                    0,
                    min(ys) - PAD_Y
                )

                y2 = min(
                    img.height,
                    max(ys) + PAD_Y
                )

                draw.rectangle(
                    (
                        x1,
                        y1,
                        x2,
                        y2
                    ),
                    fill=255
                )

                n_fallbacks += 1
                fallback_total += 1

    # --------------------------------------------------------
    # pequena dilatação:
    # evita cortar extremidades de tinta no limite do polígono
    # --------------------------------------------------------

    mask = mask.filter(
        ImageFilter.MaxFilter(9)
    )

    bbox = mask.getbbox()

    if bbox is None:

        raise RuntimeError(
            f"Máscara vazia para {line_id}"
        )

    bx1, by1, bx2, by2 = bbox

    # margem branca externa
    PAD = 10

    bx1 = max(
        0,
        bx1 - PAD
    )

    by1 = max(
        0,
        by1 - PAD
    )

    bx2 = min(
        img.width,
        bx2 + PAD
    )

    by2 = min(
        img.height,
        by2 + PAD
    )

    raw_crop = img.crop(
        (
            bx1,
            by1,
            bx2,
            by2
        )
    )

    mask_crop = mask.crop(
        (
            bx1,
            by1,
            bx2,
            by2
        )
    )

    # fundo branco fora da região da linha
    background = Image.new(
        "RGB",
        raw_crop.size,
        "white"
    )

    isolated = Image.composite(
        raw_crop,
        background,
        mask_crop
    )

    output = (
        crops_dir /
        f"{line_id}.png"
    )

    isolated.save(
        output
    )

    records.append({
        "htr_line_id":
            line_id,

        "column":
            col,

        "source_kraken_ids":
            row["source_kraken_ids"],

        "n_boundaries":
            n_boundaries,

        "n_fallbacks":
            n_fallbacks,

        "crop_width_v2":
            isolated.width,

        "crop_height_v2":
            isolated.height,

        "htr_crop_v2_path":
            str(output)
    })


# ============================================================
# MANIFESTO DERIVADO
# ============================================================

v2 = pd.DataFrame(
    records
)

v2_path = (
    htr_v2_root /
    "1QS_HTR_INPUT_MANIFEST_V2.csv"
)

v2.to_csv(
    v2_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 60)
print("HTR INPUT V2")
print("=" * 60)

print(
    "Linhas produzidas:",
    len(v2)
)

print(
    "Fallbacks de boundary:",
    fallback_total
)

print(
    "Crops ausentes:",
    284 - len(v2)
)

print("\nManifesto:")
print(v2_path)

In [ ]:
import pandas as pd
from pathlib import Path
from PIL import Image, ImageDraw

base = Path("/content/Qumran_Analytics/1QS")

gt_root = (
    base /
    "htr_ground_truth"
)

selection = pd.read_csv(
    gt_root /
    "1QS_GT_SELECTION_BATCHED.csv"
)

v2 = pd.read_csv(
    base /
    "htr_input_v2" /
    "1QS_HTR_INPUT_MANIFEST_V2.csv"
)

selection = selection.merge(
    v2[
        [
            "htr_line_id",
            "htr_crop_v2_path"
        ]
    ],
    on="htr_line_id",
    how="left"
)

batch = (
    selection[
        selection["batch"] == 1
    ]
    .copy()
)

columns = [
    "I", "II", "III", "IV", "V", "VI",
    "VII", "VIII", "IX", "X", "XI"
]

rank = {
    c: i
    for i, c in enumerate(columns)
}

batch["col_rank"] = (
    batch["column"]
    .astype(str)
    .map(rank)
)

batch = batch.sort_values(
    "col_rank"
)

assert len(batch) == 11
assert batch["htr_crop_v2_path"].notna().all()

# ============================================================
# CRIA PAINÉIS
# ============================================================

TARGET_WIDTH = 1800
HEADER = 70
MARGIN = 25

panels = []

for _, row in batch.iterrows():

    path = Path(
        row["htr_crop_v2_path"]
    )

    img = Image.open(
        path
    ).convert("RGB")

    scale = (
        TARGET_WIDTH /
        img.width
    )

    height = max(
        100,
        round(
            img.height * scale
        )
    )

    img = img.resize(
        (
            TARGET_WIDTH,
            height
        ),
        Image.Resampling.LANCZOS
    )

    panel = Image.new(
        "RGB",
        (
            TARGET_WIDTH + 2 * MARGIN,
            height + HEADER + 2 * MARGIN
        ),
        "white"
    )

    panel.paste(
        img,
        (
            MARGIN,
            HEADER + MARGIN
        )
    )

    draw = ImageDraw.Draw(
        panel
    )

    draw.text(
        (
            MARGIN,
            20
        ),
        (
            f"{row['htr_line_id']} | "
            f"Coluna {row['column']} | "
            f"{row['split']} | MASKED V2"
        ),
        fill="black"
    )

    panels.append(
        panel
    )

width = max(
    p.width
    for p in panels
)

height = sum(
    p.height
    for p in panels
)

sheet = Image.new(
    "RGB",
    (
        width,
        height
    ),
    "white"
)

y = 0

for panel in panels:

    sheet.paste(
        panel,
        (0, y)
    )

    y += panel.height

out_dir = (
    gt_root /
    "review_sheets_v2"
)

out_dir.mkdir(
    parents=True,
    exist_ok=True
)

output = (
    out_dir /
    "1QS_GT_BATCH_01_TRAIN_MASKED_V2.png"
)

sheet.save(
    output
)

print(output)

In [ ]:
from google.colab import files

files.download(
    "/content/Qumran_Analytics/1QS/"
    "htr_ground_truth/review_sheets_v2/"
    "1QS_GT_BATCH_01_TRAIN_MASKED_V2.png"
)

In [ ]:
from google.colab import files

files.download(
    "/content/Qumran_Analytics/1QS/"
    "htr_ground_truth/review_sheets_v2/"
    "1QS_GT_BATCH_01_TRAIN_MASKED_V2.png"
)

In [ ]:
from pathlib import Path
import pandas as pd
import zipfile
from google.colab import files

# ============================================================
# CAMINHOS
# ============================================================

base = Path("/content/Qumran_Analytics/1QS")

gt_root = (
    base /
    "htr_ground_truth"
)

selection_path = (
    gt_root /
    "1QS_GT_SELECTION_BATCHED.csv"
)

v2_manifest_path = (
    base /
    "htr_input_v2" /
    "1QS_HTR_INPUT_MANIFEST_V2.csv"
)

zip_path = (
    gt_root /
    "1QS_BATCH01_NATIVE_MASKED.zip"
)

# ============================================================
# VERIFICA ARQUIVOS NECESSÁRIOS
# ============================================================

if not selection_path.exists():
    raise FileNotFoundError(
        f"Não encontrei:\n{selection_path}"
    )

if not v2_manifest_path.exists():
    raise FileNotFoundError(
        f"Não encontrei:\n{v2_manifest_path}"
    )

selection = pd.read_csv(selection_path)
v2 = pd.read_csv(v2_manifest_path)

# ============================================================
# SELECIONA APENAS O BATCH 01
# ============================================================

batch = (
    selection[
        selection["batch"] == 1
    ]
    .merge(
        v2[
            [
                "htr_line_id",
                "htr_crop_v2_path"
            ]
        ],
        on="htr_line_id",
        how="left"
    )
)

print("Linhas do Batch 01:", len(batch))

if len(batch) != 11:
    raise ValueError(
        f"Esperávamos 11 linhas, mas encontrei {len(batch)}."
    )

# ============================================================
# CHECA SE OS 11 PNGs EXISTEM
# ============================================================

missing = []

for _, row in batch.iterrows():

    path = Path(
        row["htr_crop_v2_path"]
    )

    if not path.exists():
        missing.append(
            str(path)
        )

if missing:

    print("\nArquivos ausentes:")

    for m in missing:
        print(m)

    raise FileNotFoundError(
        f"{len(missing)} crop(s) não foram encontrados."
    )

# ============================================================
# CRIA O ZIP
# ============================================================

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for _, row in batch.iterrows():

        source = Path(
            row["htr_crop_v2_path"]
        )

        filename = (
            f"{row['htr_line_id']}.png"
        )

        z.write(
            source,
            arcname=filename
        )

print("\nZIP criado com sucesso:")
print(zip_path)

print(
    "Tamanho:",
    round(
        zip_path.stat().st_size / 1024 / 1024,
        2
    ),
    "MB"
)

# ============================================================
# MOSTRA O CONTEÚDO
# ============================================================

with zipfile.ZipFile(
    zip_path,
    "r"
) as z:

    print("\nArquivos dentro do ZIP:")

    for name in z.namelist():
        print("✓", name)

# ============================================================
# DOWNLOAD
# ============================================================

files.download(
    str(zip_path)
)

**[Public-release redaction]** Original research-log cell 130 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

base = Path("/content/Qumran_Analytics/1QS")

htr = pd.read_csv(
    base /
    "frozen_segmentation" /
    "1QS_FINAL_HTR_MANIFEST.csv"
)

v2 = pd.read_csv(
    base /
    "htr_input_v2" /
    "1QS_HTR_INPUT_MANIFEST_V2.csv"
)

selection = pd.read_csv(
    base /
    "htr_ground_truth" /
    "1QS_GT_SELECTION_BATCHED.csv"
)

# junta dados geométricos + crop V2
pool = htr.merge(
    v2,
    on=[
        "htr_line_id",
        "column",
        "source_kraken_ids"
    ],
    how="left",
    suffixes=("", "_v2")
)

# ============================================================
# EXCLUI TODOS OS 77 JÁ ESCOLHIDOS
# ============================================================

already_selected = set(
    selection["htr_line_id"]
    .astype(str)
)

pool = pool[
    ~pool["htr_line_id"]
    .astype(str)
    .isin(already_selected)
].copy()

target_columns = [
    "I",
    "VII",
    "IX",
    "X",
    "XI"
]

candidate_rows = []

for col in target_columns:

    sub = pool[
        pool["column"].astype(str)
        == col
    ].copy()

    if len(sub) == 0:
        raise ValueError(
            f"Sem candidatas na coluna {col}"
        )

    # --------------------------------------------------------
    # estatísticas típicas da coluna
    # --------------------------------------------------------

    med_width = sub[
        "crop_width_v2"
    ].median()

    med_height = sub[
        "crop_height_v2"
    ].median()

    sub["width_score"] = (
        sub["crop_width_v2"] /
        med_width
    )

    sub["height_deviation"] = (
        abs(
            sub["crop_height_v2"]
            - med_height
        )
        /
        med_height
    )

    # preferência:
    # linha larga + altura típica + boundary real
    sub["quality_score"] = (
        sub["width_score"]
        -
        0.60 * sub["height_deviation"]
        -
        0.40 * (
            sub["n_fallbacks"] > 0
        ).astype(int)
        -
        0.20 * (
            sub["n_source_baselines"] > 1
        ).astype(int)
    )

    best = (
        sub
        .sort_values(
            "quality_score",
            ascending=False
        )
        .head(3)
        .copy()
    )

    best["replacement_for"] = {
        "I": "1QS_I_M002",
        "VII": "1QS_VII_M001",
        "IX": "1QS_IX_M001",
        "X": "1QS_X_M001",
        "XI": "1QS_XI_M004"
    }[col]

    candidate_rows.append(best)

candidates = pd.concat(
    candidate_rows,
    ignore_index=True
)

candidate_path = (
    base /
    "htr_ground_truth_v2" /
    "1QS_GT_REPLACEMENT_CANDIDATES.csv"
)

candidates.to_csv(
    candidate_path,
    index=False,
    encoding="utf-8-sig"
)

display(
    candidates[
        [
            "replacement_for",
            "htr_line_id",
            "column",
            "crop_width_v2",
            "crop_height_v2",
            "n_source_baselines",
            "n_fallbacks",
            "quality_score"
        ]
    ]
)

print(candidate_path)

In [ ]:
from PIL import Image, ImageDraw
from pathlib import Path

base = Path("/content/Qumran_Analytics/1QS")

out_dir = (
    base /
    "htr_ground_truth_v2" /
    "replacement_review"
)

out_dir.mkdir(
    parents=True,
    exist_ok=True
)

TARGET_WIDTH = 1500
HEADER = 70
MARGIN = 20

panels = []

for _, row in candidates.iterrows():

    path = Path(
        row["htr_crop_v2_path"]
    )

    img = Image.open(
        path
    ).convert("RGB")

    scale = TARGET_WIDTH / img.width

    height = max(
        100,
        round(img.height * scale)
    )

    img = img.resize(
        (
            TARGET_WIDTH,
            height
        ),
        Image.Resampling.LANCZOS
    )

    panel = Image.new(
        "RGB",
        (
            TARGET_WIDTH + 2 * MARGIN,
            height + HEADER + 2 * MARGIN
        ),
        "white"
    )

    panel.paste(
        img,
        (
            MARGIN,
            HEADER + MARGIN
        )
    )

    draw = ImageDraw.Draw(panel)

    draw.text(
        (
            MARGIN,
            15
        ),
        (
            f"SUBSTITUI {row['replacement_for']}   |   "
            f"CANDIDATA {row['htr_line_id']}   |   "
            f"COLUNA {row['column']}"
        ),
        fill="black"
    )

    panels.append(panel)

width = max(
    p.width
    for p in panels
)

height = sum(
    p.height
    for p in panels
)

sheet = Image.new(
    "RGB",
    (
        width,
        height
    ),
    "white"
)

y = 0

for panel in panels:

    sheet.paste(
        panel,
        (0, y)
    )

    y += panel.height

output = (
    out_dir /
    "1QS_GT_REPLACEMENT_CANDIDATES.png"
)

sheet.save(output)

print(output)

In [ ]:
from google.colab import files

files.download(
    "/content/Qumran_Analytics/1QS/"
    "htr_ground_truth_v2/"
    "replacement_review/"
    "1QS_GT_REPLACEMENT_CANDIDATES.png"
)

In [ ]:
from pathlib import Path
import pandas as pd
import shutil
from datetime import datetime, timezone

base = Path("/content/Qumran_Analytics/1QS")

gt_v2 = base / "htr_ground_truth_v2"
images_v2 = gt_v2 / "images"

working_path = (
    gt_v2 /
    "1QS_GT_SELECTION_V2_WORKING.csv"
)

candidates_path = (
    gt_v2 /
    "1QS_GT_REPLACEMENT_CANDIDATES.csv"
)

v2_manifest_path = (
    base /
    "htr_input_v2" /
    "1QS_HTR_INPUT_MANIFEST_V2.csv"
)

htr_manifest_path = (
    base /
    "frozen_segmentation" /
    "1QS_FINAL_HTR_MANIFEST.csv"
)

working = pd.read_csv(working_path)
candidates = pd.read_csv(candidates_path)
v2 = pd.read_csv(v2_manifest_path)
htr = pd.read_csv(htr_manifest_path)

# ============================================================
# DECISÕES VISUAIS
# ============================================================

replacements = {
    "1QS_I_M002":   "1QS_I_M017",
    "1QS_VII_M001": "1QS_VII_M017",
    "1QS_IX_M001":  "1QS_IX_M010",
    "1QS_X_M001":   "1QS_X_M017",
    "1QS_XI_M004":  "1QS_XI_M018",
}

timestamp = datetime.now(
    timezone.utc
).isoformat()

# ============================================================
# GARANTE COLUNAS DE AUDITORIA
# ============================================================

for col in [
    "active_for_gt",
    "replacement_for",
    "replaced_by",
    "replacement_timestamp_utc"
]:
    if col not in working.columns:
        working[col] = pd.NA

# original é ativo, salvo indicação contrária
working["active_for_gt"] = (
    working["active_for_gt"]
    .fillna(True)
)

# ============================================================
# MARCA AS CINCO ANTIGAS COMO SUBSTITUÍDAS
# ============================================================

for old_id, new_id in replacements.items():

    mask = (
        working["htr_line_id"]
        .astype(str)
        == old_id
    )

    if mask.sum() != 1:
        raise ValueError(
            f"{old_id}: esperado exatamente 1 registro; "
            f"encontrado {mask.sum()}"
        )

    # proteção: só estamos alterando TRAIN
    split = str(
        working.loc[
            mask,
            "split"
        ].iloc[0]
    )

    if split != "TRAIN":
        raise RuntimeError(
            f"Tentativa de substituir {old_id}, "
            f"mas split={split}"
        )

    working.loc[
        mask,
        "active_for_gt"
    ] = False

    working.loc[
        mask,
        "gt_status_v2"
    ] = "REPLACED_FOR_GT_QUALITY"

    working.loc[
        mask,
        "replaced_by"
    ] = new_id

    working.loc[
        mask,
        "replacement_timestamp_utc"
    ] = timestamp

# ============================================================
# ADICIONA AS CINCO NOVAS
# ============================================================

new_rows = []

for old_id, new_id in replacements.items():

    # dados do corpus congelado
    h = htr[
        htr["htr_line_id"]
        .astype(str)
        == new_id
    ]

    if len(h) != 1:
        raise ValueError(
            f"{new_id} não encontrado unicamente "
            "no manifesto HTR."
        )

    row = h.iloc[0].to_dict()

    # crop MASKED V2
    v = v2[
        v2["htr_line_id"]
        .astype(str)
        == new_id
    ]

    if len(v) != 1:
        raise ValueError(
            f"{new_id} não encontrado unicamente "
            "no manifesto V2."
        )

    vrow = v.iloc[0]

    row["htr_crop_v2_path"] = (
        vrow["htr_crop_v2_path"]
    )

    row["split"] = "TRAIN"
    row["batch"] = 1

    row["transcription_status"] = "PENDING"
    row["reference_visible_during_transcription"] = False

    row["gt_status_v2"] = "PENDING"
    row["gt_source"] = pd.NA
    row["gt_text"] = pd.NA

    row["researcher_confirmed"] = False
    row["expert_verified"] = False

    row["excluded_reason"] = pd.NA

    row["active_for_gt"] = True
    row["replacement_for"] = old_id
    row["replaced_by"] = pd.NA
    row["replacement_timestamp_utc"] = timestamp
    row["last_update_utc"] = timestamp

    new_rows.append(row)

new_df = pd.DataFrame(new_rows)

# concatena preservando histórico
working_v2 = pd.concat(
    [
        working,
        new_df
    ],
    ignore_index=True,
    sort=False
)

# ============================================================
# COPIA OS 5 NOVOS CROPS E CRIA GT VAZIO
# ============================================================

for old_id, new_id in replacements.items():

    vrow = v2[
        v2["htr_line_id"]
        .astype(str)
        == new_id
    ].iloc[0]

    src = Path(
        vrow["htr_crop_v2_path"]
    )

    if not src.exists():
        raise FileNotFoundError(src)

    dst = (
        images_v2 /
        f"{new_id}.png"
    )

    shutil.copy2(
        src,
        dst
    )

    gt_file = (
        images_v2 /
        f"{new_id}.gt.txt"
    )

    # nova GT deve começar vazia
    gt_file.write_text(
        "",
        encoding="utf-8"
    )

# ============================================================
# MANIFESTO COMPLETO DE AUDITORIA
# ============================================================

audit_path = (
    gt_v2 /
    "1QS_GT_SELECTION_V2_AUDIT.csv"
)

working_v2.to_csv(
    audit_path,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# MANIFESTO ATIVO
# ============================================================

active = working_v2[
    working_v2["active_for_gt"] == True
].copy()

# sanity: nenhum ID repetido ativo
if not active["htr_line_id"].is_unique:
    raise RuntimeError(
        "Existem htr_line_id duplicados no GT ativo."
    )

active_path = (
    gt_v2 /
    "1QS_GT_SELECTION_V2_ACTIVE.csv"
)

active.to_csv(
    active_path,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# CHECAGENS
# ============================================================

print("=" * 65)
print("SUBSTITUIÇÕES GT CONCLUÍDAS")
print("=" * 65)

print("\nRegistros no audit trail:",
      len(working_v2))

print("Linhas ATIVAS:",
      len(active))

print("\nSplits ativos:")
print(
    active["split"]
    .value_counts()
)

print("\nSubstituições:")

for old_id, new_id in replacements.items():
    print(
        f"{old_id}  ->  {new_id}"
    )

print("\nGT provisórios já preenchidos:",
      (
          active["gt_status_v2"]
          == "AI_PROVISIONAL_GT"
      ).sum())

print("GT pendentes:",
      (
          active["gt_status_v2"]
          == "PENDING"
      ).sum())

print("\nManifesto ativo:")
print(active_path)

print("\nAudit trail:")
print(audit_path)

In [ ]:
from pathlib import Path
import pandas as pd
import zipfile
from google.colab import files

base = Path("/content/Qumran_Analytics/1QS")

v2_manifest = pd.read_csv(
    base /
    "htr_input_v2" /
    "1QS_HTR_INPUT_MANIFEST_V2.csv"
)

# ============================================================
# CINCO SUBSTITUTAS DEFINITIVAS DO BATCH 01
# ============================================================

replacement_ids = [
    "1QS_I_M017",
    "1QS_VII_M017",
    "1QS_IX_M010",
    "1QS_X_M017",
    "1QS_XI_M018"
]

selected = v2_manifest[
    v2_manifest["htr_line_id"]
    .astype(str)
    .isin(replacement_ids)
].copy()

# mantém ordem I, VII, IX, X, XI
order = {
    line_id: i
    for i, line_id in enumerate(replacement_ids)
}

selected["order"] = (
    selected["htr_line_id"]
    .map(order)
)

selected = selected.sort_values("order")

# ============================================================
# CHECAGENS
# ============================================================

if len(selected) != 5:
    raise ValueError(
        f"Esperávamos 5 linhas, mas encontrei {len(selected)}."
    )

missing = []

for _, row in selected.iterrows():

    path = Path(
        row["htr_crop_v2_path"]
    )

    if not path.exists():
        missing.append(str(path))

if missing:

    print("Arquivos ausentes:")

    for path in missing:
        print(path)

    raise FileNotFoundError(
        f"{len(missing)} arquivo(s) ausente(s)."
    )

# ============================================================
# CRIA ZIP NATIVO
# ============================================================

zip_path = (
    base /
    "htr_ground_truth_v2" /
    "1QS_BATCH01_REPLACEMENTS_NATIVE.zip"
)

zip_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for _, row in selected.iterrows():

        source = Path(
            row["htr_crop_v2_path"]
        )

        z.write(
            source,
            arcname=f"{row['htr_line_id']}.png"
        )

print("=" * 60)
print("BATCH 01 — CINCO SUBSTITUTAS")
print("=" * 60)

with zipfile.ZipFile(zip_path, "r") as z:

    for name in z.namelist():
        print("✓", name)

print("\nZIP:")
print(zip_path)

print(
    "Tamanho:",
    round(
        zip_path.stat().st_size / 1024 / 1024,
        2
    ),
    "MB"
)

files.download(
    str(zip_path)
)

**[Public-release redaction]** Original research-log cell 136 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


In [ ]:
from pathlib import Path
import pandas as pd
import zipfile
from google.colab import files

base = Path("/content/Qumran_Analytics/1QS")

gt_v2 = base / "htr_ground_truth_v2"

active_path = (
    gt_v2 /
    "1QS_GT_SELECTION_V2_ACTIVE.csv"
)

v2_manifest_path = (
    base /
    "htr_input_v2" /
    "1QS_HTR_INPUT_MANIFEST_V2.csv"
)

active = pd.read_csv(active_path)
v2 = pd.read_csv(v2_manifest_path)

# ============================================================
# BATCH 02 — TRAIN
# ============================================================

batch = active[
    (active["split"] == "TRAIN")
    &
    (active["batch"] == 2)
].copy()

print("Linhas ativas encontradas:", len(batch))

if len(batch) != 11:
    raise RuntimeError(
        f"Batch 02 deveria conter 11 linhas; "
        f"encontrei {len(batch)}."
    )

# ============================================================
# JUNTA CAMINHOS MASKED V2
# caso a coluna já exista no active, evita duplicação
# ============================================================

if "htr_crop_v2_path" not in batch.columns:

    batch = batch.merge(
        v2[
            [
                "htr_line_id",
                "htr_crop_v2_path"
            ]
        ],
        on="htr_line_id",
        how="left"
    )

else:

    missing_mask = (
        batch["htr_crop_v2_path"]
        .isna()
    )

    if missing_mask.any():

        lookup = (
            v2.set_index("htr_line_id")
            ["htr_crop_v2_path"]
        )

        batch.loc[
            missing_mask,
            "htr_crop_v2_path"
        ] = (
            batch.loc[
                missing_mask,
                "htr_line_id"
            ]
            .map(lookup)
        )

# ============================================================
# ORDENA I -> XI
# ============================================================

columns_order = [
    "I", "II", "III", "IV", "V", "VI",
    "VII", "VIII", "IX", "X", "XI"
]

rank = {
    col: i
    for i, col in enumerate(columns_order)
}

batch["column_rank"] = (
    batch["column"]
    .astype(str)
    .map(rank)
)

batch = (
    batch
    .sort_values("column_rank")
    .reset_index(drop=True)
)

# ============================================================
# CHECAGEM DOS 11 CROPS
# ============================================================

missing = []

for _, row in batch.iterrows():

    path = Path(
        row["htr_crop_v2_path"]
    )

    if not path.exists():
        missing.append(
            str(path)
        )

if missing:

    print("\nArquivos não encontrados:")

    for path in missing:
        print(path)

    raise FileNotFoundError(
        f"{len(missing)} crop(s) ausente(s)."
    )

# ============================================================
# MOSTRA QUAIS LINHAS FORMAM O BATCH
# ============================================================

print("\nBATCH 02:")
display(
    batch[
        [
            "htr_line_id",
            "column",
            "split"
        ]
    ]
)

# ============================================================
# CRIA ZIP NATIVO
# ============================================================

zip_path = (
    gt_v2 /
    "1QS_BATCH02_NATIVE_MASKED.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for _, row in batch.iterrows():

        source = Path(
            row["htr_crop_v2_path"]
        )

        z.write(
            source,
            arcname=(
                f"{row['htr_line_id']}.png"
            )
        )

# ============================================================
# CONFERE O ZIP
# ============================================================

print("\n" + "=" * 60)
print("1QS — BATCH 02 PREPARADO")
print("=" * 60)

with zipfile.ZipFile(
    zip_path,
    "r"
) as z:

    names = z.namelist()

    print("Arquivos no ZIP:", len(names))

    for name in names:
        print("✓", name)

print("\nZIP:")
print(zip_path)

print(
    "Tamanho:",
    round(
        zip_path.stat().st_size /
        1024 /
        1024,
        2
    ),
    "MB"
)

# ============================================================
# DOWNLOAD
# ============================================================

files.download(
    str(zip_path)
)

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

possible_roots = [
    Path("/content/drive/MyDrive/Qumran_Analytics"),
    Path("/content/drive/MyDrive/1QS"),
    Path("/content/drive/MyDrive"),
]

print("=" * 65)
print("PROCURANDO CHECKPOINTS NO GOOGLE DRIVE")
print("=" * 65)

patterns = [
    "1QS_FINAL_HTR_MANIFEST.csv",
    "1QS_FROZEN_SEGMENTATION_MANIFEST.csv",
    "1QS_GT_SELECTION_V2_ACTIVE.csv",
    "1QS_BATCH01_CLOSED_MANIFEST.csv",
    "1QS_HTR_INPUT_MANIFEST_V2.csv",
    "1QS_col_I.png",
    "1QS_zoom7_full.png",
]

found = []

for root in possible_roots:
    if not root.exists():
        continue

    for pattern in patterns:
        for p in root.rglob(pattern):
            if p not in found:
                found.append(p)

if found:
    print("\n✅ ENCONTRADO:")
    for p in found:
        print(p)
else:
    print("\n❌ Nenhum checkpoint encontrado no Google Drive.")

In [ ]:
# ============================================================
# PREPARAÇÃO DO AMBIENTE DE RECUPERAÇÃO
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import sys
import subprocess
import shutil
from importlib.metadata import version, PackageNotFoundError

def pkg_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

kraken_v = pkg_version("kraken")
safe_v = pkg_version("safetensors")

print("Kraken atual:", kraken_v)
print("safetensors atual:", safe_v)

if kraken_v != "7.1.1" or safe_v != "0.8.0":

    print("\nInstalando versões usadas no projeto...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "kraken==7.1.1",
        "safetensors==0.8.0"
    ])

print("\nKraken:", version("kraken"))
print("safetensors:", version("safetensors"))
print("Executável Kraken:", shutil.which("kraken"))

In [ ]:
# ============================================================
# INSTALAÇÃO ISOLADA DO KRAKEN NO COLAB
# ============================================================

import os
import sys
import shutil
import subprocess
from pathlib import Path

print("Python:", sys.version)
print("Executável Python:", sys.executable)

# ------------------------------------------------------------
# 1. INSTALA PIPX
# ------------------------------------------------------------

print("\n[1/4] Instalando pipx...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "pipx"
    ],
    check=True
)

# ------------------------------------------------------------
# 2. INSTALA KRAKEN 7.1.1 EM AMBIENTE ISOLADO
# ------------------------------------------------------------

print("\n[2/4] Instalando Kraken 7.1.1...")

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pipx",
        "install",
        "--force",
        "--python",
        sys.executable,
        "kraken==7.1.1"
    ],
    text=True,
    capture_output=True
)

print(result.stdout)

if result.returncode != 0:

    print("\nERRO NA INSTALAÇÃO DO KRAKEN:")
    print(result.stderr[-5000:])

    raise RuntimeError(
        "A instalação isolada do Kraken falhou."
    )

# ------------------------------------------------------------
# 3. GARANTE SAFETENSORS 0.8.0 DENTRO DO AMBIENTE KRAKEN
# ------------------------------------------------------------

print("\n[3/4] Fixando safetensors 0.8.0...")

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pipx",
        "inject",
        "--force",
        "kraken",
        "safetensors==0.8.0"
    ],
    text=True,
    capture_output=True
)

print(result.stdout)

if result.returncode != 0:

    print(result.stderr[-5000:])

    raise RuntimeError(
        "Falha ao instalar safetensors no ambiente Kraken."
    )

# ------------------------------------------------------------
# 4. COLOCA O EXECUTÁVEL NO PATH DA SESSÃO
# ------------------------------------------------------------

pipx_bin = Path.home() / ".local" / "bin"

os.environ["PATH"] = (
    str(pipx_bin)
    + os.pathsep
    + os.environ.get("PATH", "")
)

kraken_exe = shutil.which("kraken")

print("\n[4/4] Verificação")
print("Kraken encontrado:", kraken_exe)

if kraken_exe is None:

    raise RuntimeError(
        "Kraken foi instalado, mas o executável "
        "não foi localizado."
    )

# testa o CLI sem fazer processamento
test = subprocess.run(
    [
        kraken_exe,
        "--help"
    ],
    capture_output=True,
    text=True
)

if test.returncode != 0:

    print(test.stderr[-3000:])

    raise RuntimeError(
        "O executável Kraken não iniciou corretamente."
    )

print("\n" + "=" * 60)
print("✅ KRAKEN PRONTO")
print("=" * 60)
print("Executável:", kraken_exe)
print("safetensors isolado: 0.8.0")

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

# ============================================================
# CONFIGURAÇÃO
# ============================================================

DRIVE_BASE = "/content/drive/MyDrive/Qumran_Analytics/1QS"

BASE = Path(DRIVE_BASE)
BASE.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# LOCALIZA O NOTEBOOK-FONTE
# ============================================================

matches = list(
    Path("/content/drive/MyDrive")
    .rglob("Qunram Analytics.ipynb")
)

if not matches:

    matches = [
        p
        for p in Path(
            "/content/drive/MyDrive"
        ).rglob("*.ipynb")
        if (
            "unram" in p.name.lower()
            and
            "analytics" in p.name.lower()
        )
    ]

if not matches:
    raise FileNotFoundError(
        "Não encontrei Qunram Analytics.ipynb no MyDrive."
    )

NB_PATH = matches[0]

print(
    "Notebook-fonte:",
    NB_PATH
)

with open(
    NB_PATH,
    "r",
    encoding="utf-8"
) as f:

    nb = json.load(f)

# ============================================================
# ADAPTA CÉLULAS ANTIGAS PARA GOOGLE DRIVE
# ============================================================

def code_of(i):

    code = "".join(
        nb["cells"][i].get(
            "source",
            []
        )
    )

    # todos os caminhos passam a ser persistentes
    code = code.replace(
        "/content/Qumran_Analytics/1QS",
        DRIVE_BASE
    )

    # primeira célula usava caminho relativo
    code = code.replace(
        'BASE_DIR = Path("Qumran_Analytics/1QS")',
        f'BASE_DIR = Path(r"{DRIVE_BASE}")'
    )

    # os 576 tiles temporários não precisam ficar no Drive
    if i == 0:

        code = code.replace(
            'TILES_DIR = BASE_DIR / "raw_tiles"',
            'TILES_DIR = Path("/content/1QS_recovery_raw_tiles")'
        )

    return code


def run_cell(i, label):

    print(
        "\n" +
        "=" * 72
    )

    print(
        f"{label}  [célula original {i}]"
    )

    print(
        "=" * 72
    )

    exec(
        compile(
            code_of(i),
            f"<recovery_cell_{i}>",
            "exec"
        ),
        globals()
    )


# ============================================================
# RESULTADOS HISTÓRICOS QUE DEVEM SER REPRODUZIDOS
# ============================================================

columns = [
    "I", "II", "III", "IV", "V", "VI",
    "VII", "VIII", "IX", "X", "XI"
]

expected_raw = {
    "I": 31,
    "II": 30,
    "III": 27,
    "IV": 30,
    "V": 30,
    "VI": 28,
    "VII": 32,
    "VIII": 31,
    "IX": 27,
    "X": 28,
    "XI": 22
}

# ============================================================
# 1 — RECONSTRUÇÃO DO MANUSCRITO
# ============================================================

full = (
    BASE /
    "reconstructed" /
    "1QS_zoom7_full.png"
)

if not full.exists():

    run_cell(
        0,
        "1/15 — Reconstrução do 1QS"
    )

else:

    print(
        "1/15 — reconstrução já existe; pulando."
    )


# ============================================================
# 2 — 11 COLUNAS
# ============================================================

columns_exist = all(
    (
        BASE /
        "columns" /
        f"1QS_col_{c}.png"
    ).exists()
    for c in columns
)

if not columns_exist:

    run_cell(
        3,
        "2/15 — Recorte das 11 colunas"
    )

else:

    print(
        "2/15 — colunas já existem; pulando."
    )


# ============================================================
# 3 — KRAKEN
# ============================================================

def kraken_ok():

    try:

        for c, n in expected_raw.items():

            p = (
                BASE /
                "automatic_segmentation" /
                "json" /
                f"1QS_col_{c}_kraken.json"
            )

            if not p.exists():
                return False

            with open(
                p,
                "r",
                encoding="utf-8"
            ) as f:

                seg = json.load(f)

            if (
                len(
                    seg.get(
                        "lines",
                        []
                    )
                )
                != n
            ):
                return False

        return True

    except Exception:
        return False


if not kraken_ok():

    run_cell(
        82,
        "3/15 — Kraken nas 11 colunas"
    )

else:

    print(
        "3/15 — JSONs Kraken válidos já existem; pulando."
    )


# trava de segurança
counts = {}

for c, n in expected_raw.items():

    p = (
        BASE /
        "automatic_segmentation" /
        "json" /
        f"1QS_col_{c}_kraken.json"
    )

    with open(
        p,
        "r",
        encoding="utf-8"
    ) as f:

        seg = json.load(f)

    counts[c] = len(
        seg.get(
            "lines",
            []
        )
    )

if counts != expected_raw:

    raise RuntimeError(
        "STOP: contagens Kraken mudaram.\n"
        f"{counts}"
    )

print(
    "✓ Kraken reproduziu as 316 baselines esperadas."
)


# ============================================================
# 4 — DIAGNÓSTICO
# ============================================================

run_cell(
    83,
    "4/15 — Diagnóstico de baselines"
)


# ============================================================
# 5 — EVENTOS DE MERGE
# ============================================================

run_cell(
    95,
    "5/15 — Geração dos merge-events"
)

merge_events = pd.read_csv(
    BASE /
    "automatic_segmentation" /
    "1QS_MERGE_EVENTS.csv"
)

if len(merge_events) != 27:

    raise RuntimeError(
        "STOP: número de merge-events "
        f"diferente de 27: {len(merge_events)}"
    )


# ============================================================
# 6 — CLASSIFICADOR V1
# ============================================================

run_cell(
    97,
    "6/15 — Classificador estrutural V1"
)


# ============================================================
# 7 — CLASSIFICADOR V2
# ============================================================

run_cell(
    104,
    "7/15 — Classificador estrutural V2"
)

v2c = pd.read_csv(
    BASE /
    "automatic_segmentation" /
    "1QS_MERGE_EVENTS_CLASSIFIED_V2.csv"
)

expected_v2 = {
    "AUTO_TRUE_MERGE": 14,
    "MANUAL_REVIEW": 5,
    "AUTO_NOISE_FRAGMENT": 4,
    "SMALL_FRAGMENT_REVIEW": 3,
    "AUTO_KEEP_SEPARATE": 1
}

got_v2 = (
    v2c[
        "automatic_decision_v2"
    ]
    .value_counts()
    .to_dict()
)

if got_v2 != expected_v2:

    raise RuntimeError(
        "STOP: classificação V2 não "
        "reproduziu o resultado histórico.\n"
        f"Obtido: {got_v2}"
    )


# ============================================================
# 8 — 5 ADJUDICAÇÕES MANUAIS
# ============================================================

run_cell(
    100,
    "8/15 — Adjudicações manuais"
)


# ============================================================
# 9 — 3 SMALL FRAGMENTS
# ============================================================

run_cell(
    107,
    "9/15 — Small-fragment reviews"
)


# ============================================================
# 10 — DECISÕES FINAIS
# ============================================================

run_cell(
    108,
    "10/15 — Montagem das decisões finais"
)

run_cell(
    109,
    "11/15 — Aplicação das ações estruturais"
)

events_final = pd.read_csv(
    BASE /
    "automatic_segmentation" /
    "1QS_MERGE_EVENTS_FINAL.csv"
)

if (
    len(events_final) != 27
    or
    events_final[
        "final_decision"
    ].isna().any()
):

    raise RuntimeError(
        "STOP: eventos estruturais finais incompletos."
    )


# ============================================================
# 11 — CONSOLIDAÇÃO
# ============================================================

run_cell(
    110,
    "12/15 — Consolidação das linhas físicas"
)

manifest = pd.read_csv(
    BASE /
    "final_segmentation" /
    "1QS_FINAL_LINE_MANIFEST.csv"
)

if len(manifest) != 291:

    raise RuntimeError(
        "STOP: esperados 291 objetos "
        f"consolidados; obtidos {len(manifest)}."
    )


# ============================================================
# 12 — FREEZE
# ============================================================

run_cell(
    118,
    "13/15 — Freeze do corpus"
)

htr = pd.read_csv(
    BASE /
    "frozen_segmentation" /
    "1QS_FINAL_HTR_MANIFEST.csv"
)

secondary = pd.read_csv(
    BASE /
    "frozen_segmentation" /
    "1QS_SECONDARY_TEXT_MANIFEST.csv"
)

if (
    len(htr) != 284
    or
    len(secondary) != 5
):

    raise RuntimeError(
        "STOP: freeze divergente. "
        f"MAIN={len(htr)}, "
        f"SECONDARY={len(secondary)}"
    )

expected_main = {
    "I": 26,
    "II": 26,
    "III": 26,
    "IV": 26,
    "V": 26,
    "VI": 27,
    "VII": 25,
    "VIII": 28,
    "IX": 26,
    "X": 26,
    "XI": 22
}

got_main = (
    htr
    .groupby("column")
    .size()
    .reindex(columns)
    .to_dict()
)

if got_main != expected_main:

    raise RuntimeError(
        "STOP: distribuição MAIN_TEXT mudou.\n"
        f"{got_main}"
    )


# ============================================================
# 13 — HASHES
# ============================================================

run_cell(
    121,
    "14/15 — Hashes do corpus congelado"
)


# ============================================================
# 14 — MASKED V2
# ============================================================

run_cell(
    125,
    "15/15 — MASKED V2"
)

masked = pd.read_csv(
    BASE /
    "htr_input_v2" /
    "1QS_HTR_INPUT_MANIFEST_V2.csv"
)

if len(masked) != 284:

    raise RuntimeError(
        "STOP: MASKED V2 não contém 284 linhas."
    )


# ============================================================
# 15 — RESTAURA OS 77 EXEMPLOS GT
# ============================================================

print(
    "\n" +
    "=" * 72
)

print(
    "RESTABELECENDO 77 GT E BATCHES"
)

print(
    "=" * 72
)

run_cell(
    122,
    "Seleção reproduzível: 55 TRAIN + 11 VALIDATION + 11 TEST"
)

selection = pd.read_csv(
    BASE /
    "htr_ground_truth" /
    "1QS_GT_SELECTION.csv"
)

selection[
    "batch"
] = pd.NA

for col in columns:

    train_idx = (
        selection[
            (
                selection[
                    "column"
                ].astype(str)
                == col
            )
            &
            (
                selection[
                    "split"
                ]
                == "TRAIN"
            )
        ]
        .sort_values(
            "y_center"
        )
        .index
        .tolist()
    )

    if len(train_idx) != 5:
        raise RuntimeError(
            f"{col}: TRAIN != 5"
        )

    for b, idx in enumerate(
        train_idx,
        start=1
    ):

        selection.loc[
            idx,
            "batch"
        ] = b

    vi = selection[
        (
            selection[
                "column"
            ].astype(str)
            == col
        )
        &
        (
            selection[
                "split"
            ]
            == "VALIDATION"
        )
    ].index.tolist()

    ti = selection[
        (
            selection[
                "column"
            ].astype(str)
            == col
        )
        &
        (
            selection[
                "split"
            ]
            == "TEST"
        )
    ].index.tolist()

    if (
        len(vi) != 1
        or
        len(ti) != 1
    ):

        raise RuntimeError(
            f"{col}: VALIDATION/TEST inválido."
        )

    selection.loc[
        vi[0],
        "batch"
    ] = 6

    selection.loc[
        ti[0],
        "batch"
    ] = 7


selection[
    "batch"
] = selection[
    "batch"
].astype(int)

selection.to_csv(
    BASE /
    "htr_ground_truth" /
    "1QS_GT_SELECTION_BATCHED.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# CONFERE O BATCH 01 ORIGINAL
# ============================================================

orig_b1 = set(
    selection[
        (
            selection[
                "split"
            ]
            == "TRAIN"
        )
        &
        (
            selection[
                "batch"
            ]
            == 1
        )
    ][
        "htr_line_id"
    ].astype(str)
)

expected_orig_b1 = {
    "1QS_I_M002",
    "1QS_II_M004",
    "1QS_III_M001",
    "1QS_IV_M005",
    "1QS_V_M002",
    "1QS_VI_M004",
    "1QS_VII_M001",
    "1QS_VIII_M004",
    "1QS_IX_M001",
    "1QS_X_M001",
    "1QS_XI_M004"
}

if orig_b1 != expected_orig_b1:

    raise RuntimeError(
        "STOP: Batch 01 original mudou.\n"
        f"{sorted(orig_b1)}"
    )


# ============================================================
# RESTAURA GT V2
# ============================================================

run_cell(
    130,
    "Restauração do GT V2 inicial"
)


# ============================================================
# CELL 134 LÊ ESTE ARQUIVO, MAS NÃO USA O CONTEÚDO
# ============================================================

candidate_path = (
    BASE /
    "htr_ground_truth_v2" /
    "1QS_GT_REPLACEMENT_CANDIDATES.csv"
)

if not candidate_path.exists():

    pd.DataFrame({
        "htr_line_id": []
    }).to_csv(
        candidate_path,
        index=False
    )


# ============================================================
# RESTAURA AS 5 SUBSTITUIÇÕES
# ============================================================

run_cell(
    134,
    "Restauração das 5 substituições do Batch 01"
)


# ============================================================
# RESTAURA O FECHAMENTO DO BATCH 01
# ============================================================

run_cell(
    136,
    "Restauração do fechamento do Batch 01"
)


# ============================================================
# SANITY CHECK FINAL
# ============================================================

active = pd.read_csv(
    BASE /
    "htr_ground_truth_v2" /
    "1QS_GT_SELECTION_V2_ACTIVE.csv"
)

batch1 = pd.read_csv(
    BASE /
    "htr_ground_truth_v2" /
    "1QS_BATCH01_CLOSED_MANIFEST.csv"
)

if len(active) != 77:

    raise RuntimeError(
        "STOP: GT ativo deveria ter 77; "
        f"tem {len(active)}."
    )

splits = (
    active[
        "split"
    ]
    .value_counts()
    .to_dict()
)

expected_splits = {
    "TRAIN": 55,
    "VALIDATION": 11,
    "TEST": 11
}

if splits != expected_splits:

    raise RuntimeError(
        f"STOP: splits divergentes: {splits}"
    )

if (
    len(batch1) != 11
    or
    batch1[
        "gt_text"
    ]
    .fillna("")
    .str.strip()
    .eq("")
    .any()
):

    raise RuntimeError(
        "STOP: Batch 01 não foi restaurado corretamente."
    )


# ============================================================
# METADATA DE RECUPERAÇÃO
# ============================================================

metadata = {
    "status":
        "RECOVERY_COMPLETE",

    "source_notebook":
        str(NB_PATH),

    "source_notebook_drive_id":
        "1L9nlaAjtn0RGbK3nN4KhH4OPV4KEVhNl",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "main_text_lines":
        284,

    "secondary_text_objects":
        5,

    "gt_active":
        77,

    "train":
        55,

    "validation":
        11,

    "test":
        11,

    "batch01_closed":
        True
}

(
    BASE /
    "RECOVERY_METADATA.json"
).write_text(
    json.dumps(
        metadata,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    "\n" +
    "=" * 72
)

print(
    "✅ RECUPERAÇÃO CONCLUÍDA"
)

print(
    "=" * 72
)

print(
    "Base persistente:",
    BASE
)

print(
    "MAIN_TEXT:",
    len(htr)
)

print(
    "SECONDARY_TEXT:",
    len(secondary)
)

print(
    "GT ativo:",
    len(active)
)

print(
    "Batch 01 fechado:",
    len(batch1),
    "linhas"
)

print(
    "\nPróximo passo: Batch 02."
)

In [ ]:
from pathlib import Path
import pandas as pd
import zipfile

base = Path("/content/drive/MyDrive/Qumran_Analytics/1QS")

active = pd.read_csv(
    base /
    "htr_ground_truth_v2" /
    "1QS_GT_SELECTION_V2_ACTIVE.csv"
)

v2 = pd.read_csv(
    base /
    "htr_input_v2" /
    "1QS_HTR_INPUT_MANIFEST_V2.csv"
)

# ============================================================
# BATCH 02
# ============================================================

batch = active[
    (active["split"] == "TRAIN")
    &
    (active["batch"] == 2)
].copy()

if len(batch) != 11:
    raise RuntimeError(
        f"Esperávamos 11 linhas no Batch 02; encontrei {len(batch)}."
    )

# garante caminho MASKED V2
if "htr_crop_v2_path" not in batch.columns:
    batch = batch.merge(
        v2[
            [
                "htr_line_id",
                "htr_crop_v2_path"
            ]
        ],
        on="htr_line_id",
        how="left"
    )

# ordem das colunas
columns_order = [
    "I", "II", "III", "IV", "V", "VI",
    "VII", "VIII", "IX", "X", "XI"
]

rank = {
    col: i
    for i, col in enumerate(columns_order)
}

batch["column_rank"] = (
    batch["column"]
    .astype(str)
    .map(rank)
)

batch = (
    batch
    .sort_values("column_rank")
    .reset_index(drop=True)
)

# ============================================================
# CONFERE ARQUIVOS
# ============================================================

missing = []

for _, row in batch.iterrows():

    path = Path(
        row["htr_crop_v2_path"]
    )

    if not path.exists():
        missing.append(str(path))

if missing:
    raise FileNotFoundError(
        "\n".join(missing)
    )

# ============================================================
# CRIA ZIP NO DRIVE
# ============================================================

zip_path = (
    base /
    "htr_ground_truth_v2" /
    "1QS_BATCH02_NATIVE_MASKED.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for _, row in batch.iterrows():

        source = Path(
            row["htr_crop_v2_path"]
        )

        z.write(
            source,
            arcname=f"{row['htr_line_id']}.png"
        )

print("=" * 60)
print("BATCH 02 PREPARADO")
print("=" * 60)

print("\nLinhas:", len(batch))

display(
    batch[
        [
            "htr_line_id",
            "column",
            "split"
        ]
    ]
)

print("\nZIP salvo permanentemente em:")
print(zip_path)

In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/Qumran_Analytics/1QS/"
    "htr_ground_truth_v2/"
    "1QS_BATCH02_NATIVE_MASKED.zip"
)

**[Public-release redaction]** Original research-log cell 144 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


In [ ]:
from pathlib import Path
import pandas as pd
import zipfile
from google.colab import files

base = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS"
)

active = pd.read_csv(
    base /
    "htr_ground_truth_v2" /
    "1QS_GT_SELECTION_V2_ACTIVE.csv"
)

v2 = pd.read_csv(
    base /
    "htr_input_v2" /
    "1QS_HTR_INPUT_MANIFEST_V2.csv"
)

# ============================================================
# BATCH 03
# ============================================================

batch = active[
    (active["split"] == "TRAIN")
    &
    (active["batch"] == 3)
].copy()

if len(batch) != 11:
    raise RuntimeError(
        f"Esperávamos 11 linhas no Batch 03; "
        f"encontrei {len(batch)}."
    )

# ============================================================
# GARANTE CAMINHO MASKED V2
# ============================================================

if "htr_crop_v2_path" not in batch.columns:

    batch = batch.merge(
        v2[
            [
                "htr_line_id",
                "htr_crop_v2_path"
            ]
        ],
        on="htr_line_id",
        how="left"
    )

else:

    missing_mask = (
        batch["htr_crop_v2_path"]
        .isna()
    )

    if missing_mask.any():

        lookup = (
            v2.set_index("htr_line_id")
            ["htr_crop_v2_path"]
        )

        batch.loc[
            missing_mask,
            "htr_crop_v2_path"
        ] = (
            batch.loc[
                missing_mask,
                "htr_line_id"
            ]
            .map(lookup)
        )

# ============================================================
# ORDEM I -> XI
# ============================================================

columns_order = [
    "I", "II", "III", "IV", "V", "VI",
    "VII", "VIII", "IX", "X", "XI"
]

rank = {
    col: i
    for i, col in enumerate(columns_order)
}

batch["column_rank"] = (
    batch["column"]
    .astype(str)
    .map(rank)
)

batch = (
    batch
    .sort_values("column_rank")
    .reset_index(drop=True)
)

# ============================================================
# CHECA OS 11 CROPS
# ============================================================

missing = []

for _, row in batch.iterrows():

    path = Path(
        row["htr_crop_v2_path"]
    )

    if not path.exists():
        missing.append(str(path))

if missing:

    print("Arquivos ausentes:")

    for path in missing:
        print(path)

    raise FileNotFoundError(
        f"{len(missing)} crop(s) ausente(s)."
    )

# ============================================================
# CRIA ZIP NO GOOGLE DRIVE
# ============================================================

zip_path = (
    base /
    "htr_ground_truth_v2" /
    "1QS_BATCH03_NATIVE_MASKED.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for _, row in batch.iterrows():

        source = Path(
            row["htr_crop_v2_path"]
        )

        z.write(
            source,
            arcname=f"{row['htr_line_id']}.png"
        )

# ============================================================
# RESULTADO
# ============================================================

print("=" * 60)
print("1QS — BATCH 03 PREPARADO")
print("=" * 60)

print("\nLinhas:", len(batch))

display(
    batch[
        [
            "htr_line_id",
            "column",
            "split"
        ]
    ]
)

print("\nZIP salvo permanentemente em:")
print(zip_path)

print(
    "Tamanho:",
    round(
        zip_path.stat().st_size /
        1024 /
        1024,
        2
    ),
    "MB"
)

# ============================================================
# DOWNLOAD
# ============================================================

files.download(
    str(zip_path)
)

In [ ]:
from pathlib import Path
import pandas as pd
from PIL import Image, ImageDraw
from google.colab import files

base = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS"
)

# ============================================================
# ARQUIVOS
# ============================================================

htr = pd.read_csv(
    base /
    "frozen_segmentation" /
    "1QS_FINAL_HTR_MANIFEST.csv"
)

v2 = pd.read_csv(
    base /
    "htr_input_v2" /
    "1QS_HTR_INPUT_MANIFEST_V2.csv"
)

audit = pd.read_csv(
    base /
    "htr_ground_truth_v2" /
    "1QS_GT_SELECTION_V2_AUDIT.csv"
)

# ============================================================
# TODAS AS LINHAS JÁ USADAS OU JÁ TESTADAS COMO GT
# ============================================================

used_ids = set(
    audit["htr_line_id"]
    .dropna()
    .astype(str)
)

pool = htr[
    htr["column"].astype(str) == "X"
].copy()

pool = pool[
    ~pool["htr_line_id"]
    .astype(str)
    .isin(used_ids)
].copy()

# junta dados MASKED V2
pool = pool.merge(
    v2[
        [
            "htr_line_id",
            "htr_crop_v2_path",
            "crop_width_v2",
            "crop_height_v2",
            "n_boundaries",
            "n_fallbacks"
        ]
    ],
    on="htr_line_id",
    how="left"
)

# ============================================================
# SCORE DE QUALIDADE GEOMÉTRICA
# ============================================================

med_w = pool["crop_width_v2"].median()
med_h = pool["crop_height_v2"].median()

pool["width_score"] = (
    pool["crop_width_v2"] /
    med_w
)

pool["height_deviation"] = (
    (
        pool["crop_height_v2"] -
        med_h
    ).abs()
    /
    med_h
)

pool["quality_score"] = (
    pool["width_score"]
    -
    0.60 * pool["height_deviation"]
    -
    0.40 * (
        pool["n_fallbacks"] > 0
    ).astype(int)
    -
    0.20 * (
        pool["n_source_baselines"] > 1
    ).astype(int)
)

candidates = (
    pool
    .sort_values(
        "quality_score",
        ascending=False
    )
    .head(3)
    .copy()
)

print("Candidatas para substituir 1QS_X_M004:")

display(
    candidates[
        [
            "htr_line_id",
            "crop_width_v2",
            "crop_height_v2",
            "n_source_baselines",
            "n_fallbacks",
            "quality_score"
        ]
    ]
)

# ============================================================
# PRANCHA
# ============================================================

TARGET_WIDTH = 1800
HEADER = 80
MARGIN = 20

panels = []

for _, row in candidates.iterrows():

    path = Path(
        row["htr_crop_v2_path"]
    )

    img = Image.open(path).convert("RGB")

    scale = TARGET_WIDTH / img.width

    new_h = max(
        100,
        round(img.height * scale)
    )

    img = img.resize(
        (TARGET_WIDTH, new_h),
        Image.Resampling.LANCZOS
    )

    panel = Image.new(
        "RGB",
        (
            TARGET_WIDTH + 2*MARGIN,
            new_h + HEADER + 2*MARGIN
        ),
        "white"
    )

    panel.paste(
        img,
        (MARGIN, HEADER + MARGIN)
    )

    draw = ImageDraw.Draw(panel)

    draw.text(
        (MARGIN, 20),
        (
            f"SUBSTITUI 1QS_X_M004 | "
            f"CANDIDATA {row['htr_line_id']}"
        ),
        fill="black"
    )

    panels.append(panel)

width = max(p.width for p in panels)
height = sum(p.height for p in panels)

sheet = Image.new(
    "RGB",
    (width, height),
    "white"
)

y = 0

for panel in panels:
    sheet.paste(panel, (0, y))
    y += panel.height

out_dir = (
    base /
    "htr_ground_truth_v2" /
    "replacement_review"
)

out_dir.mkdir(
    parents=True,
    exist_ok=True
)

output = (
    out_dir /
    "1QS_BATCH03_X_REPLACEMENT_CANDIDATES.png"
)

sheet.save(output)

print("\nPrancha salva em:")
print(output)

files.download(str(output))

In [ ]:
from pathlib import Path
import pandas as pd

base = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS"
)

ref = pd.read_csv(
    base /
    "reference" /
    "1QS_palavras.csv"
)

# Coluna X = 10
# candidata = M016
r = ref[
    (ref["fragmento_coluna"] == 10)
    &
    (ref["linha"] == 16)
].copy()

print("=" * 65)
print("AUDITORIA DA CANDIDATA 1QS_X_M016")
print("=" * 65)

print("Tokens encontrados:", len(r))

print(
    "Reconstruídos pelo editor:",
    r["reconstruido_por_editor"]
    .fillna(False)
    .astype(bool)
    .sum()
)

# procura automaticamente qualquer coluna de incerteza
uncertainty_cols = [
    c for c in r.columns
    if (
        "incert" in c.lower()
        or
        "uncert" in c.lower()
    )
]

if uncertainty_cols:
    print("\nColunas de incerteza:")
    for c in uncertainty_cols:
        print(
            c,
            "->",
            r[c].dropna().unique()
        )
else:
    print("\nNenhuma coluna de incerteza encontrada.")

# mantém a ordem original do CSV
gt_candidate = " ".join(
    r["forma_diplomatica"]
    .dropna()
    .astype(str)
    .str.strip()
)

print("\nGT candidato:")
print(gt_candidate)

display(
    r[
        [
            c for c in [
                "fragmento_coluna",
                "linha",
                "forma_diplomatica",
                "reconstruido_por_editor"
            ]
            if c in r.columns
        ]
    ]
)

**[Public-release redaction]** Original research-log cell 148 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


In [ ]:
from pathlib import Path
import pandas as pd
import zipfile
from google.colab import files

base = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS"
)

active = pd.read_csv(
    base /
    "htr_ground_truth_v2" /
    "1QS_GT_SELECTION_V2_ACTIVE.csv"
)

v2 = pd.read_csv(
    base /
    "htr_input_v2" /
    "1QS_HTR_INPUT_MANIFEST_V2.csv"
)

# ============================================================
# BATCH 04
# ============================================================

batch = active[
    (active["split"] == "TRAIN")
    &
    (active["batch"] == 4)
].copy()

if len(batch) != 11:
    raise RuntimeError(
        f"Esperávamos 11 linhas no Batch 04; "
        f"encontrei {len(batch)}."
    )

# ============================================================
# GARANTE CAMINHO MASKED V2
# ============================================================

if "htr_crop_v2_path" not in batch.columns:

    batch = batch.merge(
        v2[
            [
                "htr_line_id",
                "htr_crop_v2_path"
            ]
        ],
        on="htr_line_id",
        how="left"
    )

else:

    missing_mask = (
        batch["htr_crop_v2_path"]
        .isna()
    )

    if missing_mask.any():

        lookup = (
            v2.set_index("htr_line_id")
            ["htr_crop_v2_path"]
        )

        batch.loc[
            missing_mask,
            "htr_crop_v2_path"
        ] = (
            batch.loc[
                missing_mask,
                "htr_line_id"
            ]
            .map(lookup)
        )

# ============================================================
# ORDEM DAS COLUNAS
# ============================================================

columns_order = [
    "I", "II", "III", "IV", "V", "VI",
    "VII", "VIII", "IX", "X", "XI"
]

rank = {
    col: i
    for i, col in enumerate(columns_order)
}

batch["column_rank"] = (
    batch["column"]
    .astype(str)
    .map(rank)
)

batch = (
    batch
    .sort_values("column_rank")
    .reset_index(drop=True)
)

# ============================================================
# CHECA OS CROPS
# ============================================================

missing = []

for _, row in batch.iterrows():

    path = Path(
        row["htr_crop_v2_path"]
    )

    if not path.exists():
        missing.append(str(path))

if missing:

    print("Arquivos ausentes:")

    for path in missing:
        print(path)

    raise FileNotFoundError(
        f"{len(missing)} crop(s) ausente(s)."
    )

# ============================================================
# CRIA ZIP NO DRIVE
# ============================================================

zip_path = (
    base /
    "htr_ground_truth_v2" /
    "1QS_BATCH04_NATIVE_MASKED.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for _, row in batch.iterrows():

        source = Path(
            row["htr_crop_v2_path"]
        )

        z.write(
            source,
            arcname=f"{row['htr_line_id']}.png"
        )

# ============================================================
# RESULTADO
# ============================================================

print("=" * 60)
print("1QS — BATCH 04 PREPARADO")
print("=" * 60)

print("\nLinhas:", len(batch))

display(
    batch[
        [
            "htr_line_id",
            "column",
            "split"
        ]
    ]
)

print("\nZIP salvo permanentemente em:")
print(zip_path)

print(
    "Tamanho:",
    round(
        zip_path.stat().st_size /
        1024 /
        1024,
        2
    ),
    "MB"
)

# ============================================================
# DOWNLOAD
# ============================================================

files.download(
    str(zip_path)
)

In [ ]:
from pathlib import Path
import pandas as pd
from PIL import Image, ImageDraw
from google.colab import files
import re

base = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS"
)

htr = pd.read_csv(
    base /
    "frozen_segmentation" /
    "1QS_FINAL_HTR_MANIFEST.csv"
)

v2 = pd.read_csv(
    base /
    "htr_input_v2" /
    "1QS_HTR_INPUT_MANIFEST_V2.csv"
)

audit = pd.read_csv(
    base /
    "htr_ground_truth_v2" /
    "1QS_GT_SELECTION_V2_AUDIT.csv"
)

ref = pd.read_csv(
    base /
    "reference" /
    "1QS_palavras.csv"
)

# ============================================================
# CASOS A SUBSTITUIR
# ============================================================

targets = {
    "IV": "1QS_IV_M017",
    "VII": "1QS_VII_M022",
    "X": "1QS_X_M006"
}

column_numbers = {
    "I":1, "II":2, "III":3, "IV":4, "V":5,
    "VI":6, "VII":7, "VIII":8, "IX":9,
    "X":10, "XI":11
}

# ============================================================
# IDs JÁ USADOS / RESERVADOS
# ============================================================

used_ids = set(
    audit["htr_line_id"]
    .dropna()
    .astype(str)
)

# ============================================================
# VERIFICA SE UMA LINHA ETCBC É LIMPA
# ============================================================

def clean_reference_line(col, line_no):

    col_no = column_numbers[col]

    r = ref[
        (ref["fragmento_coluna"] == col_no)
        &
        (ref["linha"] == line_no)
    ].copy()

    if len(r) == 0:
        return False

    # reconstrução editorial
    reconstructed = (
        r["reconstruido_por_editor"]
        .fillna(False)
        .astype(bool)
        .sum()
    )

    if reconstructed != 0:
        return False

    # incerteza
    max_unc = (
        pd.to_numeric(
            r["nivel_maximo_incerteza"],
            errors="coerce"
        )
        .fillna(0)
        .max()
    )

    if max_unc != 0:
        return False

    # marcações especiais incompatíveis com GT simples
    raw = " ".join(
        r["forma_diplomatica"]
        .fillna("")
        .astype(str)
    )

    forbidden = [
        "{{",
        "}}",
        "#",
        "(^",
        "^)"
    ]

    if any(
        marker in raw
        for marker in forbidden
    ):
        return False

    return True


# ============================================================
# SELECIONA 3 CANDIDATAS POR COLUNA
# ============================================================

candidate_sets = []

for col, replacement_for in targets.items():

    pool = htr[
        htr["column"].astype(str) == col
    ].copy()

    pool = pool[
        ~pool["htr_line_id"]
        .astype(str)
        .isin(used_ids)
    ].copy()

    pool = pool.merge(
        v2[
            [
                "htr_line_id",
                "htr_crop_v2_path",
                "crop_width_v2",
                "crop_height_v2",
                "n_boundaries",
                "n_fallbacks"
            ]
        ],
        on="htr_line_id",
        how="left"
    )

    # número Mxxx = número físico/editorial da linha
    pool["candidate_line_number"] = (
        pool["htr_line_id"]
        .str.extract(r"_M(\d+)$")[0]
        .astype(int)
    )

    pool["reference_clean"] = pool[
        "candidate_line_number"
    ].apply(
        lambda n:
        clean_reference_line(
            col,
            int(n)
        )
    )

    pool = pool[
        pool["reference_clean"]
    ].copy()

    if len(pool) < 3:
        raise RuntimeError(
            f"Coluna {col}: somente "
            f"{len(pool)} candidatas limpas."
        )

    # ========================================================
    # SCORE VISUAL/GEOMÉTRICO
    # ========================================================

    med_w = pool[
        "crop_width_v2"
    ].median()

    med_h = pool[
        "crop_height_v2"
    ].median()

    pool["width_score"] = (
        pool["crop_width_v2"] /
        med_w
    )

    pool["height_deviation"] = (
        (
            pool["crop_height_v2"]
            - med_h
        ).abs()
        /
        med_h
    )

    pool["quality_score"] = (
        pool["width_score"]
        -
        0.60 * pool["height_deviation"]
        -
        0.40 * (
            pool["n_fallbacks"] > 0
        ).astype(int)
        -
        0.20 * (
            pool["n_source_baselines"] > 1
        ).astype(int)
    )

    best = (
        pool
        .sort_values(
            "quality_score",
            ascending=False
        )
        .head(3)
        .copy()
    )

    best["replacement_for"] = (
        replacement_for
    )

    candidate_sets.append(best)

candidates = pd.concat(
    candidate_sets,
    ignore_index=True
)

# ============================================================
# SALVA TABELA
# ============================================================

candidate_path = (
    base /
    "htr_ground_truth_v2" /
    "1QS_BATCH04_REPLACEMENT_CANDIDATES.csv"
)

candidates.to_csv(
    candidate_path,
    index=False,
    encoding="utf-8-sig"
)

display(
    candidates[
        [
            "replacement_for",
            "htr_line_id",
            "column",
            "candidate_line_number",
            "crop_width_v2",
            "crop_height_v2",
            "quality_score"
        ]
    ]
)

# ============================================================
# CRIA PRANCHA DAS 9 CANDIDATAS
# ============================================================

TARGET_WIDTH = 1700
HEADER = 80
MARGIN = 20

panels = []

for _, row in candidates.iterrows():

    path = Path(
        row["htr_crop_v2_path"]
    )

    img = Image.open(
        path
    ).convert("RGB")

    scale = (
        TARGET_WIDTH /
        img.width
    )

    new_h = max(
        100,
        round(
            img.height * scale
        )
    )

    img = img.resize(
        (
            TARGET_WIDTH,
            new_h
        ),
        Image.Resampling.LANCZOS
    )

    panel = Image.new(
        "RGB",
        (
            TARGET_WIDTH + 2*MARGIN,
            new_h + HEADER + 2*MARGIN
        ),
        "white"
    )

    panel.paste(
        img,
        (
            MARGIN,
            HEADER + MARGIN
        )
    )

    draw = ImageDraw.Draw(
        panel
    )

    draw.text(
        (
            MARGIN,
            20
        ),
        (
            f"SUBSTITUI {row['replacement_for']} | "
            f"CANDIDATA {row['htr_line_id']} | "
            f"REF CLEAN"
        ),
        fill="black"
    )

    panels.append(panel)

width = max(
    p.width
    for p in panels
)

height = sum(
    p.height
    for p in panels
)

sheet = Image.new(
    "RGB",
    (
        width,
        height
    ),
    "white"
)

y = 0

for panel in panels:

    sheet.paste(
        panel,
        (0, y)
    )

    y += panel.height

out_dir = (
    base /
    "htr_ground_truth_v2" /
    "replacement_review"
)

out_dir.mkdir(
    parents=True,
    exist_ok=True
)

output = (
    out_dir /
    "1QS_BATCH04_REPLACEMENT_CANDIDATES.png"
)

sheet.save(output)

print("\nPrancha:")
print(output)

files.download(
    str(output)
)

**[Public-release redaction]** Original research-log cell 151 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


In [ ]:
from pathlib import Path
import pandas as pd
import zipfile
from google.colab import files

base = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS"
)

active = pd.read_csv(
    base /
    "htr_ground_truth_v2" /
    "1QS_GT_SELECTION_V2_ACTIVE.csv"
)

v2 = pd.read_csv(
    base /
    "htr_input_v2" /
    "1QS_HTR_INPUT_MANIFEST_V2.csv"
)

# ============================================================
# BATCH 05 — ÚLTIMO TRAIN
# ============================================================

batch = active[
    (active["split"] == "TRAIN")
    &
    (active["batch"] == 5)
].copy()

if len(batch) != 11:
    raise RuntimeError(
        f"Esperávamos 11 linhas no Batch 05; "
        f"encontrei {len(batch)}."
    )

# ============================================================
# GARANTE CAMINHO MASKED V2
# ============================================================

if "htr_crop_v2_path" not in batch.columns:

    batch = batch.merge(
        v2[
            [
                "htr_line_id",
                "htr_crop_v2_path"
            ]
        ],
        on="htr_line_id",
        how="left"
    )

else:

    missing_mask = (
        batch["htr_crop_v2_path"].isna()
    )

    if missing_mask.any():

        lookup = (
            v2
            .set_index("htr_line_id")
            ["htr_crop_v2_path"]
        )

        batch.loc[
            missing_mask,
            "htr_crop_v2_path"
        ] = (
            batch.loc[
                missing_mask,
                "htr_line_id"
            ]
            .map(lookup)
        )

# ============================================================
# ORDEM I -> XI
# ============================================================

columns_order = [
    "I", "II", "III", "IV", "V", "VI",
    "VII", "VIII", "IX", "X", "XI"
]

rank = {
    col: i
    for i, col in enumerate(columns_order)
}

batch["column_rank"] = (
    batch["column"]
    .astype(str)
    .map(rank)
)

batch = (
    batch
    .sort_values("column_rank")
    .reset_index(drop=True)
)

# ============================================================
# CONFERE CROPS
# ============================================================

missing = []

for _, row in batch.iterrows():

    path = Path(
        row["htr_crop_v2_path"]
    )

    if not path.exists():
        missing.append(str(path))

if missing:

    print("Arquivos ausentes:")

    for path in missing:
        print(path)

    raise FileNotFoundError(
        f"{len(missing)} crop(s) ausente(s)."
    )

# ============================================================
# CRIA ZIP NO DRIVE
# ============================================================

zip_path = (
    base /
    "htr_ground_truth_v2" /
    "1QS_BATCH05_NATIVE_MASKED.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for _, row in batch.iterrows():

        source = Path(
            row["htr_crop_v2_path"]
        )

        z.write(
            source,
            arcname=f"{row['htr_line_id']}.png"
        )

# ============================================================
# RESULTADO
# ============================================================

print("=" * 60)
print("1QS — BATCH 05 PREPARADO")
print("=" * 60)

print("\nLinhas:", len(batch))

display(
    batch[
        [
            "htr_line_id",
            "column",
            "split"
        ]
    ]
)

print("\nZIP salvo permanentemente em:")
print(zip_path)

print(
    "Tamanho:",
    round(
        zip_path.stat().st_size /
        1024 /
        1024,
        2
    ),
    "MB"
)

files.download(
    str(zip_path)
)

In [ ]:
from pathlib import Path
import pandas as pd
from PIL import Image, ImageDraw
from google.colab import files

base = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS"
)

htr = pd.read_csv(
    base /
    "frozen_segmentation" /
    "1QS_FINAL_HTR_MANIFEST.csv"
)

v2 = pd.read_csv(
    base /
    "htr_input_v2" /
    "1QS_HTR_INPUT_MANIFEST_V2.csv"
)

audit = pd.read_csv(
    base /
    "htr_ground_truth_v2" /
    "1QS_GT_SELECTION_V2_AUDIT.csv"
)

ref = pd.read_csv(
    base /
    "reference" /
    "1QS_palavras.csv"
)

# ============================================================
# 6 CASOS A SUBSTITUIR
# ============================================================

targets = {
    "I":   "1QS_I_M025",
    "II":  "1QS_II_M024",
    "VI":  "1QS_VI_M027",
    "VII": "1QS_VII_M025",
    "IX":  "1QS_IX_M026",
    "XI":  "1QS_XI_M016"
}

column_numbers = {
    "I": 1,
    "II": 2,
    "III": 3,
    "IV": 4,
    "V": 5,
    "VI": 6,
    "VII": 7,
    "VIII": 8,
    "IX": 9,
    "X": 10,
    "XI": 11
}

# ============================================================
# NÃO REUTILIZA LINHAS JÁ SELECIONADAS/TESTADAS
# ============================================================

used_ids = set(
    audit["htr_line_id"]
    .dropna()
    .astype(str)
)

# ============================================================
# TESTA SE A LINHA DE REFERÊNCIA É ADEQUADA PARA TRAIN
# ============================================================

def clean_reference_line(col, line_no):

    r = ref[
        (
            ref["fragmento_coluna"]
            == column_numbers[col]
        )
        &
        (
            ref["linha"]
            == int(line_no)
        )
    ].copy()

    if len(r) == 0:
        return False

    reconstructed = (
        r["reconstruido_por_editor"]
        .fillna(False)
        .astype(bool)
        .sum()
    )

    if reconstructed != 0:
        return False

    uncertainty = (
        pd.to_numeric(
            r["nivel_maximo_incerteza"],
            errors="coerce"
        )
        .fillna(0)
        .max()
    )

    if uncertainty != 0:
        return False

    raw = " ".join(
        r["forma_diplomatica"]
        .fillna("")
        .astype(str)
    )

    # marcas que não queremos no GT simples
    forbidden = [
        "{{",
        "}}",
        "#",
        "(^",
        "^)",
        "[",
        "]",
        "(<",
        ">)"
    ]

    if any(
        marker in raw
        for marker in forbidden
    ):
        return False

    return True


# ============================================================
# ESCOLHE 3 BOAS CANDIDATAS POR COLUNA
# ============================================================

candidate_sets = []

for col, replacement_for in targets.items():

    pool = htr[
        htr["column"].astype(str)
        == col
    ].copy()

    pool = pool[
        ~pool["htr_line_id"]
        .astype(str)
        .isin(used_ids)
    ].copy()

    pool = pool.merge(
        v2[
            [
                "htr_line_id",
                "htr_crop_v2_path",
                "crop_width_v2",
                "crop_height_v2",
                "n_boundaries",
                "n_fallbacks"
            ]
        ],
        on="htr_line_id",
        how="left"
    )

    pool["candidate_line_number"] = (
        pool["htr_line_id"]
        .str.extract(r"_M(\d+)$")[0]
        .astype(int)
    )

    pool["reference_clean"] = (
        pool["candidate_line_number"]
        .apply(
            lambda n:
            clean_reference_line(
                col,
                int(n)
            )
        )
    )

    pool = pool[
        pool["reference_clean"]
    ].copy()

    if len(pool) < 3:

        raise RuntimeError(
            f"Coluna {col}: somente "
            f"{len(pool)} candidatas limpas."
        )

    # ========================================================
    # SCORE GEOMÉTRICO
    # ========================================================

    med_w = pool[
        "crop_width_v2"
    ].median()

    med_h = pool[
        "crop_height_v2"
    ].median()

    pool["width_score"] = (
        pool["crop_width_v2"] /
        med_w
    )

    pool["height_deviation"] = (
        (
            pool["crop_height_v2"]
            - med_h
        ).abs()
        /
        med_h
    )

    pool["quality_score"] = (
        pool["width_score"]
        -
        0.60 * pool["height_deviation"]
        -
        0.40 * (
            pool["n_fallbacks"] > 0
        ).astype(int)
        -
        0.20 * (
            pool["n_source_baselines"] > 1
        ).astype(int)
    )

    best = (
        pool
        .sort_values(
            "quality_score",
            ascending=False
        )
        .head(3)
        .copy()
    )

    best["replacement_for"] = (
        replacement_for
    )

    candidate_sets.append(best)

candidates = pd.concat(
    candidate_sets,
    ignore_index=True
)

# ============================================================
# SALVA AS CANDIDATAS
# ============================================================

candidate_path = (
    base /
    "htr_ground_truth_v2" /
    "1QS_BATCH05_REPLACEMENT_CANDIDATES.csv"
)

candidates.to_csv(
    candidate_path,
    index=False,
    encoding="utf-8-sig"
)

display(
    candidates[
        [
            "replacement_for",
            "htr_line_id",
            "column",
            "candidate_line_number",
            "crop_width_v2",
            "crop_height_v2",
            "quality_score"
        ]
    ]
)

# ============================================================
# CRIA PRANCHA COM AS 18 CANDIDATAS
# ============================================================

TARGET_WIDTH = 1700
HEADER = 75
MARGIN = 20

panels = []

for _, row in candidates.iterrows():

    path = Path(
        row["htr_crop_v2_path"]
    )

    img = Image.open(
        path
    ).convert("RGB")

    scale = (
        TARGET_WIDTH /
        img.width
    )

    new_h = max(
        100,
        round(
            img.height * scale
        )
    )

    img = img.resize(
        (
            TARGET_WIDTH,
            new_h
        ),
        Image.Resampling.LANCZOS
    )

    panel = Image.new(
        "RGB",
        (
            TARGET_WIDTH + 2*MARGIN,
            new_h + HEADER + 2*MARGIN
        ),
        "white"
    )

    panel.paste(
        img,
        (
            MARGIN,
            HEADER + MARGIN
        )
    )

    draw = ImageDraw.Draw(
        panel
    )

    draw.text(
        (
            MARGIN,
            18
        ),
        (
            f"SUBSTITUI {row['replacement_for']} | "
            f"CANDIDATA {row['htr_line_id']} | "
            f"REF CLEAN"
        ),
        fill="black"
    )

    panels.append(panel)

width = max(
    p.width
    for p in panels
)

height = sum(
    p.height
    for p in panels
)

sheet = Image.new(
    "RGB",
    (
        width,
        height
    ),
    "white"
)

y = 0

for panel in panels:

    sheet.paste(
        panel,
        (0, y)
    )

    y += panel.height

out_dir = (
    base /
    "htr_ground_truth_v2" /
    "replacement_review"
)

out_dir.mkdir(
    parents=True,
    exist_ok=True
)

output = (
    out_dir /
    "1QS_BATCH05_REPLACEMENT_CANDIDATES.png"
)

sheet.save(output)

print("\nPrancha criada:")
print(output)

print(
    "\nCandidatas:",
    len(candidates)
)

files.download(
    str(output)
)

**[Public-release redaction]** Original research-log cell 154 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


In [ ]:
from pathlib import Path
import pandas as pd
import zipfile
import hashlib
from google.colab import files

# ============================================================
# CONFIGURAÇÃO
# ============================================================

base = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS"
)

gt_v2 = (
    base /
    "htr_ground_truth_v2"
)

active_path = (
    gt_v2 /
    "1QS_GT_SELECTION_V2_ACTIVE.csv"
)

v2_path = (
    base /
    "htr_input_v2" /
    "1QS_HTR_INPUT_MANIFEST_V2.csv"
)

active = pd.read_csv(active_path)
v2 = pd.read_csv(v2_path)

# ============================================================
# BATCH 06 = VALIDATION
# ============================================================

batch = active[
    (active["split"] == "VALIDATION")
    &
    (active["batch"] == 6)
].copy()

if len(batch) != 11:
    raise RuntimeError(
        f"VALIDATION deveria conter 11 linhas; "
        f"encontrei {len(batch)}."
    )

# ============================================================
# CONFERE UMA LINHA POR COLUNA
# ============================================================

columns_order = [
    "I", "II", "III", "IV", "V", "VI",
    "VII", "VIII", "IX", "X", "XI"
]

counts = (
    batch["column"]
    .astype(str)
    .value_counts()
    .to_dict()
)

for col in columns_order:

    if counts.get(col, 0) != 1:
        raise RuntimeError(
            f"Coluna {col}: esperado 1 exemplo "
            f"de VALIDATION; encontrei "
            f"{counts.get(col, 0)}."
        )

# ============================================================
# GARANTE CAMINHOS MASKED V2
# ============================================================

if "htr_crop_v2_path" not in batch.columns:

    batch = batch.merge(
        v2[
            [
                "htr_line_id",
                "htr_crop_v2_path"
            ]
        ],
        on="htr_line_id",
        how="left"
    )

else:

    missing_mask = (
        batch["htr_crop_v2_path"]
        .isna()
    )

    if missing_mask.any():

        lookup = (
            v2
            .set_index("htr_line_id")
            ["htr_crop_v2_path"]
        )

        batch.loc[
            missing_mask,
            "htr_crop_v2_path"
        ] = (
            batch.loc[
                missing_mask,
                "htr_line_id"
            ]
            .map(lookup)
        )

# ============================================================
# ORDEM I -> XI
# ============================================================

rank = {
    col: i
    for i, col in enumerate(columns_order)
}

batch["column_rank_validation"] = (
    batch["column"]
    .astype(str)
    .map(rank)
)

batch = (
    batch
    .sort_values("column_rank_validation")
    .reset_index(drop=True)
)

# ============================================================
# CONFERE OS 11 PNGs
# ============================================================

missing = []

for _, row in batch.iterrows():

    p = Path(
        row["htr_crop_v2_path"]
    )

    if not p.exists():
        missing.append(str(p))

if missing:

    print("Arquivos ausentes:")

    for p in missing:
        print(p)

    raise FileNotFoundError(
        f"{len(missing)} crop(s) ausente(s)."
    )

# ============================================================
# HASH DE CADA IMAGEM
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


batch["image_sha256"] = (
    batch["htr_crop_v2_path"]
    .apply(
        lambda p:
        sha256_file(Path(p))
    )
)

# ============================================================
# MANIFESTO BLOQUEADO DA VALIDATION
# ============================================================

batch["validation_locked"] = True
batch["replacement_allowed"] = False
batch["reference_review_status"] = "NOT_YET_REVIEWED"

lock_path = (
    gt_v2 /
    "1QS_BATCH06_VALIDATION_LOCKED.csv"
)

batch.to_csv(
    lock_path,
    index=False,
    encoding="utf-8-sig"
)

# hash do próprio manifesto
lock_hash = sha256_file(
    lock_path
)

hash_path = (
    gt_v2 /
    "1QS_BATCH06_VALIDATION_LOCKED.sha256.txt"
)

hash_path.write_text(
    f"{lock_hash}  {lock_path.name}\n",
    encoding="utf-8"
)

# ============================================================
# CRIA ZIP NATIVO
# ============================================================

zip_path = (
    gt_v2 /
    "1QS_BATCH06_VALIDATION_NATIVE_MASKED.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for _, row in batch.iterrows():

        src = Path(
            row["htr_crop_v2_path"]
        )

        z.write(
            src,
            arcname=f"{row['htr_line_id']}.png"
        )

# ============================================================
# RESULTADO
# ============================================================

print("=" * 68)
print("1QS — BATCH 06 VALIDATION CONGELADO")
print("=" * 68)

print("\nLinhas:", len(batch))
print("Substituições permitidas: NÃO")

display(
    batch[
        [
            "htr_line_id",
            "column",
            "split",
            "batch",
            "image_sha256"
        ]
    ]
)

print("\nManifesto congelado:")
print(lock_path)

print("\nSHA-256 do manifesto:")
print(lock_hash)

print("\nZIP:")
print(zip_path)

print(
    "Tamanho:",
    round(
        zip_path.stat().st_size / 1024 / 1024,
        2
    ),
    "MB"
)

# ============================================================
# DOWNLOAD
# ============================================================

files.download(
    str(zip_path)
)

In [ ]:
from pathlib import Path
import pandas as pd
from datetime import datetime, timezone

base = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS"
)

gt_v2 = (
    base /
    "htr_ground_truth_v2"
)

lock_path = (
    gt_v2 /
    "1QS_BATCH06_VALIDATION_LOCKED.csv"
)

reference_path = (
    base /
    "reference" /
    "1QS_palavras.csv"
)

locked = pd.read_csv(lock_path)
ref = pd.read_csv(reference_path)

timestamp = datetime.now(
    timezone.utc
).isoformat()

# ============================================================
# CLASSIFICAÇÃO RESULTANTE DA INSPEÇÃO IMAGE-ONLY
# + AUDITORIA POSTERIOR DA REFERÊNCIA
# ============================================================

classification = {

    "1QS_I_M009": {
        "validation_class":
            "CORE_CLEAN",
        "primary_metric_eligible":
            True,
        "visual_status":
            "FULL_READABLE",
        "note":
            "Full readable crop; reference has no uncertainty or editorial reconstruction."
    },

    "1QS_II_M013": {
        "validation_class":
            "CORE_SUPRALINEAR",
        "primary_metric_eligible":
            True,
        "visual_status":
            "READABLE_WITH_SUPRALINEAR",
        "note":
            "Readable line with supralinear insertion; reference uncertainty=0."
    },

    "1QS_III_M002": {
        "validation_class":
            "CORE_SUPRALINEAR",
        "primary_metric_eligible":
            True,
        "visual_status":
            "READABLE_WITH_SUPRALINEAR",
        "note":
            "Readable line with supralinear insertion; reference uncertainty=0."
    },

    "1QS_IV_M008": {
        "validation_class":
            "STRUCTURAL_PARTIAL_CROP",
        "primary_metric_eligible":
            False,
        "visual_status":
            "PARTIAL_VISIBLE_LINE",
        "note":
            "Masked crop contains only a portion of the editorial line; must not be scored against the complete ETCBC line."
    },

    "1QS_V_M024": {
        "validation_class":
            "CORE_SUPRALINEAR",
        "primary_metric_eligible":
            True,
        "visual_status":
            "READABLE_WITH_SUPRALINEAR",
        "note":
            "Readable line with supralinear insertion; reference uncertainty=0."
    },

    "1QS_VI_M026": {
        "validation_class":
            "CHALLENGE_REFERENCE_UNCERTAIN",
        "primary_metric_eligible":
            False,
        "visual_status":
            "READABLE_BUT_REFERENCE_UNCERTAIN",
        "note":
            "Reference contains maximum uncertainty level 2."
    },

    "1QS_VII_M008": {
        "validation_class":
            "CHALLENGE_COMPLEX_EDITORIAL",
        "primary_metric_eligible":
            False,
        "visual_status":
            "MULTIPLE_CORRECTIONS",
        "note":
            "Multiple supralinear corrections, deletions/lacunae, and competing textual states."
    },

    "1QS_VIII_M010": {
        "validation_class":
            "CHALLENGE_HIGH_UNCERTAINTY",
        "primary_metric_eligible":
            False,
        "visual_status":
            "DAMAGED_COMPLEX",
        "note":
            "Complex damaged region; reference maximum uncertainty level 4."
    },

    "1QS_IX_M019": {
        "validation_class":
            "CHALLENGE_HIGH_UNCERTAINTY",
        "primary_metric_eligible":
            False,
        "visual_status":
            "LACUNA_AND_UNCERTAINTY",
        "note":
            "Reference contains lacunae and maximum uncertainty level 4."
    },

    "1QS_X_M008": {
        "validation_class":
            "CHALLENGE_HIGH_UNCERTAINTY",
        "primary_metric_eligible":
            False,
        "visual_status":
            "LACUNA_AND_UNCERTAINTY",
        "note":
            "Reference contains lacunae and maximum uncertainty level 4."
    },

    "1QS_XI_M009": {
        "validation_class":
            "CHALLENGE_LACUNA",
        "primary_metric_eligible":
            False,
        "visual_status":
            "READABLE_WITH_LACUNA",
        "note":
            "Reference uncertainty level is 0 but contains an explicit lacuna."
    }
}

# ============================================================
# EXTRAI ESTATÍSTICAS OBJETIVAS DA REFERÊNCIA
# ============================================================

rows = []

for _, row in locked.iterrows():

    line_id = str(
        row["htr_line_id"]
    )

    col = int(
        row["column_rank"]
        + 1
    )

    # número Mxxx
    line_no = int(
        line_id.split("_M")[-1]
    )

    r = ref[
        (ref["fragmento_coluna"] == col)
        &
        (ref["linha"] == line_no)
    ].copy()

    if len(r) == 0:
        raise RuntimeError(
            f"Referência não encontrada para {line_id}"
        )

    reconstructed = (
        r["reconstruido_por_editor"]
        .fillna(False)
        .astype(bool)
        .sum()
    )

    uncertainty = (
        pd.to_numeric(
            r["nivel_maximo_incerteza"],
            errors="coerce"
        )
        .fillna(0)
        .max()
    )

    raw_reference = " ".join(
        r["forma_diplomatica"]
        .fillna("")
        .astype(str)
    )

    info = classification[
        line_id
    ]

    out = row.to_dict()

    out.update({

        "reference_column":
            col,

        "reference_line":
            line_no,

        "reference_token_count":
            len(r),

        "editor_reconstructed_tokens":
            int(reconstructed),

        "maximum_reference_uncertainty":
            int(uncertainty),

        "reference_has_lacuna":
            "{{" in raw_reference
            or "#" in raw_reference,

        "reference_has_supralinear":
            "(^" in raw_reference,

        "validation_class":
            info["validation_class"],

        "primary_metric_eligible":
            info["primary_metric_eligible"],

        "visual_status":
            info["visual_status"],

        "audit_note":
            info["note"],

        "replacement_allowed":
            False,

        "validation_locked":
            True,

        "audit_timestamp_utc":
            timestamp
    })

    rows.append(out)

audit = pd.DataFrame(
    rows
)

# ============================================================
# SALVA
# ============================================================

audit_path = (
    gt_v2 /
    "1QS_BATCH06_VALIDATION_AUDIT.csv"
)

audit.to_csv(
    audit_path,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# RESUMO
# ============================================================

print("=" * 70)
print("1QS — VALIDATION AUDITADA")
print("=" * 70)

print(
    "\nLinhas fixas:",
    len(audit)
)

print(
    "Substituições:",
    0
)

print(
    "Elegíveis para métrica CORE:",
    audit[
        "primary_metric_eligible"
    ].sum()
)

print(
    "Casos de challenge/estrutura:",
    (
        ~audit[
            "primary_metric_eligible"
        ]
    ).sum()
)

print("\nClasses:")

print(
    audit[
        "validation_class"
    ].value_counts()
)

print("\nDetalhes:")

display(
    audit[
        [
            "htr_line_id",
            "validation_class",
            "editor_reconstructed_tokens",
            "maximum_reference_uncertainty",
            "reference_has_lacuna",
            "reference_has_supralinear",
            "primary_metric_eligible"
        ]
    ]
)

print("\nArquivo:")
print(audit_path)

**[Public-release redaction]** Original research-log cell 157 withheld because it includes manually or editorially sourced Hebrew transcription text. See private original notebook for complete research history.


In [ ]:
from pathlib import Path
import pandas as pd
import hashlib
import json
from datetime import datetime, timezone

# ============================================================
# CONFIGURAÇÃO
# ============================================================

base = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS"
)

gt_v2 = (
    base /
    "htr_ground_truth_v2"
)

active_path = (
    gt_v2 /
    "1QS_GT_SELECTION_V2_ACTIVE.csv"
)

v2_path = (
    base /
    "htr_input_v2" /
    "1QS_HTR_INPUT_MANIFEST_V2.csv"
)

active = pd.read_csv(active_path)
v2 = pd.read_csv(v2_path)

timestamp = datetime.now(
    timezone.utc
).isoformat()

# ============================================================
# SELECIONA TEST — SEM REFERÊNCIA
# ============================================================

test = active[
    (active["split"] == "TEST")
    &
    (active["batch"] == 7)
].copy()

if len(test) != 11:

    raise RuntimeError(
        f"TEST deveria conter 11 linhas; "
        f"encontrei {len(test)}."
    )

# ============================================================
# CONFERE 1 LINHA POR COLUNA
# ============================================================

columns_order = [
    "I", "II", "III", "IV", "V", "VI",
    "VII", "VIII", "IX", "X", "XI"
]

counts = (
    test["column"]
    .astype(str)
    .value_counts()
    .to_dict()
)

for col in columns_order:

    if counts.get(col, 0) != 1:

        raise RuntimeError(
            f"Coluna {col}: esperado 1 TEST; "
            f"encontrei {counts.get(col, 0)}."
        )

# ============================================================
# GARANTE CAMINHOS MASKED V2
# sem abrir as imagens
# ============================================================

if "htr_crop_v2_path" not in test.columns:

    test = test.merge(
        v2[
            [
                "htr_line_id",
                "htr_crop_v2_path"
            ]
        ],
        on="htr_line_id",
        how="left"
    )

else:

    missing_mask = (
        test["htr_crop_v2_path"]
        .isna()
    )

    if missing_mask.any():

        lookup = (
            v2
            .set_index("htr_line_id")
            ["htr_crop_v2_path"]
        )

        test.loc[
            missing_mask,
            "htr_crop_v2_path"
        ] = (
            test.loc[
                missing_mask,
                "htr_line_id"
            ]
            .map(lookup)
        )

# ============================================================
# CONFERE EXISTÊNCIA — SEM VISUALIZAR
# ============================================================

missing_files = []

for p in test["htr_crop_v2_path"]:

    path = Path(p)

    if not path.exists():
        missing_files.append(str(path))

if missing_files:

    raise FileNotFoundError(
        "\n".join(missing_files)
    )

# ============================================================
# HASH SHA-256 DAS IMAGENS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


test["image_sha256"] = (
    test["htr_crop_v2_path"]
    .apply(
        lambda p:
        sha256_file(Path(p))
    )
)

# ============================================================
# CAMPOS DE BLOQUEIO
# ============================================================

test["test_locked"] = True
test["replacement_allowed"] = False

test["image_reviewed_before_final_evaluation"] = False

test["reference_consulted_before_final_evaluation"] = False

test["gt_created_before_final_evaluation"] = False

test["test_status"] = "SEALED_UNSEEN"

test["locked_timestamp_utc"] = timestamp

# ============================================================
# ORDEM DETERMINÍSTICA
# ============================================================

rank = {
    col: i
    for i, col in enumerate(columns_order)
}

test["_rank"] = (
    test["column"]
    .astype(str)
    .map(rank)
)

test = (
    test
    .sort_values("_rank")
    .drop(columns="_rank")
    .reset_index(drop=True)
)

# ============================================================
# SALVA MANIFESTO SELADO
# ============================================================

lock_path = (
    gt_v2 /
    "1QS_BATCH07_TEST_LOCKED.csv"
)

test.to_csv(
    lock_path,
    index=False,
    encoding="utf-8-sig"
)

lock_hash = sha256_file(
    lock_path
)

hash_path = (
    gt_v2 /
    "1QS_BATCH07_TEST_LOCKED.sha256.txt"
)

hash_path.write_text(
    f"{lock_hash}  {lock_path.name}\n",
    encoding="utf-8"
)

# ============================================================
# METADATA DO BLOQUEIO
# ============================================================

metadata = {

    "manuscript":
        "1QS",

    "split":
        "TEST",

    "batch":
        7,

    "n_lines":
        11,

    "one_line_per_column":
        True,

    "replacement_allowed":
        False,

    "images_reviewed":
        False,

    "reference_consulted":
        False,

    "ground_truth_created":
        False,

    "status":
        "SEALED_UNSEEN",

    "manifest_sha256":
        lock_hash,

    "locked_timestamp_utc":
        timestamp
}

metadata_path = (
    gt_v2 /
    "1QS_BATCH07_TEST_LOCK_METADATA.json"
)

metadata_path.write_text(
    json.dumps(
        metadata,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

# ============================================================
# ATUALIZA SOMENTE METADADOS NO ACTIVE
# ============================================================

for col, default in {
    "test_locked": False,
    "test_status": pd.NA
}.items():

    if col not in active.columns:
        active[col] = default

test_ids = set(
    test["htr_line_id"]
    .astype(str)
)

mask = (
    active["htr_line_id"]
    .astype(str)
    .isin(test_ids)
)

active.loc[
    mask,
    "test_locked"
] = True

active.loc[
    mask,
    "test_status"
] = "SEALED_UNSEEN"

active.to_csv(
    active_path,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# RESULTADO
# ============================================================

print("=" * 72)
print("1QS — BATCH 07 TEST SELADO")
print("=" * 72)

print("Linhas:", len(test))
print("Colunas representadas:", 11)
print("Substituições permitidas: NÃO")
print("Imagens visualizadas: NÃO")
print("Referência consultada: NÃO")
print("GT criado: NÃO")
print("Status: SEALED_UNSEEN")

print("\nSHA-256 do manifesto:")
print(lock_hash)

print("\nManifesto:")
print(lock_path)

print("\nMetadata:")
print(metadata_path)

In [ ]:
from pathlib import Path
import pandas as pd
import hashlib
import shutil
import subprocess
import urllib.request
import os

# ============================================================
# CONFIGURAÇÃO
# ============================================================

base = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS"
)

gt_root = (
    base /
    "htr_ground_truth_v2"
)

train_images = (
    gt_root /
    "images"
)

val_root = (
    gt_root /
    "validation_core"
)

val_images = (
    val_root /
    "images"
)

model_root = (
    base /
    "htr_model"
)

baseline_root = (
    model_root /
    "00_pretrained_baseline"
)

data_root = (
    model_root /
    "datasets"
)

for p in [
    model_root,
    baseline_root,
    data_root
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )

# ============================================================
# LOCALIZA KETOS
# ============================================================

ketos = shutil.which("ketos")

if ketos is None:

    fallback = (
        Path.home() /
        ".local" /
        "bin" /
        "ketos"
    )

    if fallback.exists():
        ketos = str(fallback)

if ketos is None:

    raise RuntimeError(
        "Não encontrei o executável ketos."
    )

print("KETOS:", ketos)

# ============================================================
# CARREGA MANIFESTOS
# ============================================================

active = pd.read_csv(
    gt_root /
    "1QS_GT_SELECTION_V2_ACTIVE.csv"
)

val_manifest = pd.read_csv(
    val_root /
    "1QS_VALIDATION_CORE_MANIFEST.csv"
)

# ============================================================
# 55 TRAIN
# ============================================================

train = active[
    active["split"] == "TRAIN"
].copy()

if len(train) != 55:

    raise RuntimeError(
        f"TRAIN deveria conter 55 linhas; "
        f"encontrei {len(train)}."
    )

train_paths = []

for line_id in train[
    "htr_line_id"
].astype(str):

    image = (
        train_images /
        f"{line_id}.png"
    )

    gt = (
        train_images /
        f"{line_id}.gt.txt"
    )

    if not image.exists():

        raise FileNotFoundError(
            image
        )

    if not gt.exists():

        raise FileNotFoundError(
            gt
        )

    if not gt.read_text(
        encoding="utf-8"
    ).strip():

        raise RuntimeError(
            f"GT vazio: {line_id}"
        )

    train_paths.append(
        str(image)
    )

# ============================================================
# 4 VALIDATION CORE
# ============================================================

if len(val_manifest) != 4:

    raise RuntimeError(
        f"VALIDATION CORE deveria conter 4; "
        f"encontrei {len(val_manifest)}."
    )

val_paths = []

for line_id in val_manifest[
    "htr_line_id"
].astype(str):

    image = (
        val_images /
        f"{line_id}.png"
    )

    gt = (
        val_images /
        f"{line_id}.gt.txt"
    )

    if not image.exists():

        raise FileNotFoundError(
            image
        )

    if not gt.exists():

        raise FileNotFoundError(
            gt
        )

    if not gt.read_text(
        encoding="utf-8"
    ).strip():

        raise RuntimeError(
            f"GT VALIDATION vazio: {line_id}"
        )

    val_paths.append(
        str(image)
    )

# ============================================================
# ESCREVE MANIFESTS
# ============================================================

train_lst = (
    data_root /
    "1QS_TRAIN_55.lst"
)

val_lst = (
    data_root /
    "1QS_VALIDATION_CORE_4.lst"
)

train_lst.write_text(
    "\n".join(train_paths) + "\n",
    encoding="utf-8"
)

val_lst.write_text(
    "\n".join(val_paths) + "\n",
    encoding="utf-8"
)

# ============================================================
# HASHES
# ============================================================

def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()


print("\nTRAIN manifest:")
print(train_lst)
print("SHA-256:", sha256(train_lst))

print("\nVALIDATION manifest:")
print(val_lst)
print("SHA-256:", sha256(val_lst))

# ============================================================
# DOWNLOAD DO MODELO PP-OCRv6 SMALL
# ============================================================

model_path = (
    baseline_root /
    "ppocrv6_small_base.safetensors"
)

model_url = (
    "https://huggingface.co/"
    "small-models-for-glam/"
    "kraken-ppocrv6-small/"
    "resolve/main/small.safetensors"
)

if not model_path.exists():

    print(
        "\nBaixando PP-OCRv6 SMALL..."
    )

    urllib.request.urlretrieve(
        model_url,
        model_path
    )

else:

    print(
        "\nModelo-base já existe."
    )

print(
    "Modelo:",
    model_path
)

print(
    "Tamanho:",
    round(
        model_path.stat().st_size /
        1024 /
        1024,
        2
    ),
    "MB"
)

print(
    "SHA-256:",
    sha256(model_path)
)

# ============================================================
# BASELINE ZERO-SHOT
# Nenhuma linha TRAIN é usada aqui.
# ============================================================

output_file = (
    baseline_root /
    "1QS_PPocrV6_SMALL_BASELINE_VALIDATION.txt"
)

cmd = [
    ketos,
    "test",

    "-m",
    str(model_path),

    "-e",
    str(val_lst),

    "-f",
    "path",

    "--base-dir",
    "R",

    "-u",
    "NFD",

    "-n"
]

print("\n" + "=" * 72)
print("BASELINE ZERO-SHOT — VALIDATION CORE")
print("=" * 72)

print("\nExecutando:")
print(" ".join(cmd))
print()

result = subprocess.run(
    cmd,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

output_file.write_text(
    result.stdout,
    encoding="utf-8"
)

print(result.stdout)

print("\nExit code:", result.returncode)

if result.returncode != 0:

    raise RuntimeError(
        "O teste do modelo-base falhou. "
        "Não iniciar treinamento ainda."
    )

print("\n" + "=" * 72)
print("✅ BASELINE PRÉ-TREINAMENTO CONCLUÍDA")
print("=" * 72)

print("TRAIN:", len(train_paths))
print(
    "VALIDATION CORE:",
    len(val_paths)
)

print(
    "Nenhuma linha TEST utilizada."
)

print(
    "Nenhum peso foi treinado ou alterado."
)

print("\nResultado salvo em:")
print(output_file)

In [ ]:
# ============================================================
# AMBIENTE PARA TREINAMENTO — GPU + KRAKEN
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import sys
import shutil
import subprocess
from pathlib import Path

# ============================================================
# VERIFICA GPU
# ============================================================

print("=" * 70)
print("VERIFICAÇÃO DO AMBIENTE")
print("=" * 70)

gpu = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)

if gpu.returncode != 0:

    raise RuntimeError(
        "GPU não detectada. "
        "Altere o runtime do Colab para GPU antes de continuar."
    )

print("✅ GPU detectada")

print(
    "\n",
    gpu.stdout.splitlines()[0]
)

# ============================================================
# INSTALA PIPX
# ============================================================

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "pipx"
    ],
    check=True
)

# ============================================================
# INSTALA KRAKEN ISOLADAMENTE
# ============================================================

subprocess.run(
    [
        sys.executable,
        "-m",
        "pipx",
        "install",
        "--force",
        "--python",
        sys.executable,
        "kraken==7.1.1"
    ],
    check=True
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pipx",
        "inject",
        "--force",
        "kraken",
        "safetensors==0.8.0"
    ],
    check=True
)

# ============================================================
# PATH
# ============================================================

pipx_bin = (
    Path.home() /
    ".local" /
    "bin"
)

os.environ["PATH"] = (
    str(pipx_bin)
    +
    os.pathsep
    +
    os.environ.get("PATH", "")
)

ketos = shutil.which("ketos")

if ketos is None:

    raise RuntimeError(
        "KETOS não encontrado."
    )

print("\n✅ KETOS:", ketos)

# ============================================================
# CONFERE OS ARQUIVOS PERSISTENTES
# ============================================================

base = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS"
)

train_lst = (
    base /
    "htr_model/datasets/1QS_TRAIN_55.lst"
)

val_lst = (
    base /
    "htr_model/datasets/1QS_VALIDATION_CORE_4.lst"
)

model_base = (
    base /
    "htr_model/00_pretrained_baseline/"
    "ppocrv6_small_base.safetensors"
)

for p in [
    train_lst,
    val_lst,
    model_base
]:

    if not p.exists():

        raise FileNotFoundError(p)

    print("✅", p.name)

print("\n" + "=" * 70)
print("✅ AMBIENTE PRONTO PARA FINE-TUNING")
print("=" * 70)

In [ ]:
from pathlib import Path
from google.colab import drive

if Path("/content/drive/MyDrive").exists():
    print("✅ Google Drive já está montado.")
else:
    print("Montando Google Drive...")
    drive.mount("/content/drive")
    print("✅ Google Drive montado.")

In [ ]:
!nvidia-smi

In [ ]:
from pathlib import Path

base = Path("/content/drive/MyDrive/Qumran_Analytics/1QS")

arquivos = [
    base / "htr_model/datasets/1QS_TRAIN_55.lst",
    base / "htr_model/datasets/1QS_VALIDATION_CORE_4.lst",
    base / "htr_model/00_pretrained_baseline/ppocrv6_small_base.safetensors",
]

print("=" * 60)
print("CHECKPOINT PRÉ-TREINAMENTO")
print("=" * 60)

for p in arquivos:
    print(
        "✅ EXISTE" if p.exists() else "❌ AUSENTE",
        p
    )

In [ ]:
# ============================================================
# 1QS — PREPARAÇÃO DA SESSÃO GPU PARA FT01
# ============================================================

import os
import sys
import shutil
import subprocess
from pathlib import Path

print("=" * 72)
print("1QS — PREPARANDO AMBIENTE DE TREINAMENTO")
print("=" * 72)

# ============================================================
# 1. GPU
# ============================================================

gpu = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)

if gpu.returncode != 0:
    raise RuntimeError(
        "❌ GPU não detectada. "
        "Confirme que o runtime está configurado para GPU."
    )

print("\n✅ GPU detectada")

# mostra apenas algumas linhas do nvidia-smi
print(
    "\n".join(
        gpu.stdout.splitlines()[:10]
    )
)

# ============================================================
# 2. PIPX
# ============================================================

print("\n[1/3] Instalando pipx...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "pipx"
    ],
    check=True
)

# ============================================================
# 3. KRAKEN 7.1.1 ISOLADO
# ============================================================

print("[2/3] Instalando Kraken 7.1.1...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pipx",
        "install",
        "--force",
        "--python",
        sys.executable,
        "kraken==7.1.1"
    ],
    check=True
)

print("[3/3] Fixando safetensors 0.8.0...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pipx",
        "inject",
        "--force",
        "kraken",
        "safetensors==0.8.0"
    ],
    check=True
)

# ============================================================
# 4. PATH
# ============================================================

pipx_bin = (
    Path.home()
    / ".local"
    / "bin"
)

os.environ["PATH"] = (
    str(pipx_bin)
    + os.pathsep
    + os.environ.get("PATH", "")
)

ketos = shutil.which("ketos")

if ketos is None:
    raise RuntimeError(
        "❌ KETOS não encontrado após instalação."
    )

print("\n✅ KETOS:", ketos)

# ============================================================
# 5. TESTA KETOS
# ============================================================

help_result = subprocess.run(
    [
        ketos,
        "train",
        "--help"
    ],
    capture_output=True,
    text=True
)

if help_result.returncode != 0:
    print(help_result.stderr)
    raise RuntimeError(
        "❌ ketos train não iniciou corretamente."
    )

# ============================================================
# 6. CONFERE DADOS
# ============================================================

base = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS"
)

required = [
    base / "htr_model/datasets/1QS_TRAIN_55.lst",
    base / "htr_model/datasets/1QS_VALIDATION_CORE_4.lst",
    base / (
        "htr_model/00_pretrained_baseline/"
        "ppocrv6_small_base.safetensors"
    ),
]

print("\nArquivos:")

for p in required:

    if not p.exists():
        raise FileNotFoundError(p)

    print("✅", p.name)

# ============================================================
# 7. VERIFICA SUPORTE CUDA DO PYTORCH
# ============================================================

torch_check = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import torch; "
            "print(torch.__version__); "
            "print(torch.cuda.is_available()); "
            "print(torch.cuda.get_device_name(0) "
            "if torch.cuda.is_available() else 'NO_GPU')"
        )
    ],
    capture_output=True,
    text=True
)

print("\nPyTorch:")
print(torch_check.stdout)

if "True" not in torch_check.stdout:
    raise RuntimeError(
        "❌ PyTorch não está reconhecendo CUDA."
    )

print("=" * 72)
print("✅ AMBIENTE PRONTO PARA FT01")
print("=" * 72)

In [ ]:

from google.colab import drive
from pathlib import Path
import os

# Conectar ao Google Drive
drive.mount("/content/drive")

print("\n=== DIAGNÓSTICO DO PROJETO QUMRAN ===")

termos = [
    "1qs", "qumran", "kraken", "ft01",
    "etcbc", "batch06", "valid", "roi"
]

bases = [
    Path("/content"),
    Path("/content/drive/MyDrive")
]

encontrados = []

for base in bases:
    if not base.exists():
        print(f"\nPasta indisponível: {base}")
        continue

    print(f"\nPesquisando em: {base}")

    for pasta, diretorios, arquivos in os.walk(base):
        # Evita pesquisar duas vezes no Drive
        if base == Path("/content"):
            diretorios[:] = [
                d for d in diretorios
                if d not in ("drive", ".config", "sample_data")
            ]

        for arquivo in arquivos:
            nome = arquivo.lower()

            if any(termo in nome for termo in termos):
                encontrados.append(
                    os.path.join(pasta, arquivo)
                )

print("\n=== ARQUIVOS ENCONTRADOS ===")
print("Total:", len(encontrados))

for caminho in encontrados[:100]:
    print(caminho)

if len(encontrados) > 100:
    print(f"\nMais {len(encontrados)-100} arquivos omitidos.")

if not encontrados:
    print("\nNenhum arquivo correspondente localizado.")
    print("Verifique se estão em outro Drive ou ZIP.")


In [ ]:

from pathlib import Path

print("=== ARQUIVOS PRIORITÁRIOS DO EXPERIMENTO ===")

categorias = {
    "FT01": ["ft01", "fine_tuning", "finetune"],
    "1QS": ["1qs"],
    "KRAKEN": ["kraken"],
    "VALIDAÇÃO": ["validation", "validated", "ground_truth"],
    "ETCBC": ["etcbc", "etcb"],
}

for categoria, termos in categorias.items():
    arquivos = [
        p for p in encontrados
        if any(t in str(p).lower() for t in termos)
    ]

    print(f"\n{'='*65}")
    print(f"{categoria}: {len(arquivos)} arquivos")

    # Priorizar resultados, modelos e métricas
    prioridades = [
        "result", "metric", "best", "model",
        "checkpoint", "line", "roi", "log"
    ]

    arquivos.sort(
        key=lambda p: (
            -sum(t in Path(p).name.lower() for t in prioridades),
            len(str(p))
        )
    )

    for caminho in arquivos[:15]:
        print(caminho)

    if len(arquivos) > 15:
        print(f"... mais {len(arquivos)-15} arquivos")


In [ ]:

from pathlib import Path
from collections import deque
import json

BASE = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS/htr_model"
)

FT01 = BASE / "01_finetune_FT01"

print("=" * 65)
print("DIAGNÓSTICO CIENTÍFICO — EXPERIMENTO FT01")
print("=" * 65)

# 1. Verificar a pasta
if not FT01.exists():
    print("ERRO: Pasta FT01 não encontrada.")
else:
    print("\n[1] ARQUIVOS DO EXPERIMENTO")

    arquivos = sorted(
        [p for p in FT01.rglob("*") if p.is_file()]
    )

    print(f"Total de arquivos: {len(arquivos)}")

    for p in arquivos[:80]:
        tamanho = p.stat().st_size / (1024**2)
        print(
            f"{p.relative_to(FT01)} | "
            f"{tamanho:.2f} MB"
        )

    # 2. Localizar modelos e checkpoints
    print("\n[2] MODELOS E CHECKPOINTS")

    extensoes = {
        ".mlmodel", ".pt", ".pth", ".ckpt",
        ".onnx", ".safetensors", ".pdparams"
    }

    modelos = [
        p for p in arquivos
        if p.suffix.lower() in extensoes
    ]

    if modelos:
        for p in modelos:
            print(
                p.relative_to(FT01),
                f"| {p.stat().st_size / 1024**2:.2f} MB"
            )
    else:
        print("Nenhum modelo encontrado nesta pasta.")

    # 3. Configuração
    print("\n[3] CONFIGURAÇÃO DO FT01")

    config = FT01 / "FT01_CONFIG.json"

    if config.exists():
        with open(config, encoding="utf-8") as f:
            dados = json.load(f)

        print(json.dumps(
            dados,
            indent=2,
            ensure_ascii=False
        )[:6000])
    else:
        print("Configuração não encontrada.")

    # 4. Comando original
    print("\n[4] COMANDO DE TREINAMENTO")

    comando = FT01 / "FT01_COMMAND.txt"

    if comando.exists():
        print(comando.read_text(
            encoding="utf-8",
            errors="replace"
        )[:4000])
    else:
        print("Comando não encontrado.")

    # 5. Últimas linhas do treinamento
    print("\n[5] FINAL DO LOG DE TREINAMENTO")

    log = FT01 / "FT01_TRAINING_LOG.txt"

    if log.exists():
        with open(
            log,
            encoding="utf-8",
            errors="replace"
        ) as f:
            ultimas = deque(f, maxlen=45)

        for linha in ultimas:
            print(linha.rstrip())
    else:
        print("Log não encontrado.")

print("\n=== DIAGNÓSTICO FINALIZADO ===")


In [ ]:

%pip install "kraken>=7.1,<8"


In [ ]:

import sys
import shutil
import subprocess
from importlib.metadata import version
from pathlib import Path

print("=== VERIFICAÇÃO DO AMBIENTE FT01 ===")

print("\nPython:", sys.version.split()[0])
print("Kraken:", version("kraken"))

ketos = shutil.which("ketos")

if ketos is None:
    candidatos = [
        Path(sys.executable).parent / "ketos",
        Path("/usr/local/bin/ketos"),
        Path("/root/.local/bin/ketos")
    ]

    ketos = next(
        (str(p) for p in candidatos if p.is_file()),
        None
    )

print("Executável ketos:", ketos)

if ketos:
    teste = subprocess.run(
        [ketos, "train", "--help"],
        capture_output=True,
        text=True
    )

    print("Código de saída:", teste.returncode)

    if teste.returncode == 0:
        print("\nKETOS DISPONÍVEL PARA TREINAMENTO")
        print(
            "Opção --resize:",
            "--resize" in teste.stdout
        )
        print(
            "Opção --warmup:",
            "--warmup" in teste.stdout
        )
    else:
        print(teste.stderr[-2000:])
else:
    print("KETOS AINDA NÃO FOI LOCALIZADO")


In [ ]:

import torch
import subprocess

print("=== DIAGNÓSTICO GPU — FT01 ===")

resultado = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
) if __import__("shutil").which("nvidia-smi") else None

if resultado and resultado.returncode == 0:
    print(resultado.stdout[:3000])
else:
    print("GPU NVIDIA não detectada pelo sistema.")

print("\nPyTorch:", torch.__version__)
print("CUDA do PyTorch:", torch.version.cuda)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("\nAMBIENTE GPU PRONTO PARA FT01")
else:
    print("\nGPU ainda indisponível para PyTorch.")


In [ ]:

import sys
import subprocess
import shutil
import torch

print("=" * 60)
print("QUMRAN ANALYTICS — RESTAURAÇÃO DO AMBIENTE")
print("=" * 60)

# Instalar versão utilizada no experimento
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "kraken==7.1.1"
    ],
    check=True
)

# Verificar executável
ketos = shutil.which("ketos")

if ketos is None:
    print("\nERRO: ketos não encontrado.")
else:
    print("\nKraken instalado com sucesso!")
    print("Executável:", ketos)

    resultado = subprocess.run(
        [ketos, "train", "--help"],
        capture_output=True,
        text=True
    )

    print("Comando de treinamento:",
          "OK" if resultado.returncode == 0 else "ERRO")

# Verificar GPU
print("\nCUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nDIAGNÓSTICO FINALIZADO")


In [ ]:

from google.colab import drive
from pathlib import Path
import torch
import shutil
import shlex
import json

# Conectar Drive, se necessário
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

BASE = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS/htr_model"
)

ORIGINAL = BASE / "01_finetune_FT01"
SAIDA = BASE / "01_finetune_FT01_R01"

print("=" * 65)
print("QUMRAN ANALYTICS — PRÉ-VALIDAÇÃO FT01_R01")
print("=" * 65)

# Verificação do ambiente
assert torch.cuda.is_available(), "GPU não disponível"

ketos = shutil.which("ketos")
assert ketos, "Kraken não instalado nesta sessão"

print("GPU:", torch.cuda.get_device_name(0))
print("Kraken:", ketos)

# Recuperar configuração original
config = json.loads(
    (ORIGINAL / "FT01_CONFIG.json").read_text()
)

# Recuperar comando original
comando = shlex.split(
    (ORIGINAL / "FT01_COMMAND.txt").read_text()
)

# Remover argumento incompatível
if "--freeze-backbone" in comando:
    idx = comando.index("--freeze-backbone")
    del comando[idx:idx + 2]

# Atualizar executável e destino
comando[0] = ketos

idx = comando.index("-o")
comando[idx + 1] = str(SAIDA)

# Verificar arquivos essenciais
for chave in [
    "base_model",
    "train_manifest",
    "validation_manifest"
]:
    arquivo = Path(config[chave])
    assert arquivo.is_file(), f"Ausente: {arquivo}"
    print("OK:", arquivo.name)

def ler_manifest(caminho):
    return [
        s.strip()
        for s in Path(caminho).read_text().splitlines()
        if s.strip() and not s.startswith("#")
    ]

treino = ler_manifest(config["train_manifest"])
validacao = ler_manifest(config["validation_manifest"])

assert len(treino) == 55
assert len(validacao) == 4
assert not set(treino).intersection(validacao)

for nome in treino + validacao:
    imagem = Path(nome)
    assert imagem.is_file(), f"Imagem ausente: {imagem}"
    gt = imagem.with_suffix(".gt.txt")
    assert gt.is_file(), f"Ground truth ausente: {gt}"
    assert gt.stat().st_size > 0

assert not SAIDA.exists(), (
    "FT01_R01 já existe. Verifique antes de sobrescrever."
)

print("\nTREINAMENTO:", len(treino), "linhas")
print("VALIDAÇÃO:", len(validacao), "linhas")
print("ÉPOCAS:", config["epochs"])
print("BATCH:", config["batch_size"])
print("LEARNING RATE:", config["learning_rate"])

print("\nCOMANDO CORRIGIDO:")
print(shlex.join(comando))

PRONTO = True
print("\nPRÉ-VALIDAÇÃO CONCLUÍDA")


In [ ]:

%pip install --upgrade "safetensors==0.8.0"


In [ ]:

import sys
import subprocess
from importlib.metadata import version

print("=" * 60)
print("QUMRAN — VALIDAÇÃO DAS DEPENDÊNCIAS")
print("=" * 60)

for pacote in [
    "kraken", "safetensors",
    "transformers", "lightning", "torch"
]:
    try:
        print(f"{pacote}: {version(pacote)}")
    except Exception:
        print(f"{pacote}: NÃO ENCONTRADO")

codigo = """
import torch
from lightning.pytorch.callbacks import ModelCheckpoint
from kraken.train import KrakenTrainer
assert torch.cuda.is_available()
print("IMPORTAÇÕES E CUDA: OK")
"""

teste = subprocess.run(
    [sys.executable, "-c", codigo],
    capture_output=True,
    text=True
)

if teste.returncode == 0:
    print("\n" + teste.stdout)
    print("AMBIENTE PRONTO PARA RETOMADA")
else:
    print("\nERRO DE DEPENDÊNCIA:")
    print(teste.stderr[-5000:])


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import subprocess
import shutil
import shlex
import json
import torch

BASE = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS/htr_model"
)

ORIGINAL = BASE / "01_finetune_FT01"
SAIDA = BASE / "01_finetune_FT01_R02"

assert torch.cuda.is_available(), "GPU indisponível"
assert shutil.which("ketos"), "Kraken indisponível"
assert not SAIDA.exists(), "FT01_R02 já existe"

config = json.loads(
    (ORIGINAL / "FT01_CONFIG.json").read_text()
)

comando = shlex.split(
    (ORIGINAL / "FT01_COMMAND.txt").read_text()
)

# Corrigir opção incompatível
if "--freeze-backbone" in comando:
    i = comando.index("--freeze-backbone")
    del comando[i:i+2]

comando[0] = shutil.which("ketos")
comando[comando.index("-o")+1] = str(SAIDA)

# Validar arquivos essenciais
for chave in [
    "base_model",
    "train_manifest",
    "validation_manifest"
]:
    assert Path(config[chave]).is_file(), (
        f"Arquivo ausente: {config[chave]}"
    )

# Registrar nova execução
SAIDA.mkdir(parents=True, exist_ok=False)

config["experiment"] = "FT01_R02"
config["previous_attempt"] = "FT01_R01"
config["freeze_backbone_samples"] = None
config["safetensors_version"] = version("safetensors")
config["kraken_version"] = version("kraken")
config["timestamp_utc"] = (
    datetime.now(timezone.utc).isoformat()
)

(SAIDA / "FT01_R02_CONFIG.json").write_text(
    json.dumps(config, indent=2, ensure_ascii=False)
)

(SAIDA / "FT01_R02_COMMAND.txt").write_text(
    shlex.join(comando)
)

LOG = SAIDA / "FT01_R02_TRAINING_LOG.txt"

print("=" * 65)
print("INICIANDO QUMRAN FT01_R02")
print("=" * 65)
print("GPU:", torch.cuda.get_device_name(0))
print("Treinamento: 55 linhas")
print("Validação: 4 linhas")
print("Épocas: 30")
print("Safetensors:", version("safetensors"))
print("=" * 65, flush=True)

with open(LOG, "w", encoding="utf-8") as arquivo:
    processo = subprocess.Popen(
        comando,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors="replace",
        bufsize=1
    )

    for linha in processo.stdout:
        print(linha, end="", flush=True)
        arquivo.write(linha)
        arquivo.flush()

    codigo = processo.wait()

print("\nCódigo de saída:", codigo)

if codigo == 0:
    print("FT01_R02 FINALIZADO COM SUCESSO")
else:
    print("FT01_R02 INTERROMPIDO")
    print("Log preservado em:", LOG)


In [ ]:

from pathlib import Path
import hashlib

BASE = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS/htr_model"
)

MODELO_BASE = (
    BASE / "00_pretrained_baseline" /
    "ppocrv6_small_base.safetensors"
)

FT02 = BASE / "01_finetune_FT01_R02"
DATASETS = BASE / "datasets"

print("=" * 65)
print("QUMRAN ANALYTICS — PREPARAÇÃO DO TESTE")
print("=" * 65)

print("\n[1] MODELO ORIGINAL")
print("Existe:", MODELO_BASE.is_file())

print("\n[2] MODELO AJUSTADO")

modelos = sorted(FT02.glob("best_*.safetensors"))

for p in modelos:
    print("Modelo:", p.name)
    print("Tamanho:", round(p.stat().st_size/1024**2, 2), "MB")
    print("SHA256:", hashlib.sha256(p.read_bytes()).hexdigest())

print("\n[3] CHECKPOINTS")

for p in sorted(FT02.glob("*.ckpt")):
    print(p.name)

print("\n[4] MANIFESTS DISPONÍVEIS")

if DATASETS.exists():
    for p in sorted(DATASETS.rglob("*.lst")):
        linhas = [
            x.strip()
            for x in p.read_text(
                encoding="utf-8"
            ).splitlines()
            if x.strip() and not x.startswith("#")
        ]
        print(f"{p.name}: {len(linhas)} entradas")
else:
    print("Pasta datasets não encontrada")

print("\n[5] CONJUNTOS DE TESTE CANDIDATOS")

for p in sorted(BASE.rglob("*")):
    if p.is_file() and p.suffix in (".lst", ".csv", ".json"):
        nome = p.name.lower()
        if any(t in nome for t in [
            "test", "sealed", "unseen", "holdout"
        ]):
            print(p)

print("\nDIAGNÓSTICO CONCLUÍDO")


In [ ]:

from pathlib import Path
from collections import Counter
import csv

ROOT = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS"
)
DATASETS = ROOT / "htr_model" / "datasets"

assert ROOT.is_dir(), "Pasta principal não encontrada"

print("=" * 68)
print("QUMRAN ANALYTICS — AUDITORIA DOS DATASETS")
print("=" * 68)

todos = sorted(p for p in ROOT.rglob("*") if p.is_file())

print("\n[1] INVENTÁRIO GERAL")
print("Total de arquivos:", len(todos))

pastas = [
    "htr_ground_truth",
    "htr_ground_truth_v2",
    "final_segmentation",
    "htr_model/datasets"
]

for nome in pastas:
    pasta = ROOT / nome
    arquivos = (
        [p for p in pasta.rglob("*") if p.is_file()]
        if pasta.is_dir() else []
    )

    print(f"\n{nome}: {len(arquivos)} arquivos")

    tipos = Counter(
        ".gt.txt" if p.name.endswith(".gt.txt")
        else p.suffix.lower()
        for p in arquivos
    )
    print("Formatos:", dict(tipos))

    for p in arquivos[:8]:
        print("  ", p.relative_to(ROOT))

print("\n[2] TODOS OS MANIFESTS")

manifests = sorted(
    p for p in todos if p.suffix.lower() == ".lst"
)

for p in manifests:
    linhas = [
        x.strip()
        for x in p.read_text(
            encoding="utf-8"
        ).splitlines()
        if x.strip() and not x.startswith("#")
    ]
    print(f"{p.relative_to(ROOT)}: {len(linhas)} entradas")

print("\n[3] GROUND TRUTH NO FORMATO KRAKEN")

gt_files = [
    p for p in todos if p.name.endswith(".gt.txt")
]

print("Transcrições .gt.txt:", len(gt_files))

extensoes = [".png", ".jpg", ".jpeg", ".tif", ".tiff"]

pares = []
for gt in gt_files:
    prefixo = gt.name[:-len(".gt.txt")]
    for ext in extensoes:
        img = gt.with_name(prefixo + ext)
        if img.is_file():
            pares.append((img, gt))
            break

print("Pares imagem + transcrição:", len(pares))

def carregar_lista(nome):
    p = DATASETS / nome
    if not p.exists():
        return []
    return [
        Path(x.strip())
        for x in p.read_text().splitlines()
        if x.strip() and not x.startswith("#")
    ]

treino = carregar_lista("1QS_TRAIN_55.lst")
validacao = carregar_lista("1QS_VALIDATION_CORE_4.lst")

usados = {
    str(p.resolve())
    for p in treino + validacao
}

nao_alocados = [
    (img, gt)
    for img, gt in pares
    if str(img.resolve()) not in usados
]

print("Imagens nos manifests:", len(usados))
print("Pares fora dos manifests:", len(nao_alocados))
print("Observação: fora dos manifests NÃO significa teste.")

print("\n[4] ORIGEM DAS IMAGENS DE TREINAMENTO")

for nome, grupo in [
    ("Treino", treino),
    ("Validação", validacao)
]:
    contagem = Counter(
        str(p.parent) for p in grupo
    )
    print(f"\n{nome}:")
    for pasta, n in contagem.items():
        print(f"  {n} arquivos — {pasta}")

print("\n[5] DOCUMENTOS DE SELEÇÃO E AUDITORIA")

arquivos_csv = [
    ROOT / "final_segmentation" /
        "1QS_FINAL_LINE_MANIFEST.csv",
    ROOT / "htr_ground_truth" /
        "1QS_GT_SELECTION.csv",
    ROOT / "htr_ground_truth_v2" /
        "1QS_BATCH01_GT_AUDIT_LOG.csv"
]

for p in arquivos_csv:
    if not p.is_file():
        print("Não encontrado:", p.name)
        continue

    with p.open(
        encoding="utf-8-sig",
        newline="",
        errors="replace"
    ) as f:
        cabecalho = next(csv.reader(f), [])

    print(f"\n{p.name}")
    print("Colunas:", cabecalho)

print("\n[6] POSSÍVEIS REGISTROS DE TESTE")

termos = [
    "test", "sealed", "unseen",
    "holdout", "split"
]

candidatos = [
    p for p in todos
    if any(t in p.name.lower() for t in termos)
]

print("Arquivos com esses termos:", len(candidatos))

for p in candidatos[:30]:
    print(p.relative_to(ROOT))

print("\n" + "=" * 68)
print("AUDITORIA CONCLUÍDA — NENHUM ARQUIVO MODIFICADO")
print("=" * 68)


In [ ]:

from pathlib import Path
import hashlib
import json
import csv
import re

ROOT = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS"
)

GT = ROOT / "htr_ground_truth_v2"
DATASETS = ROOT / "htr_model" / "datasets"

TESTE = GT / "1QS_BATCH07_TEST_LOCKED.csv"
HASH = GT / "1QS_BATCH07_TEST_LOCKED.sha256.txt"
META = GT / "1QS_BATCH07_TEST_LOCK_METADATA.json"

print("=" * 65)
print("QUMRAN ANALYTICS — AUDITORIA DO TESTE SELADO")
print("=" * 65)

# 1. Conferir existência
for p in [TESTE, HASH, META]:
    assert p.is_file(), f"Arquivo ausente: {p}"
    print("OK:", p.name)

# 2. Validar integridade
sha_atual = hashlib.sha256(TESTE.read_bytes()).hexdigest()

texto_hash = HASH.read_text(
    encoding="utf-8",
    errors="replace"
)

hashes_registrados = re.findall(
    r"\b[a-fA-F0-9]{64}\b",
    texto_hash
)

print("\n[1] INTEGRIDADE SHA-256")
print("SHA atual:", sha_atual)
print("Hashes registrados:", len(hashes_registrados))

hash_ok = sha_atual.lower() in [
    h.lower() for h in hashes_registrados
]

if hash_ok:
    print("RESULTADO: INTEGRIDADE CONFIRMADA")
else:
    print("RESULTADO: HASH NÃO CONFIRMADO")
    print("Verificar escopo do registro antes de usar o teste.")

# 3. Examinar estrutura, sem revelar transcrições
with TESTE.open(
    encoding="utf-8-sig",
    newline=""
) as f:
    leitor = csv.DictReader(f)
    colunas = leitor.fieldnames or []

    campos_id = [
        c for c in colunas
        if c.lower() in {
            "htr_line_id", "line_id",
            "image_path", "crop_path",
            "frozen_crop_path",
            "image", "filename",
            "path", "file"
        }
    ]

    total = 0
    identificadores_teste = set()

    def normalizar(valor):
        nome = Path(str(valor).strip()).name
        if nome.endswith(".gt.txt"):
            nome = nome[:-7]
        else:
            nome = Path(nome).stem
        return nome.upper()

    for linha in leitor:
        total += 1
        for campo in campos_id:
            valor = linha.get(campo)
            if valor and valor.strip():
                identificadores_teste.add(
                    normalizar(valor)
                )

print("\n[2] ESTRUTURA DO TESTE")
print("Registros:", total)
print("Colunas:", colunas)
print("Campos de identificação:", campos_id)

# 4. Ler metadados, sem revelar conteúdo textual
metadados = json.loads(
    META.read_text(encoding="utf-8")
)

print("\n[3] METADADOS")
print("Tipo:", type(metadados).__name__)

if isinstance(metadados, dict):
    print("Chaves:", list(metadados.keys()))

    for chave, valor in metadados.items():
        if isinstance(valor, (bool, int, float)):
            print(f"{chave}: {valor}")

# 5. Verificar possíveis sobreposições
def ler_ids(nome):
    p = DATASETS / nome
    return {
        normalizar(linha)
        for linha in p.read_text(
            encoding="utf-8"
        ).splitlines()
        if linha.strip() and not linha.startswith("#")
    }

ids_treino = ler_ids("1QS_TRAIN_55.lst")
ids_validacao = ler_ids("1QS_VALIDATION_CORE_4.lst")

print("\n[4] VERIFICAÇÃO DE SOBREPOSIÇÃO")
print(
    "Treino x validação:",
    len(ids_treino & ids_validacao)
)
print(
    "Teste x treino:",
    len(identificadores_teste & ids_treino)
)
print(
    "Teste x validação:",
    len(identificadores_teste & ids_validacao)
)

print("\n[5] CONCLUSÃO PRELIMINAR")

if not hash_ok:
    print("BLOQUEADO: integridade ainda não confirmada.")
elif not campos_id:
    print("Necessário identificar a chave das amostras.")
elif (identificadores_teste & (ids_treino | ids_validacao)):
    print("ATENÇÃO: possíveis amostras compartilhadas.")
else:
    print("Sem sobreposição identificada pelos campos disponíveis.")
    print("Ainda é necessário verificar independência por origem.")

print("\nAUDITORIA FINALIZADA — NENHUM ARQUIVO ALTERADO")


In [ ]:

from pathlib import Path
from collections import Counter
import csv
import json
import hashlib

ROOT = Path("/content/drive/MyDrive/Qumran_Analytics/1QS")
GT = ROOT / "htr_ground_truth_v2"
DS = ROOT / "htr_model" / "datasets"

TEST = GT / "1QS_BATCH07_TEST_LOCKED.csv"
META = GT / "1QS_BATCH07_TEST_LOCK_METADATA.json"

def sha256(arquivo):
    h = hashlib.sha256()
    with open(arquivo, "rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

def ler_lista(nome):
    return [
        Path(x.strip())
        for x in (DS / nome).read_text().splitlines()
        if x.strip() and not x.startswith("#")
    ]

def normalizar_id(valor):
    return Path(str(valor).strip()).stem.upper()

def localizar_imagens(linha):
    candidatos = []

    for campo in [
        "htr_crop_v2_path",
        "frozen_crop_path",
        "crop_path"
    ]:
        valor = (linha.get(campo) or "").strip()
        if not valor:
            continue

        p = Path(valor)
        opcoes = [p] if p.is_absolute() else [
            ROOT / p,
            GT / p,
            GT / "images" / p,
            ROOT / "final_segmentation" / p
        ]

        # Recuperar caminhos absolutos de sessões antigas
        if "Qumran_Analytics/1QS/" in valor:
            relativo = valor.split("Qumran_Analytics/1QS/", 1)[1]
            opcoes.append(ROOT / relativo)

        for opcao in opcoes:
            if opcao.is_file() and opcao not in candidatos:
                candidatos.append(opcao)

    return candidatos

treino = ler_lista("1QS_TRAIN_55.lst")
validacao = ler_lista("1QS_VALIDATION_CORE_4.lst")

with TEST.open(encoding="utf-8-sig", newline="") as f:
    linhas = list(csv.DictReader(f))

metadados = json.loads(META.read_text(encoding="utf-8"))

print("=" * 65)
print("QUMRAN — AUDITORIA FINAL DO TESTE INDEPENDENTE")
print("=" * 65)

# Integridade do manifesto
hash_atual = sha256(TEST)
hash_meta = metadados.get("manifest_sha256", "")

print("\n[1] MANIFESTO")
print("Linhas:", len(linhas))
print("SHA corresponde aos metadados:",
      hash_atual.lower() == str(hash_meta).lower())
print("Quantidade corresponde aos metadados:",
      len(linhas) == metadados.get("n_lines"))

# Distribuição por coluna
colunas = Counter(x.get("column", "") for x in linhas)

print("\n[2] DISTRIBUIÇÃO")
for coluna, n in sorted(colunas.items()):
    print(f"Coluna {coluna}: {n} linha(s)")

print("Uma linha por coluna:",
      len(colunas) == len(linhas)
      and all(n == 1 for n in colunas.values()))

# Hashes das imagens de treino e validação
hashes_usados = {
    sha256(p)
    for p in treino + validacao
    if p.is_file()
}

ids_usados = {
    normalizar_id(p.name)
    for p in treino + validacao
}

localizadas = 0
hash_confirmado = 0
hash_divergente = 0
hash_ausente = 0
duplicatas = 0
identidades_repetidas = 0
sem_imagem = []

print("\n[3] AUDITORIA DAS IMAGENS")

for numero, linha in enumerate(linhas, start=1):
    candidatos = localizar_imagens(linha)
    esperado = (linha.get("image_sha256") or "").lower().strip()

    if not candidatos:
        sem_imagem.append(numero)
        continue

    localizadas += 1

    hashes = {sha256(p) for p in candidatos}

    if esperado:
        if esperado in hashes:
            hash_confirmado += 1
        else:
            hash_divergente += 1
    else:
        hash_ausente += 1

    if hashes & hashes_usados:
        duplicatas += 1

    identidade = normalizar_id(
        linha.get("htr_line_id") or ""
    )
    if identidade and identidade in ids_usados:
        identidades_repetidas += 1

print("Imagens localizadas:", localizadas)
print("Hashes de imagem confirmados:", hash_confirmado)
print("Hashes divergentes:", hash_divergente)
print("Sem hash registrado:", hash_ausente)
print("Possíveis duplicações exatas:", duplicatas)
print("IDs HTR repetidos:", identidades_repetidas)
print("Registros sem imagem localizada:", sem_imagem)

# Verificação do estado de transcrição
print("\n[4] ESTADO DO GROUND TRUTH")

for campo in ["gt_text", "image_only_gt_text"]:
    preenchidos = sum(
        bool((linha.get(campo) or "").strip())
        for linha in linhas
    )
    print(f"{campo}: {preenchidos} registros preenchidos")

print("Ground truth declarado como criado:",
      metadados.get("ground_truth_created"))

print("\n[5] RESULTADO")

if (
    localizadas == len(linhas)
    and hash_confirmado == len(linhas)
    and duplicatas == 0
    and identidades_repetidas == 0
    and len(colunas) == len(linhas)
):
    print("Verificações básicas de integridade: OK")
    print("Ainda verificar independência pela origem textual.")
else:
    print("Há itens que precisam de investigação.")
    print("Não liberar a avaliação final ainda.")

print("\nNENHUM ARQUIVO FOI MODIFICADO")


In [ ]:

from pathlib import Path
from collections import defaultdict
import csv
import re

ROOT = Path("/content/drive/MyDrive/Qumran_Analytics/1QS")
GT = ROOT / "htr_ground_truth_v2"
DS = ROOT / "htr_model" / "datasets"

SELECAO = GT / "1QS_GT_SELECTION_V2_WORKING.csv"
TESTE = GT / "1QS_BATCH07_TEST_LOCKED.csv"

def ler_csv(p):
    with p.open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

def ler_manifest(nome):
    p = DS / nome
    return [
        Path(x.strip())
        for x in p.read_text().splitlines()
        if x.strip() and not x.startswith("#")
    ]

def chave(valor):
    nome = Path(str(valor or "").strip()).name
    if nome.endswith(".gt.txt"):
        nome = nome[:-7]
    else:
        nome = Path(nome).stem
    return nome.upper()

def identificadores(row):
    campos = [
        "htr_crop_v2_path",
        "frozen_crop_path",
        "crop_path",
        "htr_line_id",
        "secondary_id"
    ]
    return {
        chave(row.get(c))
        for c in campos
        if (row.get(c) or "").strip()
    }

def numero(valor):
    try:
        return float(str(valor).replace(",", "."))
    except (TypeError, ValueError):
        return None

def caixa(row):
    v = [numero(row.get(k)) for k in
         ("x1", "y1", "x2", "y2")]
    if any(x is None for x in v):
        return None
    x1, y1, x2, y2 = v
    if x2 <= x1 or y2 <= y1:
        return None
    return v

def sobreposicao(a, b):
    A, B = caixa(a), caixa(b)
    if A is None or B is None:
        return None
    x1 = max(A[0], B[0])
    y1 = max(A[1], B[1])
    x2 = min(A[2], B[2])
    y2 = min(A[3], B[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    area_a = (A[2]-A[0]) * (A[3]-A[1])
    area_b = (B[2]-B[0]) * (B[3]-B[1])
    return inter / min(area_a, area_b)

selecao = ler_csv(SELECAO)
teste = ler_csv(TESTE)

treino = ler_manifest("1QS_TRAIN_55.lst")
validacao = ler_manifest("1QS_VALIDATION_CORE_4.lst")
usados = treino + validacao

# Relacionar os recortes aos metadados
indice = defaultdict(list)

for i, linha in enumerate(selecao):
    for ident in identificadores(linha):
        indice[ident].append(i)

associacoes = []
nao_localizados = []
ambiguos = []

for imagem in usados:
    candidatos = sorted(set(indice.get(chave(imagem), [])))
    if len(candidatos) == 1:
        associacoes.append((imagem, selecao[candidatos[0]]))
    elif len(candidatos) == 0:
        nao_localizados.append(imagem.name)
    else:
        ambiguos.append((imagem.name, len(candidatos)))

print("=" * 68)
print("QUMRAN — AUDITORIA DE INDEPENDÊNCIA PALEOGRÁFICA")
print("=" * 68)

print("\n[1] RASTREABILIDADE")
print("Treino + validação:", len(usados))
print("Associações únicas:", len(associacoes))
print("Não localizados:", len(nao_localizados))
print("Ambíguos:", len(ambiguos))

if nao_localizados:
    print("Não localizados:", nao_localizados[:15])
if ambiguos:
    print("Ambíguos:", ambiguos[:15])

# Comparar linhas por metadados espaciais
suspeitas = []
comparacoes_sem_geometria = 0
comparacoes = 0

for t in teste:
    coluna = str(t.get("column", "")).strip().upper()
    linha_t = str(t.get("line_number", "")).strip()

    for imagem, r in associacoes:
        if str(r.get("column", "")).strip().upper() != coluna:
            continue

        comparacoes += 1
        motivos = []

        linha_r = str(r.get("line_number", "")).strip()
        if linha_t and linha_r and linha_t == linha_r:
            motivos.append("mesmo numero de linha")

        cobertura = sobreposicao(t, r)
        if cobertura is None:
            comparacoes_sem_geometria += 1
        elif cobertura >= 0.05:
            motivos.append(
                f"sobreposicao espacial {cobertura:.1%}"
            )

        if motivos:
            suspeitas.append({
                "teste": t.get("htr_line_id")
                         or t.get("line_id"),
                "treino_validacao": imagem.name,
                "motivos": motivos
            })

print("\n[2] INDEPENDÊNCIA ESPACIAL")
print("Comparações na mesma coluna:", comparacoes)
print("Sem geometria comparável:",
      comparacoes_sem_geometria)
print("Possíveis sobreposições:", len(suspeitas))

for s in suspeitas[:25]:
    print(s)

# Verificar duplicações internas do conjunto de teste
hashes = [
    str(r.get("image_sha256", "")).strip().lower()
    for r in teste
]

hashes_validos = [h for h in hashes if h]

print("\n[3] DUPLICAÇÕES INTERNAS")
print(
    "Hashes repetidos no teste:",
    len(hashes_validos) - len(set(hashes_validos))
)

print("\n[4] CONCLUSÃO")

if nao_localizados or ambiguos:
    print("PENDENTE: associação incompleta dos metadados.")
elif comparacoes_sem_geometria:
    print("PENDENTE: algumas comparações não têm geometria.")
elif suspeitas:
    print("REVISÃO NECESSÁRIA: possíveis linhas compartilhadas.")
elif len(hashes_validos) != len(teste):
    print("PENDENTE: imagens sem hash registrado.")
elif len(hashes_validos) != len(set(hashes_validos)):
    print("ATENÇÃO: há imagens duplicadas no teste.")
else:
    print("AUDITORIA ESPACIAL APROVADA")
    print("Nenhuma sobreposição detectada pelo critério adotado.")
    print("Independência textual ainda requer revisão documental.")

print("\nNenhum dado de referência textual foi utilizado.")
print("Nenhum arquivo foi modificado.")


In [ ]:

from pathlib import Path
from collections import defaultdict
import csv

ROOT = Path("/content/drive/MyDrive/Qumran_Analytics/1QS")
DS = ROOT / "htr_model" / "datasets"
TEST = ROOT / "htr_ground_truth_v2/1QS_BATCH07_TEST_LOCKED.csv"

def normalizar(valor):
    if valor is None:
        return ""
    nome = Path(str(valor).strip()).name.upper()
    for ext in (".GT.TXT", ".PNG", ".JPG", ".JPEG", ".TIF", ".TIFF"):
        if nome.endswith(ext):
            return nome[:-len(ext)]
    return nome

def ids_linha(row):
    campos = [
        "htr_line_id", "secondary_id", "line_id",
        "crop_path", "frozen_crop_path",
        "htr_crop_v2_path", "image_path",
        "replacement_for", "replaced_by"
    ]
    return {
        normalizar(row.get(c))
        for c in campos
        if row.get(c) and str(row[c]).strip()
    }

def ler_manifest(nome):
    p = DS / nome
    return [
        Path(s.strip())
        for s in p.read_text().splitlines()
        if s.strip() and not s.startswith("#")
    ]

treino = ler_manifest("1QS_TRAIN_55.lst")
validacao = ler_manifest("1QS_VALIDATION_CORE_4.lst")
amostras = treino + validacao

with TEST.open(encoding="utf-8-sig", newline="") as f:
    teste = list(csv.DictReader(f))

# Indexar metadados de todos os manifestos disponíveis
indice = defaultdict(list)
arquivos_consultados = 0

for arquivo in sorted(ROOT.rglob("*.csv")):
    nome = arquivo.name.upper()

    if "AUDIT_LOG" in nome or "TEST_LOCKED" in nome:
        continue

    try:
        with arquivo.open(
            encoding="utf-8-sig", newline="",
            errors="replace"
        ) as f:
            leitor = csv.DictReader(f)
            if not leitor.fieldnames:
                continue
            registros = list(leitor)
    except Exception as e:
        print("Falha ao ler:", arquivo.name, str(e))
        continue

    arquivos_consultados += 1

    for row in registros:
        for identificador in ids_linha(row):
            indice[identificador].append(
                (arquivo.name, row)
            )

def prioridade(item):
    nome = item[0].upper()
    if "V2_WORKING" in nome:
        return 0
    if "CLOSED_MANIFEST" in nome:
        return 1
    if "GT_SELECTION" in nome:
        return 2
    if "FINAL_LINE_MANIFEST" in nome:
        return 3
    return 4

def candidatos(identificador):
    encontrados = indice.get(normalizar(identificador), [])
    # Evitar registros repetidos da mesma fonte
    unicos = {}
    for fonte, row in encontrados:
        chave = (
            fonte,
            row.get("htr_line_id"),
            row.get("line_id"),
            row.get("htr_crop_v2_path"),
            row.get("crop_path")
        )
        unicos[chave] = (fonte, row)
    return sorted(unicos.values(), key=prioridade)

print("=" * 70)
print("QUMRAN — RECONCILIAÇÃO DOS METADADOS")
print("=" * 70)

print("Arquivos CSV consultados:", arquivos_consultados)

print("\n[1] RASTREABILIDADE DAS 59 AMOSTRAS")

sem_registro = []
com_registro = 0
varias_fontes = 0

for imagem in amostras:
    lista = candidatos(imagem.name)
    if lista:
        com_registro += 1
        if len(lista) > 1:
            varias_fontes += 1
    else:
        sem_registro.append(imagem.name)

print("Amostras com registros:", com_registro, "/59")
print("Amostras sem registros:", len(sem_registro))
print("Com múltiplas fontes:", varias_fontes)

print("\n[2] INVESTIGAÇÃO DOS 15 REGISTROS")

faltantes_anteriores = [
    "1QS_I_M017", "1QS_VII_M017", "1QS_IX_M010",
    "1QS_X_M017", "1QS_XI_M018", "1QS_X_M016",
    "1QS_IV_M018", "1QS_VII_M003", "1QS_X_M010",
    "1QS_I_M010", "1QS_II_M016", "1QS_VI_M014",
    "1QS_VII_M018", "1QS_IX_M005", "1QS_XI_M015"
]

for ident in faltantes_anteriores:
    lista = candidatos(ident)
    print(f"\n{ident} — {len(lista)} registro(s)")

    for fonte, row in lista[:4]:
        print("  Fonte:", fonte)
        print(
            "  Coluna:", row.get("column"),
            "| Linha:", row.get("line_number"),
            "| HTR:", row.get("htr_line_id")
        )
        print(
            "  Split:", row.get("split"),
            "| Ativo:", row.get("active_for_gt"),
            "| GT status:", row.get("gt_status_v2")
        )

print("\n[3] INVESTIGAÇÃO DAS QUATRO SOBREPOSIÇÕES")

pares_suspeitos = [
    ("1QS_V_M010", "1QS_V_M009"),
    ("1QS_VIII_M005", "1QS_VIII_M006"),
    ("1QS_VIII_M005", "1QS_VIII_M004"),
    ("1QS_X_M002", "1QS_X_M003")
]

def geometria(row):
    try:
        return tuple(
            float(str(row[c]).replace(",", "."))
            for c in ["x1", "y1", "x2", "y2"]
        )
    except (KeyError, ValueError, TypeError):
        return None

def dados_geometricos(row):
    g = geometria(row)
    if g is None:
        return "Geometria indisponível"
    x1, y1, x2, y2 = g
    return (
        f"Y=[{y1:.1f}, {y2:.1f}] "
        f"altura={y2-y1:.1f} "
        f"centro={(y1+y2)/2:.1f}"
    )

for id_teste, id_treino in pares_suspeitos:
    print("\n" + "-" * 60)
    print("TESTE:", id_teste)
    print("TREINO/VALIDAÇÃO:", id_treino)

    linhas_teste = [
        r for r in teste
        if normalizar(r.get("htr_line_id")) == id_teste
    ]

    linhas_treino = candidatos(id_treino)

    for r in linhas_teste[:1]:
        print(
            "  Teste | Coluna:", r.get("column"),
            "| Linha:", r.get("line_number")
        )
        print("  ", dados_geometricos(r))

    for fonte, r in linhas_treino[:3]:
        print(
            "  Fonte:", fonte,
            "| Coluna:", r.get("column"),
            "| Linha:", r.get("line_number")
        )
        print("  ", dados_geometricos(r))

print("\n" + "=" * 70)
print("DIAGNÓSTICO CONCLUÍDO")
print("Nenhum ground truth textual foi exibido.")
print("Nenhum arquivo foi modificado.")
print("=" * 70)


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import csv
import json
import hashlib
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS"
)

GT = ROOT / "htr_ground_truth_v2"
DS = ROOT / "htr_model/datasets"
MODEL_DIR = ROOT / "htr_model/01_finetune_FT01_R02"

MODEL = MODEL_DIR / "best_0.7102.safetensors"
TEST = GT / "1QS_BATCH07_TEST_LOCKED.csv"
META = GT / "1QS_BATCH07_TEST_LOCK_METADATA.json"

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for bloco in iter(lambda: f.read(1024*1024), b""):
            h.update(bloco)
    return h.hexdigest()

# Conferir modelo e manifesto
assert sha256(MODEL) == (
    "b843f7ba9a03d4c1c670d1d5e713957fee9137be5e157c4d3e90eb3bc254b10e"
), "Modelo diferente do FT01_R02 validado"

metadata = json.loads(META.read_text())

assert sha256(TEST) == metadata["manifest_sha256"], (
    "Manifesto selado alterado"
)

with TEST.open(encoding="utf-8-sig", newline="") as f:
    test_rows = {
        r["htr_line_id"]: r
        for r in csv.DictReader(f)
    }

def ler_manifest(nome):
    p = DS / nome
    return [
        Path(s.strip())
        for s in p.read_text().splitlines()
        if s.strip() and not s.startswith("#")
    ]

train_paths = (
    ler_manifest("1QS_TRAIN_55.lst")
    + ler_manifest("1QS_VALIDATION_CORE_4.lst")
)

train_images = {
    p.stem: p for p in train_paths
}

def localizar_teste(row):
    esperado = row["image_sha256"].strip().lower()
    candidatos = []

    for campo in [
        "htr_crop_v2_path",
        "frozen_crop_path",
        "crop_path"
    ]:
        valor = (row.get(campo) or "").strip()
        if not valor or valor.lower() in ("nan", "none"):
            continue

        p = Path(valor)
        opcoes = [
            p,
            ROOT / valor,
            GT / valor,
            GT / "images" / p.name,
            ROOT / "final_segmentation" / valor
        ]

        marcador = "Qumran_Analytics/1QS/"
        if marcador in valor:
            relativo = valor.split(marcador, 1)[1]
            opcoes.append(ROOT / relativo)

        candidatos.extend(opcoes)

    for p in candidatos:
        if p.is_file() and sha256(p) == esperado:
            return p

    raise FileNotFoundError(
        "Imagem do teste não localizada ou hash divergente: "
        + row["htr_line_id"]
    )

pares = [
    ("1QS_V_M010", "1QS_V_M009"),
    ("1QS_VIII_M005", "1QS_VIII_M006"),
    ("1QS_VIII_M005", "1QS_VIII_M004"),
    ("1QS_X_M002", "1QS_X_M003")
]

# Resolver e verificar tudo antes de abrir imagens
pares_resolvidos = []

for id_teste, id_treino in pares:
    assert id_teste in test_rows
    assert id_treino in train_images

    p_teste = localizar_teste(test_rows[id_teste])
    p_treino = train_images[id_treino]

    assert p_treino.is_file()

    pares_resolvidos.append(
        (id_teste, id_treino, p_teste, p_treino)
    )

# Registrar o início da inspeção, fora do conjunto selado
AUDIT = ROOT / "htr_model/evaluation_audit"
AUDIT.mkdir(parents=True, exist_ok=True)
LOG = AUDIT / "FT01_R02_VISUAL_QC_START.json"

if not LOG.exists():
    registro = {
        "event": "visual_qc_started",
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "model_sha256": sha256(MODEL),
        "test_manifest_sha256": sha256(TEST),
        "purpose": "Investigate adjacent-line crop overlap",
        "model_predictions_consulted": False,
        "reference_text_consulted": False,
        "test_manifest_modified": False,
        "pairs": [[a, b] for a, b in pares]
    }
    LOG.write_text(
        json.dumps(registro, indent=2),
        encoding="utf-8"
    )

# Exibir os recortes sem alterar imagens
fig, axes = plt.subplots(
    4, 2,
    figsize=(15, 12),
    constrained_layout=True
)

for i, (id_teste, id_treino, p_teste, p_treino) in enumerate(
    pares_resolvidos
):
    for j, (ident, caminho, titulo) in enumerate([
        (id_teste, p_teste, "TESTE SELADO"),
        (id_treino, p_treino, "TREINO / VALIDAÇÃO")
    ]):
        with Image.open(caminho) as im:
            imagem = im.convert("RGB")

        axes[i, j].imshow(imagem)
        axes[i, j].set_title(f"{titulo} — {ident}")
        axes[i, j].axis("off")

plt.suptitle(
    "QUMRAN — Controle visual de sobreposição de recortes",
    fontsize=15
)
plt.show()

print("\nQuatro pares exibidos para revisão.")
print("Modelo e manifesto verificados por SHA-256.")
print("Nenhum ground truth ou previsão foi consultado.")
print("Registro de inspeção:", LOG)


In [ ]:

from pathlib import Path
from PIL import Image
import numpy as np
import csv

ROOT = Path("/content/drive/MyDrive/Qumran_Analytics/1QS")

ACTIVE = (
    ROOT / "htr_ground_truth_v2"
    / "1QS_GT_SELECTION_V2_ACTIVE.csv"
)

with ACTIVE.open(encoding="utf-8-sig", newline="") as f:
    registros = list(csv.DictReader(f))

indice = {
    r["htr_line_id"]: r
    for r in registros
    if r.get("htr_line_id")
    and str(r.get("active_for_gt", "")).lower() == "true"
}

def coordenadas(row):
    return tuple(
        int(round(float(row[k])))
        for k in ("x1", "y1", "x2", "y2")
    )

def carregar_mascara(caminho, limite):
    imagem = np.asarray(
        Image.open(caminho).convert("L")
    )
    return imagem < limite

print("=" * 65)
print("QUMRAN — TESTE DE SOBREPOSIÇÃO DE TINTA")
print("=" * 65)

resultados = []

# pares_resolvidos foi criado na célula anterior
assert "pares_resolvidos" in globals(), (
    "Execute primeiro a célula que gerou os quatro pares."
)

for id_teste, id_treino, img_t, img_r in pares_resolvidos:

    rt = test_rows[id_teste]
    rr = indice[id_treino]

    a = coordenadas(rt)
    b = coordenadas(rr)

    x0 = max(a[0], b[0])
    y0 = max(a[1], b[1])
    x1 = min(a[2], b[2])
    y1 = min(a[3], b[3])

    print(f"\nTESTE: {id_teste}")
    print(f"TREINO: {id_treino}")

    if x1 <= x0 or y1 <= y0:
        print("Sem interseção geométrica.")
        continue

    with Image.open(img_t) as im:
        tamanho_t = im.size

    with Image.open(img_r) as im:
        tamanho_r = im.size

    esperado_t = (a[2]-a[0], a[3]-a[1])
    esperado_r = (b[2]-b[0], b[3]-b[1])

    if tamanho_t != esperado_t or tamanho_r != esperado_r:
        print("INCONCLUSIVO: dimensões e coordenadas diferem.")
        print("Teste:", tamanho_t, "esperado:", esperado_t)
        print("Treino:", tamanho_r, "esperado:", esperado_r)
        continue

    for limite in (100, 125, 150):
        mt = carregar_mascara(img_t, limite)
        mr = carregar_mascara(img_r, limite)

        trecho_t = mt[
            y0-a[1]:y1-a[1],
            x0-a[0]:x1-a[0]
        ]

        trecho_r = mr[
            y0-b[1]:y1-b[1],
            x0-b[0]:x1-b[0]
        ]

        comuns = int(np.logical_and(
            trecho_t, trecho_r
        ).sum())

        pixels_teste = int(trecho_t.sum())

        percentual = (
            100 * comuns / pixels_teste
            if pixels_teste else 0.0
        )

        print(
            f"Limiar {limite}: "
            f"{comuns} pixels escuros coincidentes; "
            f"{percentual:.2f}% dos pixels escuros "
            f"do teste na interseção"
        )

        resultados.append({
            "teste": id_teste,
            "treino": id_treino,
            "limiar": limite,
            "pixels_coincidentes": comuns,
            "percentual": percentual
        })

print("\n" + "=" * 65)
print("ANÁLISE FINALIZADA")
print("Nenhuma transcrição foi consultada.")
print("Nenhum arquivo foi modificado.")


In [ ]:

from pathlib import Path
from PIL import Image
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
import csv

ROOT = Path(
    "/content/drive/MyDrive/Qumran_Analytics/1QS"
)

ROI_DIR = ROOT / "automatic_segmentation/roi"

TEST_CSV = (
    ROOT / "htr_ground_truth_v2"
    / "1QS_BATCH07_TEST_LOCKED.csv"
)

ACTIVE_CSV = (
    ROOT / "htr_ground_truth_v2"
    / "1QS_GT_SELECTION_V2_ACTIVE.csv"
)

def carregar_csv(caminho):
    with open(
        caminho,
        encoding="utf-8-sig",
        newline=""
    ) as f:
        return list(csv.DictReader(f))

teste = {
    r["htr_line_id"]: r
    for r in carregar_csv(TEST_CSV)
}

treino = {
    r["htr_line_id"]: r
    for r in carregar_csv(ACTIVE_CSV)
    if r.get("htr_line_id")
    and str(r.get("active_for_gt", "")).lower() == "true"
}

def bbox(row):
    return tuple(
        float(str(row[k]).replace(",", "."))
        for k in ("x1", "y1", "x2", "y2")
    )

pares = [
    ("V", "1QS_V_M010", "1QS_V_M009"),
    ("VIII", "1QS_VIII_M005", "1QS_VIII_M006"),
    ("VIII", "1QS_VIII_M005", "1QS_VIII_M004"),
    ("X", "1QS_X_M002", "1QS_X_M003")
]

fig, axes = plt.subplots(
    4, 1,
    figsize=(16, 14),
    constrained_layout=True
)

print("=" * 68)
print("QUMRAN — SOBREPOSIÇÃO NO MANUSCRITO ORIGINAL")
print("=" * 68)

for i, (coluna, id_teste, id_treino) in enumerate(pares):

    roi_path = ROI_DIR / f"1QS_col_{coluna}_ROI.png"

    assert roi_path.is_file(), (
        f"ROI não localizada: {roi_path}"
    )
    assert id_teste in teste
    assert id_treino in treino

    a = bbox(teste[id_teste])
    b = bbox(treino[id_treino])

    with Image.open(roi_path) as imagem:
        largura, altura = imagem.size

        # Confirmar que as coordenadas cabem na ROI
        for caixa in (a, b):
            assert (
                0 <= caixa[0] < caixa[2] <= largura
                and 0 <= caixa[1] < caixa[3] <= altura
            ), (
                f"Coordenadas incompatíveis com a ROI {coluna}. "
                "É necessário verificar o sistema de coordenadas."
            )

        margem = 25

        esquerda = max(0, int(min(a[0], b[0]) - margem))
        superior = max(0, int(min(a[1], b[1]) - margem))
        direita = min(largura, int(max(a[2], b[2]) + margem))
        inferior = min(altura, int(max(a[3], b[3]) + margem))

        recorte = imagem.crop(
            (esquerda, superior, direita, inferior)
        ).convert("RGB")

    ax = axes[i]
    ax.imshow(recorte)

    for caixa, cor, nome in [
        (a, "dodgerblue", "TESTE"),
        (b, "darkorange", "TREINO")
    ]:
        x1, y1, x2, y2 = caixa

        retangulo = Rectangle(
            (x1-esquerda, y1-superior),
            x2-x1,
            y2-y1,
            linewidth=2,
            edgecolor=cor,
            facecolor="none"
        )

        ax.add_patch(retangulo)

    # Interseção real dos retângulos
    ix1 = max(a[0], b[0])
    iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2])
    iy2 = min(a[3], b[3])

    if ix2 > ix1 and iy2 > iy1:
        inter = Rectangle(
            (ix1-esquerda, iy1-superior),
            ix2-ix1,
            iy2-iy1,
            facecolor="red",
            alpha=0.20,
            edgecolor="red",
            linewidth=1
        )
        ax.add_patch(inter)

        print(
            f"\nColuna {coluna}: "
            f"{id_teste} x {id_treino}"
        )
        print(
            f"Interseção vertical: "
            f"{iy2-iy1:.1f} pixels"
        )
    else:
        print(f"\nColuna {coluna}: sem interseção")

    ax.set_title(
        f"COLUNA {coluna} | "
        f"AZUL: {id_teste} | "
        f"LARANJA: {id_treino}",
        fontsize=11
    )

    ax.axis("off")

plt.suptitle(
    "QUMRAN — Auditoria de regiões sobrepostas no 1QS",
    fontsize=16
)

plt.show()

print("\nANÁLISE GEOMÉTRICA CONCLUÍDA")
print("Nenhuma transcrição consultada.")
print("Nenhum arquivo modificado.")
print(
    "A interseção dos retângulos, isoladamente, "
    "não comprova compartilhamento de tinta."
)


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import csv
import json
import hashlib
import shutil

ROOT = Path("/content/drive/MyDrive/Qumran_Analytics/1QS")

GT = ROOT / "htr_ground_truth_v2"
AUDIT = ROOT / "htr_model/evaluation_audit"

MANIFEST = GT / "1QS_BATCH07_TEST_LOCKED.csv"
META = GT / "1QS_BATCH07_TEST_LOCK_METADATA.json"

OUT = AUDIT / "BATCH07_TRANSCRIPTION_PACKET_v1"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

with MANIFEST.open(
    encoding="utf-8-sig", newline=""
) as f:
    linhas = list(csv.DictReader(f))

metadata = json.loads(META.read_text(encoding="utf-8"))

assert sha256(MANIFEST) == metadata["manifest_sha256"]
assert len(linhas) == 11
assert not OUT.exists(), "Pacote já existe: não sobrescrever."

# Indexar imagens por nome, sem abrir transcrições
imagens = [
    p for p in ROOT.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {
        ".png", ".jpg", ".jpeg", ".tif", ".tiff"
    }
]

indice = {}

for p in imagens:
    indice.setdefault(p.name, []).append(p)

def localizar(row):
    esperado = row["image_sha256"].strip().lower()
    candidatos = []

    for campo in [
        "htr_crop_v2_path",
        "frozen_crop_path",
        "crop_path"
    ]:
        valor = (row.get(campo) or "").strip()
        if not valor:
            continue

        p = Path(valor)

        if p.is_file():
            candidatos.append(p)

        marcador = "Qumran_Analytics/1QS/"
        if marcador in valor:
            relativo = valor.split(marcador, 1)[1]
            candidatos.append(ROOT / relativo)

        candidatos.extend(indice.get(p.name, []))

    for p in dict.fromkeys(candidatos):
        if p.is_file() and sha256(p) == esperado:
            return p

    raise FileNotFoundError(
        "Imagem não localizada com hash correto: "
        + row["htr_line_id"]
    )

# Resolver tudo antes de criar a pasta
resolvidos = []

for row in linhas:
    assert not (row.get("gt_text") or "").strip()
    assert not (row.get("image_only_gt_text") or "").strip()

    origem = localizar(row)
    resolvidos.append((row, origem))

# Criar pacote
IMG_DIR = OUT / "images"
IMG_DIR.mkdir(parents=True)

campos = [
    "htr_line_id",
    "column",
    "line_number",
    "image_filename",
    "transcription_hebrew",
    "uncertain_readings",
    "unreadable_regions",
    "transcriber_code",
    "transcription_date_utc",
    "reviewer_code",
    "review_status"
]

planilha = []
hashes = []

for row, origem in resolvidos:
    identificador = row["htr_line_id"]
    destino = IMG_DIR / f"{identificador}{origem.suffix.lower()}"

    shutil.copy2(origem, destino)

    assert sha256(destino) == row["image_sha256"].lower()

    planilha.append({
        "htr_line_id": identificador,
        "column": row["column"],
        "line_number": row["line_number"],
        "image_filename": destino.name,
        "transcription_hebrew": "",
        "uncertain_readings": "",
        "unreadable_regions": "",
        "transcriber_code": "",
        "transcription_date_utc": "",
        "reviewer_code": "",
        "review_status": "PENDING"
    })

    hashes.append({
        "image_filename": destino.name,
        "sha256": sha256(destino)
    })

def salvar_csv(path, rows, fields):
    with path.open(
        "w", encoding="utf-8-sig", newline=""
    ) as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)

salvar_csv(
    OUT / "TRANSCRIPTION_BLANK.csv",
    planilha,
    campos
)

salvar_csv(
    OUT / "IMAGE_HASHES.csv",
    hashes,
    ["image_filename", "sha256"]
)

instrucoes = """
QUMRAN ANALYTICS — BATCH07

PROTOCOLO DE TRANSCRIÇÃO PALEOGRÁFICA

1. Transcrever exclusivamente a partir das imagens.
2. Não consultar previsões dos modelos HTR.
3. Não consultar ETCBC ou edições de referência
   durante a primeira leitura independente.
4. Registrar incertezas e regiões ilegíveis.
5. Utilizar Unicode hebraico e normalização NFD.
6. Não preencher caracteres por conjectura
   sem registrar explicitamente a incerteza.
7. Submeter as leituras a uma segunda revisão,
   preferencialmente por especialista independente.
8. Não alterar os arquivos de imagem.
9. Não utilizar estas transcrições para
   reajustar os modelos antes da avaliação final.
10. Registrar decisões de normalização,
    segmentação e tratamento de lacunas.
"""

(OUT / "TRANSCRIPTION_PROTOCOL.txt").write_text(
    instrucoes.strip(),
    encoding="utf-8"
)

registro = {
    "event": "transcription_packet_created",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "source_manifest_sha256": sha256(MANIFEST),
    "n_images": len(resolvidos),
    "predictions_included": False,
    "reference_transcriptions_included": False,
    "source_manifest_modified": False,
    "visual_qc_previously_performed": True,
    "geometric_overlap_risk_ids": [
        "1QS_V_M010",
        "1QS_VIII_M005",
        "1QS_X_M002"
    ]
}

(OUT / "PACKET_METADATA.json").write_text(
    json.dumps(registro, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

zip_path = shutil.make_archive(
    str(OUT),
    "zip",
    root_dir=OUT
)

print("=" * 65)
print("QUMRAN — PACOTE DE TRANSCRIÇÃO PREPARADO")
print("=" * 65)
print("Imagens:", len(resolvidos))
print("Integridade SHA-256: CONFIRMADA")
print("Transcrições existentes incluídas: NÃO")
print("Previsões dos modelos incluídas: NÃO")
print("Pasta:", OUT)
print("ZIP:", zip_path)
print("Manifesto selado: PRESERVADO")
